# Healthcare Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 2: ED Cost Prediction**.

**Goal**: Predict `ed_cost_next3y_usd` for each patient
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular ED cost data with a simple Random Forest regressor.


In [1]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="adsb_AJp9fgtjuTRZ4Tk0mNnYDWxG_1759039055",        # Get from your team dashboard
    team_name="clai"     # Your exact team name
)

# Iteration 1 
- 775+

In [5]:
import os, re, gc, random, warnings
import numpy as np
import pandas as pd
import joblib
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor, Pool

# ==============================================================================
# CONFIG & HYPERPARAMETERS
# ==============================================================================
SEED = 42
N_FOLDS = 5   # 保持 5 折，但增加 Seeds 数量来稳定
N_SEEDS = 5   # 5 Seeds * 5 Folds = 25 models per architecture (Robust!)

# 路径配置 (自动适配环境)
BASE_DIR = r"D:\AgentDs\agent_ds_healthcare"
if not os.path.exists(BASE_DIR):
    BASE_DIR = "." # Fallback for other envs

TRAIN_PATH = os.path.join(BASE_DIR, "ed_cost_train.csv")
TEST_PATH  = os.path.join(BASE_DIR, "ed_cost_test.csv")
PATIENTS_PATH = os.path.join(BASE_DIR, "patients.csv")
ADM_TRAIN_PATH = os.path.join(BASE_DIR, "admissions_train.csv")
ADM_TEST_PATH  = os.path.join(BASE_DIR, "admissions_test.csv")
RECEIPTS_JOBLIB = os.path.join(BASE_DIR, "receipts_parsed.joblib")
SUBMISSION_PATH = os.path.join(BASE_DIR, "submission_gemini.csv")

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)
warnings.filterwarnings("ignore")

print(f"Code 17: Portfolio Strategy (Robust Shallow + Constrained Deep + Log Transform)")
print(f"Plan: {N_SEEDS} Seeds x {N_FOLDS} Folds x 3 Model Architectures")

# ==============================================================================
# DATA LOADING & PREPROCESSING (Streamlined from Code16)
# ==============================================================================
def load_data():
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    patients = pd.read_csv(PATIENTS_PATH) if os.path.exists(PATIENTS_PATH) else None
    
    # Simple Admissions Aggregation
    adm = pd.DataFrame()
    if os.path.exists(ADM_TRAIN_PATH) and os.path.exists(ADM_TEST_PATH):
        at = pd.read_csv(ADM_TRAIN_PATH)
        ae = pd.read_csv(ADM_TEST_PATH)
        # Drop leakage
        if 'readmit_30d' in at.columns: at = at.drop(columns=['readmit_30d'])
        if 'readmit_30d' in ae.columns: ae = ae.drop(columns=['readmit_30d'])
        
        adm_all = pd.concat([at, ae], ignore_index=True)
        # Aggregates
        adm = adm_all.groupby('patient_id').agg(
            adm_count=('patient_id', 'size'),
            adm_los_mean=('los_days', 'mean'),
            adm_los_max=('los_days', 'max'),
            adm_emergent_rate=('acuity_emergent', 'mean'),
            adm_charlson_max=('charlson_band', 'max') # Capture worst case health
        ).reset_index()

    # Receipts Loading (Prioritize Joblib for speed)
    rcpt = pd.DataFrame()
    if os.path.exists(RECEIPTS_JOBLIB):
        try:
            r_data = joblib.load(RECEIPTS_JOBLIB)
            if isinstance(r_data, dict):
                rcpt = pd.DataFrame.from_dict(r_data, orient='index').reset_index()
                rcpt.rename(columns={'index': 'patient_id'}, inplace=True)
            elif isinstance(r_data, pd.DataFrame):
                rcpt = r_data
            
            # --- FIX: Ensure patient_id is Integer ---
            if 'patient_id' in rcpt.columns:
                rcpt['patient_id'] = pd.to_numeric(rcpt['patient_id'], errors='coerce')
                rcpt = rcpt.dropna(subset=['patient_id']) # Drop invalid IDs
                rcpt['patient_id'] = rcpt['patient_id'].astype(int)
            
            # Simple aggregations if it's line-item level
            if 'patient_id' in rcpt.columns and 'line_total' in rcpt.columns:
                 # It's long format, aggregate it
                 rcpt = rcpt.groupby('patient_id').agg(
                     rcpt_count=('line_total', 'size'),
                     rcpt_total=('line_total', 'sum'),
                     rcpt_max=('line_total', 'max')
                 ).reset_index()
        except Exception as e:
            print(f"Warning: Failed to load/parse receipts joblib: {e}")
    
    # Merge
    # 1. Base
    df_tr = train.merge(patients, on='patient_id', how='left')
    df_te = test.merge(patients, on='patient_id', how='left')
    
    # 2. Adm
    if not adm.empty:
        df_tr = df_tr.merge(adm, on='patient_id', how='left')
        df_te = df_te.merge(adm, on='patient_id', how='left')
    
    # 3. Receipts
    if not rcpt.empty:
        df_tr = df_tr.merge(rcpt, on='patient_id', how='left')
        df_te = df_te.merge(rcpt, on='patient_id', how='left')
        
    return df_tr, df_te

train_df, test_df = load_data()

# ==============================================================================
# FEATURE ENGINEERING (The "Less is More" approach)
# ==============================================================================
def engineer_features(df):
    df = df.copy()
    
    # 1. Log transforms for skew
    df['log_prior_cost'] = np.log1p(df['prior_ed_cost_5y_usd'])
    df['log_prior_visits'] = np.log1p(df['prior_ed_visits_5y'])
    
    # 2. Ratios (Cost per visit)
    df['cost_per_visit'] = df['prior_ed_cost_5y_usd'] / df['prior_ed_visits_5y'].clip(lower=1)
    
    # 3. Interactions (Medical intuition)
    # Older people with chronic conditions = higher risk
    if 'age' in df.columns and 'primary_chronic' in df.columns:
        # Simple encoding for interaction
        chronic_map = {k: i for i, k in enumerate(df['primary_chronic'].unique())}
        df['chronic_idx'] = df['primary_chronic'].map(chronic_map)
        df['age_x_chronic'] = df['age'] * (df['chronic_idx'] + 1)

    # 4. Fill NaNs sensibly
    num_cols = df.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        if c not in ['ed_cost_next3y_usd', 'patient_id']:
            df[c] = df[c].fillna(0) # For count/cost features, 0 is usually safe default
            
    return df

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

# Feature Lists
target = 'ed_cost_next3y_usd'
drop_cols = [target, 'patient_id', 'primary_chronic', 'zip3'] # Drop raw cats if not encoded by CatBoost
features = [c for c in train_df.columns if c not in drop_cols]

# Categorical features indices for CatBoost
cat_features = []
for c in ['sex', 'insurance']: # Keep these as raw categories
    if c in features:
        cat_features.append(c)
        # Ensure string type
        train_df[c] = train_df[c].astype(str)
        test_df[c] = test_df[c].astype(str)

print(f"Training on {len(features)} features: {features[:5]}...")

# ==============================================================================
# MODEL STRATEGY: THE PORTFOLIO
# ==============================================================================

# Model A: The "Stabilizer" (Code16 style)
# Shallow, Heavy Pruning (via RSM), Robust
params_A = {
    'loss_function': 'MAE',
    'iterations': 2500,
    'learning_rate': 0.04,
    'depth': 4,              # Shallow
    'l2_leaf_reg': 3,
    'rsm': 0.6,              # Aggressive column subsampling (Feature Pruning on the fly)
    'subsample': 0.8,
    'random_strength': 1,
    'verbose': 0,
    'allow_writing_files': False
}

# Model B: The "Deep & Constrained" (Addressing your point)
# Allows Depth 6, BUT enforces high L2 regularization.
# Captures complex interactions if they exist, but penalized heavily if they are noise.
params_B = {
    'loss_function': 'MAE',
    'iterations': 3000,
    'learning_rate': 0.02,   # Slower learning for deeper trees
    'depth': 6,              # Allowed deeper
    'l2_leaf_reg': 10,       # Very High Reg to prevent overfit
    'rsm': 0.7,
    'subsample': 0.8,
    'bagging_temperature': 1,
    'verbose': 0,
    'allow_writing_files': False
}

# Model C: The "Log Transformer" (Hedge)
# Trains on Log1p target to handle outliers differently.
params_C = {
    'loss_function': 'RMSE', # Log-space usually minimizes RMSE
    'iterations': 2500,
    'learning_rate': 0.03,
    'depth': 5,
    'l2_leaf_reg': 5,
    'rsm': 0.8,
    'verbose': 0,
    'allow_writing_files': False
}

models_config = {
    'A_Shallow': params_A,
    'B_DeepConstrained': params_B,
    'C_LogTarget': params_C
}

# ==============================================================================
# TRAINING LOOP
# ==============================================================================
oof_preds = {k: np.zeros(len(train_df)) for k in models_config.keys()}
test_preds = {k: np.zeros(len(test_df)) for k in models_config.keys()}
scores = {k: [] for k in models_config.keys()}

y = train_df[target].values
# Binning for stratification
y_bins = pd.qcut(y, q=10, labels=False, duplicates='drop')

for seed in range(N_SEEDS):
    print(f"\n--- Seed {seed+1}/{N_SEEDS} ---")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + seed)
    
    for fold, (idx_tr, idx_va) in enumerate(skf.split(train_df, y_bins)):
        X_tr = train_df.iloc[idx_tr][features]
        y_tr = y[idx_tr]
        X_va = train_df.iloc[idx_va][features]
        y_va = y[idx_va]
        X_te = test_df[features]
        
        # 1. Train Model A
        print(f"Training Model A (Shallow) on Fold {fold+1}...")
        p_A = params_A.copy()
        p_A['random_seed'] = SEED + seed
        m_A = CatBoostRegressor(**p_A)
        m_A.fit(X_tr, y_tr, cat_features=cat_features, eval_set=(X_va, y_va), early_stopping_rounds=100)
        
        pred_val_A = m_A.predict(X_va)
        pred_test_A = m_A.predict(X_te)
        
        oof_preds['A_Shallow'][idx_va] += pred_val_A / N_SEEDS # Average over seeds
        test_preds['A_Shallow'] += pred_test_A / (N_SEEDS * N_FOLDS)
        scores['A_Shallow'].append(mean_absolute_error(y_va, pred_val_A))
        
        # 2. Train Model B
        print(f"Training Model B (Deep & Constrained) on Fold {fold+1}...")
        p_B = params_B.copy()
        p_B['random_seed'] = SEED + seed
        m_B = CatBoostRegressor(**p_B)
        m_B.fit(X_tr, y_tr, cat_features=cat_features, eval_set=(X_va, y_va), early_stopping_rounds=100)
        
        pred_val_B = m_B.predict(X_va)
        pred_test_B = m_B.predict(X_te)
        
        oof_preds['B_DeepConstrained'][idx_va] += pred_val_B / N_SEEDS
        test_preds['B_DeepConstrained'] += pred_test_B / (N_SEEDS * N_FOLDS)
        scores['B_DeepConstrained'].append(mean_absolute_error(y_va, pred_val_B))

        # 3. Train Model C (Log Target)
        print(f"Training Model C (Log Target) on Fold {fold+1}...")
        p_C = params_C.copy()
        p_C['random_seed'] = SEED + seed
        # Transform Target
        m_C = CatBoostRegressor(**p_C)
        m_C.fit(X_tr, np.log1p(y_tr), cat_features=cat_features, eval_set=(X_va, np.log1p(y_va)), early_stopping_rounds=100)
        
        # Inverse Transform
        pred_val_C = np.expm1(m_C.predict(X_va))
        pred_test_C = np.expm1(m_C.predict(X_te))
        
        oof_preds['C_LogTarget'][idx_va] += pred_val_C / N_SEEDS
        test_preds['C_LogTarget'] += pred_test_C / (N_SEEDS * N_FOLDS)
        scores['C_LogTarget'].append(mean_absolute_error(y_va, pred_val_C))

# ==============================================================================
# ENSEMBLE OPTIMIZATION (SLSQP)
# ==============================================================================
print("\n--- Model Performance (Avg MAE across all folds/seeds) ---")
for k, v in scores.items():
    print(f"{k}: {np.mean(v):.4f}")

print("\n--- Optimizing Ensemble Weights ---")
def loss_func(weights):
    final_pred = (weights[0] * oof_preds['A_Shallow'] + 
                  weights[1] * oof_preds['B_DeepConstrained'] + 
                  weights[2] * oof_preds['C_LogTarget'])
    # Optional: Clip negative predictions
    final_pred = np.clip(final_pred, 0, None)
    return mean_absolute_error(y, final_pred)

# Constraint: Sum of weights = 1
cons = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
# Bounds: 0 <= weight <= 1
bnds = ((0, 1), (0, 1), (0, 1))
init_weights = [0.33, 0.33, 0.33]

res = minimize(loss_func, init_weights, method='SLSQP', bounds=bnds, constraints=cons)
best_w = res.x

print(f"Best Weights: A={best_w[0]:.3f}, B={best_w[1]:.3f}, C={best_w[2]:.3f}")
print(f"Optimized CV MAE: {res.fun:.4f}")

# ==============================================================================
# SUBMISSION
# ==============================================================================
final_test_pred = (best_w[0] * test_preds['A_Shallow'] + 
                   best_w[1] * test_preds['B_DeepConstrained'] + 
                   best_w[2] * test_preds['C_LogTarget'])
final_test_pred = np.clip(final_test_pred, 0, None)

sub = pd.DataFrame({
    'patient_id': test_df['patient_id'],
    'ed_cost_next3y_usd': final_test_pred
})

print(f"Submission shape: {sub.shape}")
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved to {SUBMISSION_PATH}")

Code 17: Portfolio Strategy (Robust Shallow + Constrained Deep + Log Transform)
Plan: 5 Seeds x 5 Folds x 3 Model Architectures
Training on 15 features: ['prior_ed_visits_5y', 'prior_ed_cost_5y_usd', 'age', 'sex', 'insurance']...

--- Seed 1/5 ---
Training Model A (Shallow) on Fold 1...
Training Model B (Deep & Constrained) on Fold 1...
Training Model C (Log Target) on Fold 1...
Training Model A (Shallow) on Fold 2...
Training Model B (Deep & Constrained) on Fold 2...
Training Model C (Log Target) on Fold 2...
Training Model A (Shallow) on Fold 3...
Training Model B (Deep & Constrained) on Fold 3...
Training Model C (Log Target) on Fold 3...
Training Model A (Shallow) on Fold 4...
Training Model B (Deep & Constrained) on Fold 4...
Training Model C (Log Target) on Fold 4...
Training Model A (Shallow) on Fold 5...
Training Model B (Deep & Constrained) on Fold 5...
Training Model C (Log Target) on Fold 5...

--- Seed 2/5 ---
Training Model A (Shallow) on Fold 1...
Training Model B (Deep &

# Iteration 2
- 450.5367

In [7]:
# === ITERATION 9: The "Iter7-Refined" (Hierarchical Shift Optimization) ===
# Back to Iteration 7 codebase exactly (proven best features).
# IMPROVEMENT: Replace manual "chronic shift" with an automated, stratified shift optimization.
# Rationale: Iter 8 failed due to feature noise; Code 17 underfitted. Iter 7 was the sweet spot.
#            We squeeze the last juice by optimally shifting predictions per chronic group.

import os, re, sys, gc, math, warnings, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
TARGET = "ed_cost_next3y_usd"

print("="*90)
print("ITERATION 9 | Back to Iter7 (Winner) + Automated Stratified Shift Optimization")
print("Logic: Iter7 (MAE~451) was best. We refine it by optimizing shifts per chronic group.")
print("="*90)

# -----------------------------
# Deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Config (Iter7 defaults)
# -----------------------------
class CFG:
    N_FOLDS = 7
    N_SEEDS = 5
    ITERS = 3000
    ES_ROUNDS = 120
    LR = 0.03
    RSM = 0.80
    
    # Ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.03, 0.05, 0.08, 0.10, 0.15]
    # We will remove global SHIFT_GRID from ensemble search and move it to post-processing optimization
    
    # Penalties (Iter7)
    STD_PEN = 0.20
    LAM_PEN = 8.0

# -----------------------------
# Utilities
# -----------------------------
def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan": return None
    if re.fullmatch(r"\d+\.0+", s): s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)): return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s: return None
    return s.zfill(3)

def qdict(x, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

# -----------------------------
# Data Loading & Feats (Iter7 Exact)
# -----------------------------
def load_admissions_features(adm_train_path, adm_test_path):
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns: df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs: return None
    adm = pd.concat(dfs, ignore_index=True)
    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")
    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()
    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    return out

def build_pdf_features_from_lineitems(li):
    li = li.copy()
    # Locate cols
    code_col = next((c for c in ["code","cpt","cpt_code","hcpcs","proc_code"] if c in li.columns), None)
    total_col = next((c for c in ["line_total","line_total_usd","total","amount"] if c in li.columns), None)
    if not code_col or not total_col or "patient_id" not in li.columns: return None

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)
    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()
    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).clip(lower=0.0)

    # Base totals
    total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)

    # Buckets (Iter7 definition)
    is_em = li["code"].isin(["99281","99282","99283","99284","99285"])
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)
    
    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])
    
    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")
    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    # Features
    share = (li["amt"] / denom).fillna(0.0)
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    
    # Flags
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")
    has_obs = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    
    # Specifics
    def has_c(c, n): return (li["code"].eq(c).astype(int).groupby(li["patient_id"]).max()).rename(n)
    has_99285 = has_c("99285","has_99285")
    
    # Bucket totals
    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    out = pd.concat([
        n_unique_codes, cost_hhi, n_em_codes, max_em_level,
        has_critical_care, has_high_acuity, has_imaging, has_obs, has_99285,
        total
    ], axis=1).reset_index()

    for s in [em_total, crit_total, proc_total, high_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")
    
    for c in ["em_total","crit_total","proc_total","high_total","receipt_total"]:
        if c not in out.columns: out[c] = 0.0
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
        
    d2 = out["receipt_total"].replace(0.0, np.nan)
    out["pct_cost_critical"] = (out["crit_total"] / d2).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / d2).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / d2).fillna(0.0)
    out["cost_per_em"] = np.where(out["n_em_codes"]>0, out["receipt_total"]/out["n_em_codes"].clip(lower=1), out["receipt_total"])
    
    # Composite
    out["n_high_acuity_total"] = (
        out["has_high_acuity"].fillna(0) + out["has_critical_care"].fillna(0)
    ).astype(int) # Simplified proxy from Iter7 logic

    return out

def load_receipts(path):
    if not os.path.exists(path): return None
    try:
        d = joblib_load(path)
        if isinstance(d, dict):
            for k in ["lineitems_df","lineitems","items_df"]:
                if k in d and isinstance(d[k], pd.DataFrame): return build_pdf_features_from_lineitems(d[k])
        if isinstance(d, pd.DataFrame): return build_pdf_features_from_lineitems(d)
    except: pass
    return None

def build_features(ed, pts, adm, pdf):
    f = ed.copy()
    # Chronic
    cmap = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    f["primary_chronic"] = f["primary_chronic"].astype(str)
    f["chronic_encoded"] = f["primary_chronic"].str.upper().map(cmap).fillna(-1).astype(float)
    
    # Prior
    f["prior_ed_visits_5y"] = pd.to_numeric(f["prior_ed_visits_5y"], errors="coerce").fillna(0.0)
    f["prior_ed_cost_5y_usd"] = pd.to_numeric(f["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0)
    
    f["sqrt_prior_cost"] = np.sqrt(f["prior_ed_cost_5y_usd"].clip(lower=0))
    f["log_prior_cost"] = np.log1p(f["prior_ed_cost_5y_usd"].clip(lower=0))
    f["log_visits"] = np.log1p(f["prior_ed_visits_5y"].clip(lower=0))
    f["cost_per_visit"] = f["prior_ed_cost_5y_usd"] / f["prior_ed_visits_5y"].clip(lower=1)
    f["prior_cost_cap20k"] = f["prior_ed_cost_5y_usd"].clip(upper=20000)
    f["log_prior_cost_cap20k"] = np.log1p(f["prior_cost_cap20k"])
    
    # Baseline
    f["baseline_next3y"] = f["prior_ed_cost_5y_usd"] * 0.6
    
    # Patients
    p = pts.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper()=="M").astype(int)
    ins_map = {"private":2,"public":1,"self_pay":0,"selfpay":0}
    p["insurance_encoded"] = p["insurance"].astype(str).str.lower().map(ins_map).fillna(-1)
    z = p["zip3"].apply(standardize_zip3).astype(str).str[0]
    p["zip_region"] = pd.to_numeric(z, errors="coerce").fillna(-1)
    
    f = f.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    f["ins_x_chronic"] = f["insurance_encoded"] * f["chronic_encoded"]
    
    # Adm
    if adm is not None:
        f = f.merge(adm, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in f.columns: f[c] = f[c].fillna(f[c].median())
            
    # PDF
    if pdf is not None:
        f = f.merge(pdf, on="patient_id", how="left")
        for c in pdf.columns:
            if c!="patient_id" and pd.api.types.is_numeric_dtype(pdf[c]):
                f[c] = f[c].fillna(f[c].median())
                
    # Interactions
    if "pct_cost_critical" in f.columns:
        f["logprior_x_pctcritical"] = f["log_prior_cost"] * f["pct_cost_critical"]
    if "n_high_acuity_total" in f.columns:
        f["logprior_x_highacu"] = f["log_prior_cost"] * f["n_high_acuity_total"]
    if "n_unique_codes" in f.columns:
        f["cost_per_code"] = f["prior_ed_cost_5y_usd"] / f["n_unique_codes"].clip(lower=1)

    # Clean
    for c in f.columns:
        if c not in ["patient_id","primary_chronic",TARGET]:
            f[c] = pd.to_numeric(f[c], errors="coerce").fillna(0.0)
            
    return f

# -----------------------------
# Main Execution
# -----------------------------
print("\n[load] Reading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
pts_df = pd.read_csv(PATIENTS_PATH)
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
pdf_df = load_receipts(RECEIPTS_JOBLIB_PATH)

for d in [train_df, test_df, pts_df]: d["patient_id"] = pd.to_numeric(d["patient_id"], errors="coerce").astype(int)

train_feat = build_features(train_df, pts_df, adm_df, pdf_df)
test_feat = build_features(test_df, pts_df, adm_df, pdf_df)

# Feature Selection (Iter7 Pruned Set)
pruned_cols = [
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k","cost_per_visit","log_visits",
    "baseline_next3y",
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    "charlson_max","charlson_mean","pct_emergent",
    "cost_per_em","cost_hhi","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","n_unique_codes",
    "logprior_x_pctcritical","logprior_x_highacu","cost_per_code"
]
# Full numeric set for Model A
all_cols = [c for c in train_feat.columns if c not in ["patient_id","primary_chronic",TARGET,"sex","insurance","zip3"] and pd.api.types.is_numeric_dtype(train_feat[c])]
# Remove constants
feat_full = [c for c in all_cols if train_feat[c].nunique() > 1]
feat_pruned = [c for c in pruned_cols if c in train_feat.columns and train_feat[c].nunique() > 1]

print(f"Features: Full={len(feat_full)}, Pruned={len(feat_pruned)}")

# -----------------------------
# Training (Iter7 Specs)
# -----------------------------
y = train_feat[TARGET].values.astype(float)
tmp = train_feat[["primary_chronic", TARGET]].copy()
tmp["bin"] = pd.qcut(tmp[TARGET], 5, labels=False)
tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["bin"].astype(str)
strat = LabelEncoder().fit_transform(tmp["strat"])

model_specs = {
    "A_RMSE_full_d5": dict(
        loss_function="RMSE", eval_metric="MAE", depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=1.0
    ),
    "B_RMSE_pruned_d4": dict(
        loss_function="RMSE", eval_metric="MAE", depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=2.0
    ),
    "C_MAE_pruned_d4": dict(
        loss_function="MAE", eval_metric="MAE", depth=4, l2_leaf_reg=7, min_data_in_leaf=36,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=1.5
    )
}
cols_map = {"A_RMSE_full_d5": feat_full, "B_RMSE_pruned_d4": feat_pruned, "C_MAE_pruned_d4": feat_pruned}

oof_by_seed = {m: [] for m in model_specs}
test_by_seed = {m: [] for m in model_specs}

print("\n[Training] 5 Seeds x 7 Folds...")
for seed_idx in range(CFG.N_SEEDS):
    rs = SEED + seed_idx * 7
    kf = StratifiedKFold(CFG.N_FOLDS, shuffle=True, random_state=rs)
    
    s_oof = {m: np.zeros(len(y)) for m in model_specs}
    s_test = {m: np.zeros(len(test_feat)) for m in model_specs}
    
    for fold, (ti, vi) in enumerate(kf.split(train_feat, strat)):
        for m, p in model_specs.items():
            cols = cols_map[m]
            X_tr, y_tr = train_feat[cols].iloc[ti], y[ti]
            X_va, y_va = train_feat[cols].iloc[vi], y[vi]
            
            cb = CatBoostRegressor(**p, task_type="CPU", thread_count=-1, verbose=0, random_seed=rs+fold*99, early_stopping_rounds=CFG.ES_ROUNDS)
            cb.fit(X_tr, y_tr, eval_set=(X_va, y_va))
            
            s_oof[m][vi] = cb.predict(X_va)
            s_test[m] += cb.predict(test_feat[cols]) / CFG.N_FOLDS
            
    for m in model_specs:
        oof_by_seed[m].append(s_oof[m])
        test_by_seed[m].append(s_test[m])
        
    print(f"  Seed {seed_idx+1} MAE: " + " | ".join([f"{m}={mean_absolute_error(y, s_oof[m]):.2f}" for m in model_specs]))

# Averaging seeds
oof_avg = {m: np.mean(np.vstack(oof_by_seed[m]), axis=0) for m in model_specs}
test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in model_specs}

# -----------------------------
# Ensemble Selection (Iter7 Stable)
# -----------------------------
# We do stable search WITHOUT global shift first, then do refined shift later
baseline_tr = train_feat["baseline_next3y"].values
baseline_te = test_feat["baseline_next3y"].values
models = list(model_specs.keys())

best_ens = None
best_score = float("inf")

# Grid search for weights & lambda only (no shift yet)
print("\n[Ensemble] Searching weights...")
grid = np.arange(0.0, 1.01, CFG.W_STEP)
for wA in grid:
    for wB in grid:
        wC = 1.0 - wA - wB
        if wC < -1e-9: continue
        wC = max(0.0, wC)
        if abs(wA+wB+wC-1.0) > 1e-5: continue
        
        for lam in CFG.LAM_GRID:
            # Calc objective
            maes = []
            for s in range(CFG.N_SEEDS):
                p = wA*oof_by_seed[models[0]][s] + wB*oof_by_seed[models[1]][s] + wC*oof_by_seed[models[2]][s]
                p = (1-lam)*p + lam*baseline_tr
                maes.append(mean_absolute_error(y, p))
            
            mean_m = np.mean(maes)
            std_m = np.std(maes)
            obj = mean_m + CFG.STD_PEN*std_m + CFG.LAM_PEN*lam
            
            if obj < best_score:
                best_score = obj
                best_ens = (wA, wB, wC, lam)

wA, wB, wC, lam = best_ens
print(f"  Best Weights: {models[0]}={wA:.2f}, {models[1]}={wB:.2f}, {models[2]}={wC:.2f}")
print(f"  Best Lambda: {lam:.2f}")

# Construct raw ensemble
oof_raw = wA*oof_avg[models[0]] + wB*oof_avg[models[1]] + wC*oof_avg[models[2]]
oof_raw = (1-lam)*oof_raw + lam*baseline_tr
test_raw = wA*test_avg[models[0]] + wB*test_avg[models[1]] + wC*test_avg[models[2]]
test_raw = (1-lam)*test_raw + lam*baseline_te

print(f"  Ensemble MAE (Raw): {mean_absolute_error(y, oof_raw):.3f}")

# -----------------------------
# Post-Processing: Stratified Shift Optimization
# -----------------------------
# Instead of one global shift, we optimize 3 shifts for chronic conditions 0, 1, 2
# Objective: Minimize MAE on OOF

chronic_tr = train_feat["chronic_encoded"].values.astype(int) # -1,0,1,2
chronic_te = test_feat["chronic_encoded"].values.astype(int)

def shift_objective(shifts):
    # shifts is [s_neg1, s_0, s_1, s_2] or just [s_0, s_1, s_2]
    # We'll map -1 to 0 (default)
    s0, s1, s2 = shifts
    mapping = {-1:s0, 0:s0, 1:s1, 2:s2}
    
    # Vectorized application
    # Create shift vector
    shift_vec = np.zeros_like(oof_raw)
    for c_val, s_val in mapping.items():
        shift_vec[chronic_tr == c_val] = s_val
        
    return mean_absolute_error(y, oof_raw + shift_vec)

print("\n[Post-Process] Optimizing Stratified Shifts...")
# Initial guess: median residuals per group
resid = y - oof_raw
init_shifts = []
for g in [0, 1, 2]: # approximate groups
    r = resid[(chronic_tr == g) | (chronic_tr == -1)] if g==0 else resid[chronic_tr == g]
    init_shifts.append(np.median(r))

print(f"  Initial residual medians: {init_shifts}")
res = minimize(shift_objective, x0=init_shifts, method='Nelder-Mead', tol=1e-4)

opt_shifts = res.x
best_mae = res.fun
print(f"  Optimized Shifts: Group0/Unk={opt_shifts[0]:.4f}, Group1={opt_shifts[1]:.4f}, Group2={opt_shifts[2]:.4f}")
print(f"  Final Optimized OOF MAE: {best_mae:.3f}")

# Apply to Test
test_final = test_raw.copy()
shift_map = {-1:opt_shifts[0], 0:opt_shifts[0], 1:opt_shifts[1], 2:opt_shifts[2]}
for i in range(len(test_final)):
    c = chronic_te[i]
    test_final[i] += shift_map.get(c, 0.0)

# Clip
y_max = np.max(y)
test_final = np.clip(test_final, 0, y_max * 1.5)

# -----------------------------
# Submission
# -----------------------------
sub = pd.DataFrame({
    "patient_id": test_df["patient_id"].values,
    "ed_cost_next3y_usd": np.round(test_final, 2)
})
out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print("\n[Sanity Check]")
print("Submission Stats:")
print(sub[TARGET].describe())
print(f"\nDone. Saved to {OUT_SUB_PATH}")
print("Paste back: (1) Final Optimized OOF MAE, (2) Optimized Shifts values.")

ITERATION 9 | Back to Iter7 (Winner) + Automated Stratified Shift Optimization
Logic: Iter7 (MAE~451) was best. We refine it by optimizing shifts per chronic group.

[load] Reading data...
Features: Full=40, Pruned=30

[Training] 5 Seeds x 7 Folds...
  Seed 1 MAE: A_RMSE_full_d5=429.64 | B_RMSE_pruned_d4=432.67 | C_MAE_pruned_d4=436.64
  Seed 2 MAE: A_RMSE_full_d5=432.63 | B_RMSE_pruned_d4=432.56 | C_MAE_pruned_d4=437.97
  Seed 3 MAE: A_RMSE_full_d5=429.25 | B_RMSE_pruned_d4=431.26 | C_MAE_pruned_d4=439.05
  Seed 4 MAE: A_RMSE_full_d5=430.80 | B_RMSE_pruned_d4=431.66 | C_MAE_pruned_d4=437.34
  Seed 5 MAE: A_RMSE_full_d5=430.62 | B_RMSE_pruned_d4=429.18 | C_MAE_pruned_d4=443.23

[Ensemble] Searching weights...
  Best Weights: A_RMSE_full_d5=0.60, B_RMSE_pruned_d4=0.40, C_MAE_pruned_d4=0.00
  Best Lambda: 0.00
  Ensemble MAE (Raw): 427.638

[Post-Process] Optimizing Stratified Shifts...
  Initial residual medians: [np.float64(-22.904879716014875), np.float64(-11.996815943285583), np.floa

# Iteration 3
- 450.5367

In [9]:
# === ITERATION 10: The "Hybrid-Log" Ensemble (Raw RMSE + Log RMSE + Stratified Shift) ===
# Core Logic:
#   1. Features: Strictly KEEP Iter 7/9 clean features (proven robust).
#   2. Models: Add "Model D" (Log-Target RMSE) to the ensemble.
#      - Model A (Raw): Good at absolute scale.
#      - Model B (Raw): Stable, pruned features.
#      - Model D (Log): Good at relative structure, uncorrelated errors.
#   3. Post-Process: Keep the automated Stratified Shift Optimization to fix Log-model bias.

import os, re, sys, gc, math, warnings, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
TARGET = "ed_cost_next3y_usd"

print("="*90)
print("ITERATION 10 | Hybrid-Log Ensemble: Adding Log-Target Model for Variance Reduction")
print("Goal: Break 450 barrier by mixing Raw-Target and Log-Target models + Stratified Shift.")
print("="*90)

# -----------------------------
# Deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Config
# -----------------------------
class CFG:
    N_FOLDS = 7
    N_SEEDS = 5 # Keep 5 seeds for stability
    ITERS = 3000
    ES_ROUNDS = 120
    LR = 0.03
    RSM = 0.80
    
    # Ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.02, 0.05] # Simplified, usually 0 or small is best
    
    # Penalties
    STD_PEN = 0.20
    LAM_PEN = 5.0

# -----------------------------
# Utilities
# -----------------------------
def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan": return None
    if re.fullmatch(r"\d+\.0+", s): s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)): return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s: return None
    return s.zfill(3)

# -----------------------------
# Feature Engineering (Strictly Iter 7/9 Version - No Changes)
# -----------------------------
def load_admissions_features(adm_train_path, adm_test_path):
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns: df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs: return None
    adm = pd.concat(dfs, ignore_index=True)
    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")
    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()
    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    return out

def build_pdf_features_from_lineitems(li):
    li = li.copy()
    code_col = next((c for c in ["code","cpt","cpt_code","hcpcs","proc_code"] if c in li.columns), None)
    total_col = next((c for c in ["line_total","line_total_usd","total","amount"] if c in li.columns), None)
    if not code_col or not total_col or "patient_id" not in li.columns: return None

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)
    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()
    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).clip(lower=0.0)

    total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)

    # Buckets
    is_em = li["code"].isin(["99281","99282","99283","99284","99285"])
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)
    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])
    
    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    share = (li["amt"] / denom).fillna(0.0)
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")
    has_obs = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    def has_c(c, n): return (li["code"].eq(c).astype(int).groupby(li["patient_id"]).max()).rename(n)
    has_99285 = has_c("99285","has_99285")
    
    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    out = pd.concat([n_unique_codes, cost_hhi, n_em_codes, max_em_level, has_critical_care, has_high_acuity, has_imaging, has_obs, has_99285, total], axis=1).reset_index()
    for s in [em_total, crit_total, proc_total, high_total]: out = out.merge(s.reset_index(), on="patient_id", how="left")
    for c in ["em_total","crit_total","proc_total","high_total","receipt_total"]:
        if c not in out.columns: out[c] = 0.0
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
        
    d2 = out["receipt_total"].replace(0.0, np.nan)
    out["pct_cost_critical"] = (out["crit_total"] / d2).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / d2).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / d2).fillna(0.0)
    out["cost_per_em"] = np.where(out["n_em_codes"]>0, out["receipt_total"]/out["n_em_codes"].clip(lower=1), out["receipt_total"])
    out["n_high_acuity_total"] = (out["has_high_acuity"].fillna(0) + out["has_critical_care"].fillna(0)).astype(int)

    return out

def load_receipts(path):
    if not os.path.exists(path): return None
    try:
        d = joblib_load(path)
        if isinstance(d, dict):
            for k in ["lineitems_df","lineitems","items_df"]:
                if k in d and isinstance(d[k], pd.DataFrame): return build_pdf_features_from_lineitems(d[k])
        if isinstance(d, pd.DataFrame): return build_pdf_features_from_lineitems(d)
    except: pass
    return None

def build_features(ed, pts, adm, pdf):
    f = ed.copy()
    cmap = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    f["primary_chronic"] = f["primary_chronic"].astype(str)
    f["chronic_encoded"] = f["primary_chronic"].str.upper().map(cmap).fillna(-1).astype(float)
    
    f["prior_ed_visits_5y"] = pd.to_numeric(f["prior_ed_visits_5y"], errors="coerce").fillna(0.0)
    f["prior_ed_cost_5y_usd"] = pd.to_numeric(f["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0)
    
    f["sqrt_prior_cost"] = np.sqrt(f["prior_ed_cost_5y_usd"].clip(lower=0))
    f["log_prior_cost"] = np.log1p(f["prior_ed_cost_5y_usd"].clip(lower=0))
    f["log_visits"] = np.log1p(f["prior_ed_visits_5y"].clip(lower=0))
    f["cost_per_visit"] = f["prior_ed_cost_5y_usd"] / f["prior_ed_visits_5y"].clip(lower=1)
    f["prior_cost_cap20k"] = f["prior_ed_cost_5y_usd"].clip(upper=20000)
    f["log_prior_cost_cap20k"] = np.log1p(f["prior_cost_cap20k"])
    f["baseline_next3y"] = f["prior_ed_cost_5y_usd"] * 0.6
    
    p = pts.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper()=="M").astype(int)
    ins_map = {"private":2,"public":1,"self_pay":0,"selfpay":0}
    p["insurance_encoded"] = p["insurance"].astype(str).str.lower().map(ins_map).fillna(-1)
    z = p["zip3"].apply(standardize_zip3).astype(str).str[0]
    p["zip_region"] = pd.to_numeric(z, errors="coerce").fillna(-1)
    
    f = f.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    f["ins_x_chronic"] = f["insurance_encoded"] * f["chronic_encoded"]
    
    if adm is not None:
        f = f.merge(adm, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in f.columns: f[c] = f[c].fillna(f[c].median())
    if pdf is not None:
        f = f.merge(pdf, on="patient_id", how="left")
        for c in pdf.columns:
            if c!="patient_id" and pd.api.types.is_numeric_dtype(pdf[c]): f[c] = f[c].fillna(f[c].median())

    if "pct_cost_critical" in f.columns: f["logprior_x_pctcritical"] = f["log_prior_cost"] * f["pct_cost_critical"]
    if "n_high_acuity_total" in f.columns: f["logprior_x_highacu"] = f["log_prior_cost"] * f["n_high_acuity_total"]
    if "n_unique_codes" in f.columns: f["cost_per_code"] = f["prior_ed_cost_5y_usd"] / f["n_unique_codes"].clip(lower=1)

    for c in f.columns:
        if c not in ["patient_id","primary_chronic",TARGET]: f[c] = pd.to_numeric(f[c], errors="coerce").fillna(0.0)
    return f

# -----------------------------
# Execution
# -----------------------------
print("\n[load] Reading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
pts_df = pd.read_csv(PATIENTS_PATH)
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
pdf_df = load_receipts(RECEIPTS_JOBLIB_PATH)

for d in [train_df, test_df, pts_df]: d["patient_id"] = pd.to_numeric(d["patient_id"], errors="coerce").astype(int)

train_feat = build_features(train_df, pts_df, adm_df, pdf_df)
test_feat = build_features(test_df, pts_df, adm_df, pdf_df)

pruned_cols = [
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k","cost_per_visit","log_visits",
    "baseline_next3y", "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    "charlson_max","charlson_mean","pct_emergent",
    "cost_per_em","cost_hhi","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","n_unique_codes",
    "logprior_x_pctcritical","logprior_x_highacu","cost_per_code"
]
all_cols = [c for c in train_feat.columns if c not in ["patient_id","primary_chronic",TARGET,"sex","insurance","zip3"] and pd.api.types.is_numeric_dtype(train_feat[c])]
feat_full = [c for c in all_cols if train_feat[c].nunique() > 1]
feat_pruned = [c for c in pruned_cols if c in train_feat.columns and train_feat[c].nunique() > 1]
print(f"Features: Full={len(feat_full)}, Pruned={len(feat_pruned)}")

# -----------------------------
# Hybrid-Log Training
# -----------------------------
y = train_feat[TARGET].values.astype(float)
y_log = np.log1p(y)

# Stratification
tmp = train_feat[["primary_chronic", TARGET]].copy()
tmp["bin"] = pd.qcut(tmp[TARGET], 5, labels=False)
tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["bin"].astype(str)
strat = LabelEncoder().fit_transform(tmp["strat"])

# Model Specs: A (Raw), B (Raw), D (Log)
model_specs = {
    "A_RMSE_full_d5": dict(
        loss_function="RMSE", eval_metric="MAE", depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=1.0, 
        early_stopping_rounds=CFG.ES_ROUNDS
    ),
    "B_RMSE_pruned_d4": dict(
        loss_function="RMSE", eval_metric="MAE", depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=2.0,
        early_stopping_rounds=CFG.ES_ROUNDS
    ),
    "D_LOG_pruned_d4": dict(
        loss_function="RMSE", eval_metric="MAE", depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=1.5,
        early_stopping_rounds=CFG.ES_ROUNDS
    )
}
# Log model uses "pruned" features for stability
cols_map = {"A_RMSE_full_d5": feat_full, "B_RMSE_pruned_d4": feat_pruned, "D_LOG_pruned_d4": feat_pruned}

oof_by_seed = {m: [] for m in model_specs}
test_by_seed = {m: [] for m in model_specs}

print("\n[Training] 5 Seeds x 7 Folds (Hybrid Raw/Log)...")
for seed_idx in range(CFG.N_SEEDS):
    rs = SEED + seed_idx * 7
    kf = StratifiedKFold(CFG.N_FOLDS, shuffle=True, random_state=rs)
    
    s_oof = {m: np.zeros(len(y)) for m in model_specs}
    s_test = {m: np.zeros(len(test_feat)) for m in model_specs}
    
    for fold, (ti, vi) in enumerate(kf.split(train_feat, strat)):
        for m, p in model_specs.items():
            cols = cols_map[m]
            X_tr = train_feat[cols].iloc[ti]
            X_va = train_feat[cols].iloc[vi]
            
            # Target handling: Raw vs Log
            is_log = (m == "D_LOG_pruned_d4")
            y_tr_cur = y_log[ti] if is_log else y[ti]
            y_va_cur = y_log[vi] if is_log else y[vi]
            
            cb = CatBoostRegressor(**p, task_type="CPU", thread_count=-1, verbose=0, random_seed=rs+fold*99)
            cb.fit(X_tr, y_tr_cur, eval_set=(X_va, y_va_cur))
            
            # Predict & Invert if needed
            pred_va = cb.predict(X_va)
            pred_te = cb.predict(test_feat[cols])
            
            if is_log:
                pred_va = np.expm1(pred_va)
                pred_te = np.expm1(pred_te)
            
            s_oof[m][vi] = np.clip(pred_va, 0, None)
            s_test[m] += np.clip(pred_te, 0, None) / CFG.N_FOLDS
            
    for m in model_specs:
        oof_by_seed[m].append(s_oof[m])
        test_by_seed[m].append(s_test[m])
        
    print(f"  Seed {seed_idx+1} MAE: " + " | ".join([f"{m}={mean_absolute_error(y, s_oof[m]):.2f}" for m in model_specs]))

oof_avg = {m: np.mean(np.vstack(oof_by_seed[m]), axis=0) for m in model_specs}
test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in model_specs}

# -----------------------------
# Ensemble Selection
# -----------------------------
# Models: A (Raw), B (Raw), D (Log)
baseline_tr = train_feat["baseline_next3y"].values
baseline_te = test_feat["baseline_next3y"].values
models = ["A_RMSE_full_d5", "B_RMSE_pruned_d4", "D_LOG_pruned_d4"]

best_ens = None
best_score = float("inf")

print("\n[Ensemble] Searching weights (Raw A + Raw B + Log D)...")
grid = np.arange(0.0, 1.01, CFG.W_STEP)
for wA in grid:
    for wB in grid:
        wD = 1.0 - wA - wB
        if wD < -1e-9: continue
        wD = max(0.0, wD)
        if abs(wA+wB+wD-1.0) > 1e-5: continue
        
        for lam in CFG.LAM_GRID:
            # Objective: Minimize Mean OOF MAE + Penalty on Variance across seeds
            # This is crucial for hybrid ensembles to ensure the Log model doesn't destabilize
            maes = []
            for s in range(CFG.N_SEEDS):
                p = wA*oof_by_seed[models[0]][s] + wB*oof_by_seed[models[1]][s] + wD*oof_by_seed[models[2]][s]
                p = (1-lam)*p + lam*baseline_tr
                maes.append(mean_absolute_error(y, p))
            
            mean_m = np.mean(maes)
            std_m = np.std(maes)
            obj = mean_m + CFG.STD_PEN*std_m + CFG.LAM_PEN*lam
            
            if obj < best_score:
                best_score = obj
                best_ens = (wA, wB, wD, lam)

wA, wB, wD, lam = best_ens
print(f"  Best Weights: A(Raw)={wA:.2f}, B(Raw)={wB:.2f}, D(Log)={wD:.2f}")
print(f"  Best Lambda: {lam:.2f}")

# Construct Raw Ensemble (Before Shift)
oof_raw = wA*oof_avg[models[0]] + wB*oof_avg[models[1]] + wD*oof_avg[models[2]]
oof_raw = (1-lam)*oof_raw + lam*baseline_tr
test_raw = wA*test_avg[models[0]] + wB*test_avg[models[1]] + wD*test_avg[models[2]]
test_raw = (1-lam)*test_raw + lam*baseline_te

print(f"  Ensemble MAE (Pre-Shift): {mean_absolute_error(y, oof_raw):.3f}")

# -----------------------------
# Post-Processing: Stratified Shift Optimization
# -----------------------------
# This is even MORE important now because Log models often have a bias after inversion (Jensen's inequality)
chronic_tr = train_feat["chronic_encoded"].values.astype(int)
chronic_te = test_feat["chronic_encoded"].values.astype(int)

def shift_objective(shifts):
    s0, s1, s2 = shifts
    mapping = {-1:s0, 0:s0, 1:s1, 2:s2}
    shift_vec = np.zeros_like(oof_raw)
    for c_val, s_val in mapping.items():
        shift_vec[chronic_tr == c_val] = s_val
    return mean_absolute_error(y, oof_raw + shift_vec)

print("\n[Post-Process] Optimizing Stratified Shifts...")
resid = y - oof_raw
init_shifts = []
for g in [0, 1, 2]:
    r = resid[(chronic_tr == g) | (chronic_tr == -1)] if g==0 else resid[chronic_tr == g]
    init_shifts.append(np.median(r))

print(f"  Initial residuals: {init_shifts}")
res = minimize(shift_objective, x0=init_shifts, method='Nelder-Mead', tol=1e-4)

opt_shifts = res.x
best_mae = res.fun
print(f"  Optimized Shifts: G0={opt_shifts[0]:.4f}, G1={opt_shifts[1]:.4f}, G2={opt_shifts[2]:.4f}")
print(f"  Final Optimized OOF MAE: {best_mae:.3f}")

# Apply to Test
test_final = test_raw.copy()
shift_map = {-1:opt_shifts[0], 0:opt_shifts[0], 1:opt_shifts[1], 2:opt_shifts[2]}
for i in range(len(test_final)):
    c = chronic_te[i]
    test_final[i] += shift_map.get(c, 0.0)

test_final = np.clip(test_final, 0, np.max(y) * 1.5)

# -----------------------------
# Submission
# -----------------------------
sub = pd.DataFrame({
    "patient_id": test_df["patient_id"].values,
    "ed_cost_next3y_usd": np.round(test_final, 2)
})
out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print("\n[Sanity Check]")
print(sub[TARGET].describe())
print(f"\nDone. Saved to {OUT_SUB_PATH}")
print("Paste back: (1) Best Weights, (2) Final Optimized OOF MAE, (3) Shifts.")

ITERATION 10 | Hybrid-Log Ensemble: Adding Log-Target Model for Variance Reduction
Goal: Break 450 barrier by mixing Raw-Target and Log-Target models + Stratified Shift.

[load] Reading data...
Features: Full=40, Pruned=30

[Training] 5 Seeds x 7 Folds (Hybrid Raw/Log)...
  Seed 1 MAE: A_RMSE_full_d5=429.64 | B_RMSE_pruned_d4=432.67 | D_LOG_pruned_d4=435.51
  Seed 2 MAE: A_RMSE_full_d5=432.63 | B_RMSE_pruned_d4=432.56 | D_LOG_pruned_d4=440.53
  Seed 3 MAE: A_RMSE_full_d5=429.25 | B_RMSE_pruned_d4=431.26 | D_LOG_pruned_d4=442.36
  Seed 4 MAE: A_RMSE_full_d5=430.80 | B_RMSE_pruned_d4=431.66 | D_LOG_pruned_d4=439.34
  Seed 5 MAE: A_RMSE_full_d5=430.62 | B_RMSE_pruned_d4=429.18 | D_LOG_pruned_d4=441.54

[Ensemble] Searching weights (Raw A + Raw B + Log D)...
  Best Weights: A(Raw)=0.60, B(Raw)=0.40, D(Log)=0.00
  Best Lambda: 0.00
  Ensemble MAE (Pre-Shift): 427.638

[Post-Process] Optimizing Stratified Shifts...
  Initial residuals: [np.float64(-22.904879716014875), np.float64(-11.9968159

# Iteration 4
- 450.7318

In [11]:
# === ITERATION 8: Tail+Calibration++ (LB-oriented, practical, fast) ===
# Upgrades vs Iter7:
#   (1) Add ONE diverse model: log1p-target CatBoost (tail friendly)
#   (2) Replace leakage-prone group shift with CROSS-FIT group-median calibration (chronic x insurance [+ charlson_bin])
#   (3) Add a few ultra-stable low-dim receipt/admission stats (n_line_items, top1_share, entropy, max_line, n_adm ...)
# Output: D:\AgentDs\agent_ds_healthcare\submission.csv

import os, re, sys, gc, math, warnings, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Paths (must match prompt)
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"  # last resort only (we will NOT parse)
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
TARGET = "ed_cost_next3y_usd"

print("="*92)
print("ITERATION 8 | Tail+Calibration++: +log1p model + cross-fit group calibration + stable stats")
print("Goal: push LB below 450 by reducing tail bias + calibration leakage + improving systematic bias correction.")
print("="*92)

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor


# -----------------------------
# Config (keep fast, but add 1 model => keep runtime similar via folds/seeds balance)
# -----------------------------
class CFG:
    # Keep runtime similar after adding 4th model
    N_FOLDS = 5
    N_SEEDS = 6

    ITERS = 3000
    ES_ROUNDS = 140
    LR = 0.03
    RSM = 0.80

    # extra regularization (helps generalization, usually LB-positive)
    BOOTSTRAP_TYPE = "Bernoulli"
    SUBSAMPLE = 0.85

    # ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.03, 0.05, 0.08, 0.10, 0.15]  # baseline blend (small)
    SHIFT_GRID = [0.0, 0.5, 1.0]  # global median shift multiplier

    # stability penalties
    STD_PEN = 0.20
    LAM_PEN = 8.0
    SHIFT_PEN = 0.002


# -----------------------------
# Utilities
# -----------------------------
def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

def safe_clip_nonneg(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    a[a < 0] = 0.0
    return a

# -----------------------------
# Admissions features (add a couple stable counts)
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs:
        return None

    adm = pd.concat(dfs, ignore_index=True)
    need = ["patient_id","charlson_band","acuity_emergent"]
    if not all(c in adm.columns for c in need):
        return None

    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")
    adm = adm.dropna(subset=["patient_id"]).copy()
    adm["patient_id"] = adm["patient_id"].astype(int)

    out = adm.groupby("patient_id").agg(
        n_adm=("patient_id","size"),
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        charlson_sum=("charlson_band","sum"),
        pct_emergent=("acuity_emergent","mean"),
        n_emergent=("acuity_emergent","sum"),
    ).reset_index()

    for c in ["n_adm","charlson_max","charlson_mean","charlson_sum","pct_emergent","n_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    return out

# -----------------------------
# Low-dim receipts features from parsed lineitems
# Add: n_line_items, mean/std/max amt, top1_share, entropy
# -----------------------------
def build_pdf_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()

    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code"]:
        if c in li.columns:
            code_col = c
            break
    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"]:
        if c in li.columns:
            total_col = c
            break
    if code_col is None or total_col is None or "patient_id" not in li.columns:
        raise ValueError("Lineitems DF missing required columns.")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()

    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    li.loc[li["amt"] < 0, "amt"] = 0.0

    # totals
    total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)

    # code numeric where possible
    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")

    # buckets
    is_em = li["code"].isin(["99281","99282","99283","99284","99285"])
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)

    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])  # airway/lines/chest tube/cpr

    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    # per-line shares
    share = (li["amt"] / denom).fillna(0.0)
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    entropy = (-(share * np.log(share + 1e-12))).groupby(li["patient_id"]).sum().rename("cost_entropy")
    top1_share = li.groupby("patient_id").apply(lambda g: float(g["amt"].max() / (g["amt"].sum() + 1e-12))).rename("top1_share")

    # stable amount stats
    n_line_items = li.groupby("patient_id").size().rename("n_line_items")
    mean_line_amt = li.groupby("patient_id")["amt"].mean().rename("mean_line_amt")
    std_line_amt  = li.groupby("patient_id")["amt"].std(ddof=0).fillna(0.0).rename("std_line_amt")
    max_line_amt  = li.groupby("patient_id")["amt"].max().rename("max_line_amt")

    # basic counts
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")

    # EM stats
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    sum_em_level = (em_level * is_em.astype(int)).groupby(li["patient_id"]).sum().rename("sum_em_level")
    avg_em_level = (sum_em_level / n_em_codes.replace(0, np.nan)).fillna(0.0).rename("avg_em_level")
    n_high_em = ((em_level >= 4) & is_em).astype(int).groupby(li["patient_id"]).sum().rename("n_high_em")

    # totals by bucket
    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    img_total = li.loc[is_imaging].groupby("patient_id")["amt"].sum().rename("img_total")
    lab_total = li.loc[is_lab].groupby("patient_id")["amt"].sum().rename("lab_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    # counts by bucket
    n_procedures = is_proc_any.astype(int).groupby(li["patient_id"]).sum().rename("n_procedures")
    n_imaging = is_imaging.astype(int).groupby(li["patient_id"]).sum().rename("n_imaging")
    n_lab = is_lab.astype(int).groupby(li["patient_id"]).sum().rename("n_lab")

    # flags
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_observation = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")

    def has_code(code: str, name: str):
        return (li["code"].eq(code).astype(int).groupby(li["patient_id"]).max()).rename(name)

    has_intub_31500 = has_code("31500","has_intub_31500")
    has_cvc_36556 = has_code("36556","has_cvc_36556")
    has_cpr_92950 = has_code("92950","has_cpr_92950")
    has_artline_36620 = has_code("36620","has_artline_36620")
    has_ct_head_70450 = has_code("70450","has_ct_head_70450")
    has_99285 = has_code("99285","has_99285")
    has_ct_abdpel_74177 = has_code("74177","has_ct_abdpel_74177")
    has_troponin_84484 = has_code("84484","has_troponin_84484")
    has_obs_G0378 = has_code("G0378","has_obs_G0378")

    out = pd.concat([
        n_unique_codes,
        n_line_items, mean_line_amt, std_line_amt, max_line_amt,
        top1_share, entropy, cost_hhi,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_procedures, n_imaging, n_lab,
        total
    ], axis=1).reset_index()

    # merge totals (may be missing)
    for s in [em_total, crit_total, proc_total, img_total, lab_total, high_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")

    for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"]:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    denom2 = out["receipt_total"].replace(0.0, np.nan)
    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_imaging"] = (out["img_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_lab"] = (out["lab_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    out["cost_per_em"] = np.where(out["n_em_codes"] > 0, out["receipt_total"] / out["n_em_codes"].clip(lower=1), out["receipt_total"])

    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int)
        + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int)
        + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    # drop helper totals
    out.drop(columns=[c for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"] if c in out.columns], inplace=True)

    # fill numeric
    for c in out.columns:
        if c == "patient_id":
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    return out

def load_receipts_joblib(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    data = joblib_load(joblib_path)

    if isinstance(data, dict):
        for k in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if k in data and isinstance(data[k], pd.DataFrame):
                return build_pdf_features_from_lineitems(data[k])
        try:
            df = pd.DataFrame.from_dict(data, orient="index")
            df.index.name = "patient_id"
            df = df.reset_index()
            return df
        except Exception:
            return None

    if isinstance(data, pd.DataFrame):
        df = data
        if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
            return build_pdf_features_from_lineitems(df)
        return df

    if isinstance(data, (list, tuple)):
        dfs = [x for x in data if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
                return build_pdf_features_from_lineitems(df)
        for df in dfs:
            if "patient_id" in df.columns:
                return df

    return None


# -----------------------------
# Feature engineering (numeric-only, keep v3 spirit)
# -----------------------------
def build_features(ed_df: pd.DataFrame,
                   patients_df: pd.DataFrame,
                   adm_df: Optional[pd.DataFrame],
                   pdf_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    feat = ed_df.copy()

    chronic_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    feat["prior_ed_visits_5y"] = pd.to_numeric(feat["prior_ed_visits_5y"], errors="coerce").fillna(0.0)
    feat["prior_ed_cost_5y_usd"] = pd.to_numeric(feat["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0)

    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost_cap20k"] = np.log1p(feat["prior_cost_cap20k"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)

    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    # patients
    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce")
    p["age"] = p["age"].fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)

    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)
    p["insurance_cat"] = p["insurance_encoded"].fillna(-1).astype(int)

    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    # a stable age bin (helps calibration & sometimes model)
    p["age_bin"] = pd.cut(p["age"], bins=[0,30,50,70,200], labels=False, include_lowest=True).astype(float).fillna(0).astype(int)

    feat = feat.merge(p[["patient_id","age","age_bin","sex_encoded","insurance_encoded","insurance_cat","zip_region"]],
                      on="patient_id", how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    # admissions
    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in ["n_adm","charlson_max","charlson_mean","charlson_sum","pct_emergent","n_emergent"]:
            if c in feat.columns:
                feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(feat[c].median() if not feat[c].isna().all() else 0.0)
        if "charlson_max" in feat.columns:
            feat["charlson_bin"] = np.clip(np.round(feat["charlson_max"]).astype(int), 0, 3)
        else:
            feat["charlson_bin"] = 0
    else:
        feat["charlson_bin"] = 0

    # receipts
    if pdf_df is not None:
        feat = feat.merge(pdf_df, on="patient_id", how="left")
        for c in pdf_df.columns:
            if c == "patient_id": continue
            feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(feat[c].median() if not feat[c].isna().all() else 0.0)

    # light interactions
    if "pct_cost_critical" in feat.columns:
        feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * feat["pct_cost_critical"]
    if "n_high_acuity_total" in feat.columns:
        feat["logprior_x_highacu"] = feat["log_prior_cost"] * feat["n_high_acuity_total"]

    # stable ratio
    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)

    # numeric safety
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in ["patient_id", "primary_chronic", TARGET]:
            continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median() if not feat[c].isna().all() else 0.0)

    return feat

def get_numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3"}
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def drop_constants(cols: List[str], df: pd.DataFrame) -> List[str]:
    out = []
    for c in cols:
        if c not in df.columns:
            continue
        if df[c].nunique(dropna=False) <= 1:
            continue
        out.append(c)
    return out

def make_strat_labels(df_train_feat: pd.DataFrame, y: np.ndarray) -> np.ndarray:
    tmp = df_train_feat[["primary_chronic"]].copy()
    tmp["y"] = y
    tmp["cost_bin"] = pd.qcut(tmp["y"], q=5, labels=False, duplicates="drop")
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"].values)
    return strat


# -----------------------------
# Training (multi-seed + fold bagging) with 4 models (add log1p)
# -----------------------------
def train_models(train_feat: pd.DataFrame, test_feat: pd.DataFrame,
                 feat_full: List[str], feat_pruned: List[str]) -> Tuple[Dict[str, List[np.ndarray]], Dict[str, List[np.ndarray]], Dict[str, List[int]]]:

    y = train_feat[TARGET].values.astype(float)
    strat = make_strat_labels(train_feat, y)

    # 4 models: add one log1p target model
    model_specs = {
        "A_RMSE_full_d5": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.0,
            bootstrap_type=CFG.BOOTSTRAP_TYPE, subsample=CFG.SUBSAMPLE,
        ),
        "B_RMSE_pruned_d4": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=2.0,
            bootstrap_type=CFG.BOOTSTRAP_TYPE, subsample=CFG.SUBSAMPLE,
        ),
        "C_MAE_pruned_d4": dict(
            loss_function="MAE", eval_metric="MAE",
            depth=4, l2_leaf_reg=7, min_data_in_leaf=36,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.5,
            bootstrap_type=CFG.BOOTSTRAP_TYPE, subsample=CFG.SUBSAMPLE,
        ),
        "D_LOGRMSE_pruned_d5": dict(
            loss_function="RMSE", eval_metric="RMSE",   # in log-space
            depth=5, l2_leaf_reg=6, min_data_in_leaf=30,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.3,
            bootstrap_type=CFG.BOOTSTRAP_TYPE, subsample=CFG.SUBSAMPLE,
        ),
    }
    model_featcols = {
        "A_RMSE_full_d5": feat_full,
        "B_RMSE_pruned_d4": feat_pruned,
        "C_MAE_pruned_d4": feat_pruned,
        "D_LOGRMSE_pruned_d5": feat_pruned,
    }
    model_target_mode = {
        "A_RMSE_full_d5": "raw",
        "B_RMSE_pruned_d4": "raw",
        "C_MAE_pruned_d4": "raw",
        "D_LOGRMSE_pruned_d5": "log1p",
    }

    oof_by_seed = {m: [] for m in model_specs.keys()}
    test_by_seed = {m: [] for m in model_specs.keys()}
    best_iters = {m: [] for m in model_specs.keys()}

    print("\n[training] CatBoost CPU | shallow + rsm + subsample | multi-seed bagging")
    print("Models:", list(model_specs.keys()))
    print(f"Seeds={CFG.N_SEEDS}, Folds={CFG.N_FOLDS}\n")

    for seed_idx in range(CFG.N_SEEDS):
        rs = SEED + seed_idx * 7
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs.keys()}
        test_seed = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs.keys()}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat), 1):
            for mname, params in model_specs.items():
                cols = model_featcols[mname]
                mode = model_target_mode[mname]

                X_tr = train_feat[cols].iloc[ti]
                y_tr = y[ti]
                X_va = train_feat[cols].iloc[vi]
                y_va = y[vi]
                X_te = test_feat[cols]

                if mode == "log1p":
                    y_tr_fit = np.log1p(y_tr)
                    y_va_fit = np.log1p(y_va)
                else:
                    y_tr_fit = y_tr
                    y_va_fit = y_va

                cb = CatBoostRegressor(
                    **params,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=rs + fold * 31 + (hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )
                cb.fit(X_tr, y_tr_fit, eval_set=(X_va, y_va_fit), verbose=0)

                try:
                    best_iters[mname].append(int(cb.get_best_iteration()))
                except Exception:
                    pass

                pred_va = cb.predict(X_va)
                pred_te = cb.predict(X_te)

                if mode == "log1p":
                    pred_va = np.expm1(pred_va)
                    pred_te = np.expm1(pred_te)

                pred_va = safe_clip_nonneg(pred_va)
                pred_te = safe_clip_nonneg(pred_te)

                oof_seed[mname][vi] = pred_va
                test_seed[mname] += pred_te / CFG.N_FOLDS

                del cb
                gc.collect()

        # per-seed MAE on RAW scale (important)
        seed_maes = {m: float(mean_absolute_error(y, oof_seed[m])) for m in model_specs.keys()}
        print(f"  Seed {seed_idx+1}/{CFG.N_SEEDS} OOF MAE: " + " | ".join([f"{m}={seed_maes[m]:.2f}" for m in model_specs.keys()]))

        for m in model_specs.keys():
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed[m])

    print("\n[seed-averaged OOF MAE per model]")
    for m in oof_by_seed.keys():
        avg_oof = np.mean(np.vstack(oof_by_seed[m]), axis=0)
        print(f"  {m:20s}: {mean_absolute_error(y, avg_oof):.2f}")

    print("\n[median best_iteration per model] (for reference)")
    for m in best_iters.keys():
        if best_iters[m]:
            print(f"  {m:20s}: {int(np.median(best_iters[m]))}")
        else:
            print(f"  {m:20s}: (n/a)")

    return oof_by_seed, test_by_seed, best_iters


# -----------------------------
# Ensemble selection (stability across seeds) — generic N-model simplex grid
# -----------------------------
def stable_ensemble_search(train_feat: pd.DataFrame,
                           oof_by_seed: Dict[str, List[np.ndarray]],
                           test_by_seed: Dict[str, List[np.ndarray]],
                           baseline_oof: np.ndarray,
                           baseline_test: np.ndarray) -> Tuple[np.ndarray, np.ndarray, Dict]:
    y = train_feat[TARGET].values.astype(float)
    model_names = list(oof_by_seed.keys())
    n_models = len(model_names)

    # seed-avg
    oof_avg = {m: np.mean(np.vstack(oof_by_seed[m]), axis=0) for m in model_names}
    test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in model_names}

    # stack for speed
    OOF_AVG = np.vstack([oof_avg[m] for m in model_names])               # (n_models, n_train)
    TEST_AVG = np.vstack([test_avg[m] for m in model_names])             # (n_models, n_test)
    OOF_SEEDS = [np.vstack([oof_by_seed[m][s] for m in model_names]) for s in range(CFG.N_SEEDS)]  # list of (n_models, n_train)

    # integer simplex grid
    step = CFG.W_STEP
    units = int(round(1.0 / step))
    def compositions(total, k):
        if k == 1:
            yield (total,)
        else:
            for i in range(total + 1):
                for rest in compositions(total - i, k - 1):
                    yield (i,) + rest

    best = None
    top_list = []

    for w_int in compositions(units, n_models):
        w = np.array(w_int, dtype=float) / units
        if abs(w.sum() - 1.0) > 1e-9:
            continue

        # precompute base avg pred
        pred_avg_base = w @ OOF_AVG

        for lam in CFG.LAM_GRID:
            # baseline blend on avg
            pred_avg = (1.0 - lam) * pred_avg_base + lam * baseline_oof
            for sm in CFG.SHIFT_GRID:
                shift = float(np.median(y - pred_avg)) * sm

                maes = []
                for s in range(CFG.N_SEEDS):
                    pred = w @ OOF_SEEDS[s]
                    pred = (1.0 - lam) * pred + lam * baseline_oof
                    pred = pred + shift
                    maes.append(float(mean_absolute_error(y, pred)))

                mean_m = float(np.mean(maes))
                std_m  = float(np.std(maes, ddof=0))
                obj = mean_m + CFG.STD_PEN * std_m + CFG.LAM_PEN * lam + CFG.SHIFT_PEN * abs(sm)

                rec = (obj, mean_m, std_m, w.copy(), float(lam), float(sm), float(shift))
                top_list.append(rec)
                if best is None or obj < best[0]:
                    best = rec

    top_list.sort(key=lambda x: x[0])
    print("\n[ensemble search] Top candidates by robust objective (mean + std + simplicity penalties):")
    for i, rec in enumerate(top_list[:10], 1):
        obj, mean_m, std_m, w, lam, sm, shift = rec
        w_str = ",".join([f"{x:.2f}" for x in w])
        print(f"  #{i:02d} obj={obj:.3f} meanMAE={mean_m:.3f} std={std_m:.3f} | w=({w_str}) | lam={lam:.2f} | shift_mult={sm:.1f}")

    obj, mean_m, std_m, w, lam, sm, shift = best

    oof_final = w @ OOF_AVG
    test_final = w @ TEST_AVG

    oof_final = (1.0 - lam) * oof_final + lam * baseline_oof
    test_final = (1.0 - lam) * test_final + lam * baseline_test

    oof_final = oof_final + shift
    test_final = test_final + shift

    meta = {
        "models_order": model_names,
        "weights": tuple(float(x) for x in w),
        "lam_baseline": float(lam),
        "shift_mult": float(sm),
        "shift_value": float(shift),
        "oof_mean_mae_across_seeds": float(mean_m),
        "oof_std_mae_across_seeds": float(std_m),
    }
    return oof_final, test_final, meta


# -----------------------------
# CROSS-FIT group-median calibration (avoids leakage)
# -----------------------------
def _group_key(df: pd.DataFrame, cols: List[str]) -> pd.Series:
    # stable string key
    tmp = df[cols].copy()
    for c in cols:
        if pd.api.types.is_numeric_dtype(tmp[c]):
            # make bins stable for numeric discrete columns
            tmp[c] = tmp[c].fillna(-1).astype(int).astype(str)
        else:
            tmp[c] = tmp[c].astype(str)
    return tmp.astype(str).agg("|".join, axis=1)

def crossfit_group_calibration(train_feat: pd.DataFrame,
                               base_oof: np.ndarray,
                               group_cols: List[str],
                               shrink: float,
                               min_count: int = 25,
                               k_smooth: float = 60.0,
                               n_splits: int = 5,
                               random_state: int = 42) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Returns:
      oof_cal (cross-fitted)
      shifts_full (fit on full residuals, for applying to test)
    """
    y = train_feat[TARGET].values.astype(float)
    strat = make_strat_labels(train_feat, y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_cal = base_oof.copy()

    for tr, va in skf.split(train_feat, strat):
        resid_tr = y[tr] - base_oof[tr]
        key_tr = _group_key(train_feat.iloc[tr], group_cols)
        # compute median residual per group on TR only
        df_tr = pd.DataFrame({"key": key_tr.values, "resid": resid_tr})
        grp = df_tr.groupby("key")["resid"]
        med = grp.median()
        cnt = grp.size()

        # shrink by count
        shrink_n = (cnt / (cnt + k_smooth)).clip(lower=0.0, upper=1.0)
        shift_map = (med * shrink * shrink_n).to_dict()

        key_va = _group_key(train_feat.iloc[va], group_cols)
        oof_cal[va] = oof_cal[va] + key_va.map(shift_map).fillna(0.0).values

    # fit shifts on full data for test application
    resid_full = y - base_oof
    key_full = _group_key(train_feat, group_cols)
    df_full = pd.DataFrame({"key": key_full.values, "resid": resid_full})
    grp_full = df_full.groupby("key")["resid"]
    med_full = grp_full.median()
    cnt_full = grp_full.size()
    shrink_n_full = (cnt_full / (cnt_full + k_smooth)).clip(lower=0.0, upper=1.0)
    shifts_full = (med_full * shrink * shrink_n_full).to_dict()

    return oof_cal, shifts_full

def apply_shifts_to_test(test_feat: pd.DataFrame, base_test: np.ndarray, group_cols: List[str], shifts_full: Dict[str, float]) -> np.ndarray:
    key_te = _group_key(test_feat, group_cols)
    return base_test + key_te.map(shifts_full).fillna(0.0).values


# -----------------------------
# Main load
# -----------------------------
must_exist(TRAIN_PATH, "TRAIN")
must_exist(TEST_PATH, "TEST")
must_exist(PATIENTS_PATH, "patients")
must_exist(ADM_TRAIN_PATH, "admissions_train")
must_exist(ADM_TEST_PATH, "admissions_test")
if not os.path.exists(RECEIPTS_JOBLIB_PATH):
    print("[warn] receipts_parsed.joblib missing -> receipts features will be empty (likely worse).")

print("\n[load] reading CSVs...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)
adm_tr = pd.read_csv(ADM_TRAIN_PATH)
adm_te = pd.read_csv(ADM_TEST_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)
log_shape("admissions_train", adm_tr)
log_shape("admissions_test", adm_te)

print("\n[target stats]")
print(train[TARGET].describe().to_string())

for df in [train, test, patients]:
    df["patient_id"] = pd.to_numeric(df["patient_id"], errors="coerce").astype(int)

print("\n[admissions] building robust aggregates (+counts)...")
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
if adm_df is None:
    print("  admissions features: None")
else:
    print(f"  admissions features: {adm_df.shape} | cols={list(adm_df.columns)}")

print("\n[receipts] loading receipts_parsed.joblib and building low-dim receipt features (+stable stats)...")
pdf_df = None
if os.path.exists(RECEIPTS_JOBLIB_PATH):
    try:
        pdf_df = load_receipts_joblib(RECEIPTS_JOBLIB_PATH)
        if pdf_df is not None:
            pdf_df["patient_id"] = pd.to_numeric(pdf_df["patient_id"], errors="coerce").astype("Int64")
            pdf_df = pdf_df.dropna(subset=["patient_id"]).copy()
            pdf_df["patient_id"] = pdf_df["patient_id"].astype(int)
            pdf_df = pdf_df.drop_duplicates("patient_id", keep="last")
            print(f"  receipt_feat shape: {pdf_df.shape}")
            print(f"  receipt_feat cols ({len(pdf_df.columns)-1}): {[c for c in pdf_df.columns if c!='patient_id']}")
        else:
            print("  [warn] could not build receipt features from joblib structure.")
    except Exception as e:
        print(f"  [warn] receipts joblib load/build failed: {e}")
        pdf_df = None
else:
    print("  [warn] receipts joblib missing; skipping receipts features.")

print("\n[features] building train/test feature frames...")
train_feat = build_features(train, patients, adm_df, pdf_df)
test_feat  = build_features(test,  patients, adm_df, pdf_df)

feat_full = get_numeric_feature_cols(train_feat)
feat_full = [c for c in feat_full if c != TARGET]

# PRUNED set: stable features + new stable stats
pruned_candidates = [
    # priors + transforms
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k","cost_per_visit","log_visits",
    "baseline_next3y",

    # demographics
    "chronic_encoded","age","age_bin","sex_encoded","insurance_encoded","insurance_cat","zip_region","ins_x_chronic",

    # admissions (expanded)
    "n_adm","charlson_max","charlson_mean","charlson_sum","charlson_bin","pct_emergent","n_emergent",

    # receipt robust + new stable stats
    "cost_per_em","cost_hhi","cost_entropy","top1_share","n_line_items","mean_line_amt","std_line_amt","max_line_amt",
    "pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","n_unique_codes",

    # light interactions
    "logprior_x_pctcritical","logprior_x_highacu",

    # stable ratio
    "cost_per_code",
]
feat_pruned = [c for c in pruned_candidates if c in train_feat.columns]

# fill safety
for c in set(feat_full + feat_pruned):
    if c in train_feat.columns:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0.0
        train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").fillna(med)
        test_feat[c]  = pd.to_numeric(test_feat[c], errors="coerce").fillna(med)

feat_full = drop_constants(feat_full, train_feat)
feat_pruned = drop_constants(feat_pruned, train_feat)

print(f"  FULL feature count:   {len(feat_full)}")
print(f"  PRUNED feature count: {len(feat_pruned)}")
print(f"  PRUNED features: {feat_pruned}")

miss_train = int(train_feat[feat_full].isna().sum().sum())
miss_test  = int(test_feat[feat_full].isna().sum().sum())
print(f"  Missing cells after fill (full): train={miss_train} test={miss_test}")

# train
oof_by_seed, test_by_seed, best_iters = train_models(train_feat, test_feat, feat_full, feat_pruned)

# baseline vectors
baseline_oof = train_feat["baseline_next3y"].values.astype(float)
baseline_test = test_feat["baseline_next3y"].values.astype(float)

# stable ensemble search (now 4 models)
oof_ens, test_ens, meta = stable_ensemble_search(train_feat, oof_by_seed, test_by_seed, baseline_oof, baseline_test)

y = train_feat[TARGET].values.astype(float)
base_mae = float(mean_absolute_error(y, oof_ens))
print("\n" + "="*70)
print("[BASE ENSEMBLE OOF]")
print(f"  OOF MAE (stable search): {base_mae:.3f}")
print("  ensemble meta:", meta)
print("="*70)

# -----------------------------
# Cross-fit group calibration candidates (LB-oriented)
# -----------------------------
print("\n[calibration] trying CROSS-FIT group-median calibration candidates (no leakage)...")

calib_candidates = []
# always safe
calib_candidates.append((["primary_chronic"], 0.30))
# usually best ROI
calib_candidates.append((["primary_chronic","insurance_cat"], 0.25))
# optional: add charlson bin if present (can help HF vs others)
if "charlson_bin" in train_feat.columns:
    calib_candidates.append((["primary_chronic","insurance_cat","charlson_bin"], 0.20))

best_oof = oof_ens.copy()
best_test = test_ens.copy()
best_calib = {"type": "none"}
best_mae = base_mae

for cols, shrink in calib_candidates:
    try:
        oof_cal, shifts_full = crossfit_group_calibration(
            train_feat, oof_ens, group_cols=cols,
            shrink=shrink, min_count=25, k_smooth=60.0,
            n_splits=5, random_state=SEED+11
        )
        mae_cal = float(mean_absolute_error(y, oof_cal))
        print(f"  candidate cols={cols} shrink={shrink:.2f} -> OOF MAE={mae_cal:.3f} (delta={mae_cal-best_mae:+.3f})")
        # accept only if improvement is meaningful to reduce LB-overfit risk
        if mae_cal < best_mae - 0.08:
            best_mae = mae_cal
            best_oof = oof_cal
            best_test = apply_shifts_to_test(test_feat, test_ens, cols, shifts_full)
            best_calib = {"type":"group_median_crossfit", "cols": cols, "shrink": float(shrink)}
    except Exception as e:
        print(f"  [warn] calibration candidate failed cols={cols}: {e}")

print("\n[calibration] chosen:", best_calib)

# final clip (LB-safe, but not too aggressive)
y_max = float(np.max(y))
best_test = np.clip(best_test, 0.0, y_max * 1.6)

final_oof_mae = float(mean_absolute_error(y, best_oof))
print("\n" + "="*70)
print("[FINAL OOF]")
print(f"  final OOF MAE (ensemble + optional cross-fit calibration): {final_oof_mae:.3f}")
print("  ensemble meta:", meta)
print("  calibration:", best_calib)
print("  OOF pred quantiles:", qdict(best_oof, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("="*70)

# submission
sub = pd.DataFrame({
    "patient_id": test["patient_id"].values.astype(int),
    "ed_cost_next3y_usd": np.round(best_test.astype(float), 2)
})
sub = sub[["patient_id", "ed_cost_next3y_usd"]]

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print("\n[SUBMISSION sanity checks]")
print("submission shape:", sub.shape)
print("submission columns exactly:", list(sub.columns))
print("any NaNs in preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(sub["ed_cost_next3y_usd"].min()), float(sub["ed_cost_next3y_usd"].median()), float(sub["ed_cost_next3y_usd"].max()))
print("pred quantiles:", qdict(sub["ed_cost_next3y_usd"].values))
print("\nSaved submission to:", str(out_path))
print("\nPaste back: (1) leaderboard MAE, (2) FINAL OOF MAE, (3) ensemble meta, (4) calibration chosen, (5) pred quantiles.")


ITERATION 8 | Tail+Calibration++: +log1p model + cross-fit group calibration + stable stats
Goal: push LB below 450 by reducing tail bias + calibration leakage + improving systematic bias correction.

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB
[admissions_train] shape=(5000, 9) | cols=9 | mem=0.57 MB
[admissions_test] shape=(5000, 8) | cols=8 | mem=0.53 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[admissions] building robust aggregates (+counts)...
  admissions features: (4000, 7) | cols=['patient_id', 'n_adm', 'charlson_max', 'charlson_mean', 'charlson_sum', 'pct_emergent', 'n_emergent']

[receipts] loading receipts_parsed.joblib and building low-dim receipt features (+stable stats)...
  receipt_feat shape:

# Iteration 5
- 450.6385

In [17]:
# === ITERATION 10 (FIXED): Iter7-Core + AdvWeights + RobustStack + Shrunk Chronic Shifts ===
# Fix: CatBoostRegressor.fit() does NOT support eval_sample_weight in your version.
#      Use Pool(weight=...) for train/eval.

import os, re, sys, gc, warnings, random
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"

ID_COL = "patient_id"
TARGET = "ed_cost_next3y_usd"

print("="*100)
print("ITERATION 10 | FIXED (Pool weights) | Iter7-Core + AdvWeights + RobustStack + Shrunk Chronic Shifts")
print("="*100)

def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error, roc_auc_score
    from sklearn.linear_model import HuberRegressor
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error, roc_auc_score
    from sklearn.linear_model import HuberRegressor

try:
    from catboost import CatBoostRegressor, CatBoostClassifier, Pool
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor, CatBoostClassifier, Pool

class CFG:
    # Base CV bagging
    N_FOLDS = 7
    N_SEEDS = 5

    # CatBoost base
    ITERS = 3000
    ES_ROUNDS = 120
    LR = 0.03
    RSM = 0.80

    # Adversarial weighting
    ADV_FOLDS = 5
    ADV_ITERS = 2000
    ADV_CLIP = (0.60, 1.80)
    ADV_POWER = 0.60

    # Stacker
    META_FOLDS = 7
    HUBER_ALPHA = 1e-3
    HUBER_EPS = 1.35

    # Shrunk group shift
    SHIFT_K = 120
    SHIFT_CAP = 80.0
    PRED_CLIP_MULT = 1.5

def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

# -----------------------------
# Admissions aggregates
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs:
        return None

    adm = pd.concat(dfs, ignore_index=True)
    need = ["patient_id", "charlson_band", "acuity_emergent"]
    if not all(c in adm.columns for c in need):
        return None

    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")

    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band", "max"),
        charlson_mean=("charlson_band", "mean"),
        pct_emergent=("acuity_emergent", "mean"),
    ).reset_index()

    for c in ["charlson_max", "charlson_mean", "pct_emergent"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    return out

# -----------------------------
# Receipts loader + Iter7 low-dim builder (same as before, keep minimal)
# -----------------------------
def _extract_lineitems_df(obj) -> Optional[pd.DataFrame]:
    if isinstance(obj, dict):
        for k in ["lineitems_df", "lineitems", "items_df", "items", "line_items_df", "line_items"]:
            if k in obj and isinstance(obj[k], pd.DataFrame):
                return obj[k].copy()
        for k, v in obj.items():
            if isinstance(v, pd.DataFrame) and ("patient_id" in v.columns):
                return v.copy()
        return None
    if isinstance(obj, pd.DataFrame):
        return obj.copy()
    if isinstance(obj, (list, tuple)):
        for v in obj:
            if isinstance(v, pd.DataFrame) and ("patient_id" in v.columns):
                return v.copy()
    return None

def _choose_amount_column(li: pd.DataFrame) -> Tuple[Optional[str], str]:
    cols = set(li.columns)
    direct_candidates = ["line_total", "line_total_usd", "extended_price", "item_total", "line_cost", "amount"]
    for c in direct_candidates:
        if c in cols:
            return c, "direct"

    qty_col = next((c for c in ["qty", "quantity"] if c in cols), None)
    unit_col = next((c for c in ["unit_price", "price", "unit_cost"] if c in cols), None)
    if qty_col and unit_col:
        return f"{qty_col}__x__{unit_col}", "qtyxunit"

    total_candidates = ["sum_line_total", "receipt_total", "sum_items", "total_amount", "sum_total"]
    total_col = next((c for c in total_candidates if c in cols), None)
    if total_col:
        return total_col, "uniform_total"

    if "total" in cols:
        return "total", "direct"
    return None, "none"

def build_receipt_features_iter7(li: pd.DataFrame) -> Optional[pd.DataFrame]:
    li = li.copy()
    if ID_COL not in li.columns:
        return None

    code_col = None
    for c in ["code", "cpt", "cpt_code", "hcpcs", "proc_code"]:
        if c in li.columns:
            code_col = c
            break
    if code_col is None:
        return None

    amt_col, amt_mode = _choose_amount_column(li)
    if amt_col is None or amt_mode == "none":
        return None

    li[ID_COL] = pd.to_numeric(li[ID_COL], errors="coerce").astype("Int64")
    li = li.dropna(subset=[ID_COL]).copy()
    li[ID_COL] = li[ID_COL].astype(int)

    li["code_norm"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code_norm"]).copy()

    if amt_mode == "direct":
        li["amt"] = pd.to_numeric(li[amt_col], errors="coerce").fillna(0.0).astype(float)
    elif amt_mode == "qtyxunit":
        qty_col, unit_col = amt_col.split("__x__")
        qty = pd.to_numeric(li[qty_col], errors="coerce").fillna(0.0).astype(float)
        unit = pd.to_numeric(li[unit_col], errors="coerce").fillna(0.0).astype(float)
        li["amt"] = (qty * unit).astype(float)
    else:  # uniform_total
        total_per_patient = pd.to_numeric(li[amt_col], errors="coerce").fillna(0.0).astype(float)
        total_per_patient = total_per_patient.groupby(li[ID_COL]).transform("first")
        sizes = li.groupby(ID_COL)["code_norm"].transform("size").astype(float).clip(lower=1.0)
        li["amt"] = (total_per_patient / sizes).astype(float)

    li.loc[li["amt"] < 0, "amt"] = 0.0

    receipt_total = li.groupby(ID_COL)["amt"].sum().rename("receipt_total")
    li = li.join(receipt_total, on=ID_COL)
    denom = li["receipt_total"].replace(0.0, np.nan)

    is_em = li["code_norm"].isin(["99281","99282","99283","99284","99285"])
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code_norm"].map(em_map).fillna(0).astype(float)

    is_crit = li["code_norm"].isin(["99291","99292"])
    is_obs  = li["code_norm"].str.startswith("G037", na=False)
    is_high = li["code_norm"].isin(["31500","36556","32551","36620","92950"])

    code_num = pd.to_numeric(li["code_norm"].where(li["code_norm"].str.fullmatch(r"\d+"), None), errors="coerce")
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    share = (li["amt"] / denom).fillna(0.0)
    cost_hhi = (share * share).groupby(li[ID_COL]).sum().rename("cost_hhi")

    n_unique_codes = li.groupby(ID_COL)["code_norm"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li[ID_COL]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li[ID_COL]).max().rename("max_em_level")

    has_critical_care = is_crit.astype(int).groupby(li[ID_COL]).max().rename("has_critical_care")
    has_high_acuity   = is_high.astype(int).groupby(li[ID_COL]).max().rename("has_high_acuity")
    has_imaging       = is_imaging.astype(int).groupby(li[ID_COL]).max().rename("has_imaging")
    has_observation   = is_obs.astype(int).groupby(li[ID_COL]).max().rename("has_observation")
    has_99285         = (li["code_norm"].eq("99285").astype(int).groupby(li[ID_COL]).max()).rename("has_99285")

    em_total   = li.loc[is_em].groupby(ID_COL)["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby(ID_COL)["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby(ID_COL)["amt"].sum().rename("proc_total")
    high_total = li.loc[is_high].groupby(ID_COL)["amt"].sum().rename("high_total")

    out = pd.concat([
        n_unique_codes, cost_hhi, n_em_codes, max_em_level,
        has_critical_care, has_high_acuity, has_imaging, has_observation, has_99285,
        receipt_total
    ], axis=1).reset_index()

    for s in [em_total, crit_total, proc_total, high_total]:
        out = out.merge(s.reset_index(), on=ID_COL, how="left")

    for c in ["em_total","crit_total","proc_total","high_total","receipt_total"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    d2 = out["receipt_total"].replace(0.0, np.nan)
    out["pct_cost_critical"] = (out["crit_total"] / d2).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / d2).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / d2).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    out["cost_per_em"] = np.where(
        out["n_em_codes"] > 0,
        out["receipt_total"] / out["n_em_codes"].clip(lower=1),
        out["receipt_total"]
    )
    out["n_high_acuity_total"] = (out["has_high_acuity"].fillna(0).astype(int) + out["has_critical_care"].fillna(0).astype(int)).astype(int)

    out.drop(columns=["em_total","crit_total","proc_total","high_total","receipt_total"], inplace=True, errors="ignore")
    for c in out.columns:
        if c == ID_COL: continue
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    return out

def load_receipts_features(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    try:
        obj = joblib_load(joblib_path)
        li = _extract_lineitems_df(obj)
        if li is None:
            return None
        feat = build_receipt_features_iter7(li)
        if feat is None:
            return None
        feat[ID_COL] = pd.to_numeric(feat[ID_COL], errors="coerce").astype("Int64")
        feat = feat.dropna(subset=[ID_COL]).copy()
        feat[ID_COL] = feat[ID_COL].astype(int)
        feat = feat.drop_duplicates(ID_COL, keep="last")
        return feat
    except Exception as e:
        print(f"[warn] receipts joblib load failed: {type(e).__name__}: {e}")
        return None

# -----------------------------
# Build features (Iter7-core)
# -----------------------------
def build_features(ed_df: pd.DataFrame,
                   patients_df: pd.DataFrame,
                   adm_df: Optional[pd.DataFrame],
                   rcpt_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    feat = ed_df.copy()

    chronic_map = {"PNEUMONIA": 0, "DIABETESCOMP": 1, "HF": 2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    feat["prior_ed_visits_5y"] = pd.to_numeric(feat["prior_ed_visits_5y"], errors="coerce").fillna(0.0)
    feat["prior_ed_cost_5y_usd"] = pd.to_numeric(feat["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0)

    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost_cap20k"] = np.log1p(feat["prior_cost_cap20k"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)

    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    p = patients_df.copy()
    p[ID_COL] = pd.to_numeric(p[ID_COL], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)

    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private": 2, "public": 1, "self_pay": 0, "selfpay": 0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)

    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D", "", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    feat = feat.merge(p[[ID_COL, "age", "sex_encoded", "insurance_encoded", "zip_region"]], on=ID_COL, how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    if adm_df is not None:
        feat = feat.merge(adm_df, on=ID_COL, how="left")
        for c in ["charlson_max", "charlson_mean", "pct_emergent"]:
            if c in feat.columns:
                feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(0.0)

    if rcpt_df is not None:
        feat = feat.merge(rcpt_df, on=ID_COL, how="left")
        for c in rcpt_df.columns:
            if c == ID_COL: continue
            feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(0.0)

    feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * pd.to_numeric(feat.get("pct_cost_critical", 0.0), errors="coerce").fillna(0.0)
    feat["logprior_x_highacu"] = feat["log_prior_cost"] * pd.to_numeric(feat.get("n_high_acuity_total", 0.0), errors="coerce").fillna(0.0)

    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)
    else:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"]

    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in [ID_COL, "primary_chronic", TARGET]:
            continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce").fillna(0.0)
    return feat

def get_numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    exclude = {ID_COL, "primary_chronic", TARGET, "sex", "insurance", "zip3"}
    cols = []
    for c in df.columns:
        if c in exclude: 
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def drop_constants(cols: List[str], df: pd.DataFrame) -> List[str]:
    out = []
    for c in cols:
        if c in df.columns and df[c].nunique(dropna=False) > 1:
            out.append(c)
    return out

# -----------------------------
# Adversarial weights
# -----------------------------
def build_importance_weights(train_feat: pd.DataFrame,
                             test_feat: pd.DataFrame,
                             cols: List[str],
                             seed: int = 42,
                             clip: Tuple[float, float] = (0.6, 1.8),
                             power: float = 0.6) -> np.ndarray:
    X_tr = train_feat[cols].copy()
    X_te = test_feat[cols].copy()

    X = pd.concat([X_tr, X_te], axis=0, ignore_index=True)
    y = np.r_[np.zeros(len(X_tr)), np.ones(len(X_te))].astype(int)

    oof_p = np.zeros(len(X), dtype=float)
    skf = StratifiedKFold(n_splits=CFG.ADV_FOLDS, shuffle=True, random_state=seed)

    for fold, (ti, vi) in enumerate(skf.split(X, y), 1):
        clf = CatBoostClassifier(
            loss_function="Logloss",
            depth=4,
            learning_rate=0.05,
            iterations=CFG.ADV_ITERS,
            l2_leaf_reg=8.0,
            random_strength=1.0,
            rsm=0.85,
            verbose=0,
            allow_writing_files=False,
            random_seed=seed + 1000 + fold * 17,
        )
        clf.fit(X.iloc[ti], y[ti])
        oof_p[vi] = clf.predict_proba(X.iloc[vi])[:, 1]
        del clf
        gc.collect()

    auc = roc_auc_score(y, oof_p)
    print(f"[adv] train-vs-test AUC={auc:.4f}")

    p_tr = np.clip(oof_p[:len(X_tr)], 1e-4, 1 - 1e-4)
    odds = p_tr / (1 - p_tr)
    w = np.clip(odds, clip[0], clip[1]) ** power
    print(f"[adv] weights stats: min={w.min():.3f} p50={np.median(w):.3f} p90={np.quantile(w,0.9):.3f} max={w.max():.3f}")
    return w

# -----------------------------
# TRAIN MODELS (FIXED: use Pool for weights)
# -----------------------------
def train_models(train_feat: pd.DataFrame,
                 test_feat: pd.DataFrame,
                 feat_full: List[str],
                 feat_pruned: List[str],
                 w_imp: Optional[np.ndarray]) -> Tuple[Dict[str, List[np.ndarray]], Dict[str, List[np.ndarray]]]:
    y = train_feat[TARGET].values.astype(float)
    y_log = np.log1p(y)

    tmp = train_feat[["primary_chronic", TARGET]].copy()
    tmp["cost_bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop")
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"].values)

    model_specs = {
        "A_RMSE_full_d5": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.0,
        ),
        "B_RMSE_pruned_d4": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=4, l2_leaf_reg=4, min_data_in_leaf=32,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=2.0,
        ),
        "C_LOGRMSE_pruned_d4": dict(
            loss_function="RMSE", eval_metric="RMSE",
            depth=4, l2_leaf_reg=5, min_data_in_leaf=36,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, random_strength=1.0,
        ),
    }
    model_cols = {
        "A_RMSE_full_d5": feat_full,
        "B_RMSE_pruned_d4": feat_pruned,
        "C_LOGRMSE_pruned_d4": feat_pruned,
    }

    oof_by_seed = {m: [] for m in model_specs}
    test_by_seed = {m: [] for m in model_specs}

    print("\n[base training] (FIXED) using Pool(weight=...) for train/eval")
    print(f"Seeds={CFG.N_SEEDS}, Folds={CFG.N_FOLDS}, Models={list(model_specs.keys())}\n")

    for seed_idx in range(CFG.N_SEEDS):
        rs = SEED + seed_idx * 7
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs}
        test_seed = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat), 1):
            for mname, params in model_specs.items():
                cols = model_cols[mname]

                X_tr = train_feat[cols].iloc[ti]
                X_va = train_feat[cols].iloc[vi]
                X_te = test_feat[cols]

                if mname.startswith("C_LOG"):
                    y_tr = y_log[ti]
                    y_va = y_log[vi]
                else:
                    y_tr = y[ti]
                    y_va = y[vi]

                cb = CatBoostRegressor(
                    **params,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=rs + fold * 31 + (hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )

                if w_imp is None:
                    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
                else:
                    sw_tr = w_imp[ti]
                    sw_va = w_imp[vi]
                    train_pool = Pool(X_tr, y_tr, weight=sw_tr)
                    val_pool = Pool(X_va, y_va, weight=sw_va)
                    cb.fit(train_pool, eval_set=val_pool, verbose=0)

                pred_va = cb.predict(X_va)
                pred_te = cb.predict(X_te)

                if mname.startswith("C_LOG"):
                    pred_va = np.expm1(pred_va)
                    pred_te = np.expm1(pred_te)

                pred_va = np.clip(pred_va, 0.0, None)
                pred_te = np.clip(pred_te, 0.0, None)

                oof_seed[mname][vi] = pred_va
                test_seed[mname] += pred_te / CFG.N_FOLDS

                del cb
                gc.collect()

        maes = {m: float(mean_absolute_error(y, oof_seed[m])) for m in model_specs}
        print(f"  Seed {seed_idx+1}/{CFG.N_SEEDS} OOF MAE: " + " | ".join([f"{m}={maes[m]:.2f}" for m in model_specs]))

        for m in model_specs:
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed[m])

    print("\n[seed-averaged OOF MAE per model]")
    for m in model_specs:
        oof_avg_m = np.mean(np.vstack(oof_by_seed[m]), axis=0)
        print(f"  {m:20s}: {mean_absolute_error(y, oof_avg_m):.2f}")

    return oof_by_seed, test_by_seed

# -----------------------------
# Robust stacking (Huber) with meta CV
# -----------------------------
def robust_stack_huber(y: np.ndarray,
                       Z_tr: np.ndarray,
                       Z_te: np.ndarray,
                       strat_labels: np.ndarray,
                       w_imp: Optional[np.ndarray]) -> Tuple[np.ndarray, np.ndarray, Dict]:
    oof = np.zeros(len(y), dtype=float)
    kf = StratifiedKFold(n_splits=CFG.META_FOLDS, shuffle=True, random_state=SEED + 999)

    for fold, (ti, vi) in enumerate(kf.split(Z_tr, strat_labels), 1):
        reg = HuberRegressor(alpha=CFG.HUBER_ALPHA, epsilon=CFG.HUBER_EPS, max_iter=5000)
        sw = None if w_imp is None else w_imp[ti]
        reg.fit(Z_tr[ti], y[ti], sample_weight=sw)
        oof[vi] = reg.predict(Z_tr[vi])

    oof = np.clip(oof, 0.0, None)
    mae = float(mean_absolute_error(y, oof))
    print(f"[stack] meta OOF MAE (Huber CV): {mae:.3f}")

    reg_full = HuberRegressor(alpha=CFG.HUBER_ALPHA, epsilon=CFG.HUBER_EPS, max_iter=5000)
    reg_full.fit(Z_tr, y, sample_weight=w_imp if w_imp is not None else None)
    te = np.clip(reg_full.predict(Z_te), 0.0, None)

    meta = {"coef": reg_full.coef_.tolist(), "intercept": float(reg_full.intercept_), "meta_oof_mae": mae}
    print(f"[stack] full coef={np.round(reg_full.coef_, 4)} | intercept={reg_full.intercept_:.4f}")
    return oof, te, meta

# -----------------------------
# Shrunk chronic shifts
# -----------------------------
def compute_shrunk_group_shifts(y: np.ndarray,
                                pred_oof: np.ndarray,
                                group: np.ndarray,
                                groups=(0, 1, 2),
                                unk_value=-1,
                                k=120,
                                cap=80.0) -> Tuple[Dict[int, float], float]:
    resid = y - pred_oof
    global_med = float(np.median(resid))

    group2 = group.copy()
    group2[group2 == unk_value] = 0

    shifts = {}
    for g in groups:
        mask = (group2 == g)
        r = resid[mask]
        raw = float(np.median(r)) if len(r) else global_med
        n = int(mask.sum()) if len(r) else 0
        alpha = n / (n + k)
        shrunk = alpha * raw + (1 - alpha) * global_med
        shrunk = float(np.clip(shrunk, -cap, cap))
        shifts[g] = shrunk

    shifts[unk_value] = shifts[0]
    return shifts, global_med

def apply_group_shifts(pred: np.ndarray, group: np.ndarray, shifts: Dict[int, float]) -> np.ndarray:
    out = pred.copy()
    for g, s in shifts.items():
        out[group == g] += s
    mask_unknown = ~np.isin(group, list(shifts.keys()))
    if mask_unknown.any():
        out[mask_unknown] += shifts.get(0, 0.0)
    return out

# =============================
# MAIN RUN
# =============================
must_exist(TRAIN_PATH, "TRAIN")
must_exist(TEST_PATH, "TEST")
must_exist(PATIENTS_PATH, "patients")
must_exist(ADM_TRAIN_PATH, "admissions_train")
must_exist(ADM_TEST_PATH, "admissions_test")

if not os.path.exists(RECEIPTS_JOBLIB_PATH):
    print("[warn] receipts_parsed.joblib missing -> receipts features absent (likely worse).")

print("\n[load] reading CSVs...")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
patients_df = pd.read_csv(PATIENTS_PATH)

log_shape("ed_cost_train", train_df)
log_shape("ed_cost_test", test_df)
log_shape("patients", patients_df)

print("\n[target stats]")
print(train_df[TARGET].describe().to_string())

for df in [train_df, test_df, patients_df]:
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype(int)

print("\n[admissions] building aggregates...")
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
print("  admissions features:", None if adm_df is None else adm_df.shape)

print("\n[receipts] loading receipts_parsed.joblib -> Iter7 low-dim...")
rcpt_df = load_receipts_features(RECEIPTS_JOBLIB_PATH) if os.path.exists(RECEIPTS_JOBLIB_PATH) else None
print("  receipts features:", None if rcpt_df is None else rcpt_df.shape)

print("\n[features] building train/test frames...")
train_feat = build_features(train_df, patients_df, adm_df, rcpt_df)
test_feat  = build_features(test_df,  patients_df, adm_df, rcpt_df)

feat_full = drop_constants(get_numeric_feature_cols(train_feat), train_feat)

pruned_candidates = [
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","prior_cost_cap20k","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k",
    "cost_per_visit","log_visits",
    "baseline_next3y",
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    "charlson_max","charlson_mean","pct_emergent",
    "cost_per_em","cost_hhi","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","n_unique_codes",
    "logprior_x_pctcritical","logprior_x_highacu","cost_per_code",
]
feat_pruned = drop_constants([c for c in pruned_candidates if c in train_feat.columns], train_feat)

print(f"  FULL feature count:   {len(feat_full)}")
print(f"  PRUNED feature count: {len(feat_pruned)}")

# fill medians
for c in set(feat_full + feat_pruned + [TARGET, "baseline_next3y", "chronic_encoded"]):
    if c in train_feat.columns:
        med = train_feat[c].median() if not train_feat[c].isna().all() else 0.0
        train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").fillna(med)
    if c in test_feat.columns:
        med = train_feat[c].median() if (c in train_feat.columns and not train_feat[c].isna().all()) else 0.0
        test_feat[c] = pd.to_numeric(test_feat[c], errors="coerce").fillna(med)

y = train_feat[TARGET].values.astype(float)
baseline_tr = train_feat["baseline_next3y"].values.astype(float)
baseline_te = test_feat["baseline_next3y"].values.astype(float)

# strat labels
tmp = train_feat[["primary_chronic", TARGET]].copy()
tmp["cost_bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop")
tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
strat_labels = LabelEncoder().fit_transform(tmp["strat"].values)

print("\n" + "="*70)
print("[STEP 1] Adversarial importance weights")
print("="*70)
w_imp = build_importance_weights(train_feat, test_feat, feat_pruned, seed=SEED, clip=CFG.ADV_CLIP, power=CFG.ADV_POWER)

print("\n" + "="*70)
print("[STEP 2] Base models with weights (Pool fix)")
print("="*70)
oof_by_seed, test_by_seed = train_models(train_feat, test_feat, feat_full, feat_pruned, w_imp)

models = list(oof_by_seed.keys())
oof_avg = {m: np.mean(np.vstack(oof_by_seed[m]), axis=0) for m in models}
test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in models}

print("\n[base avg] OOF MAE:")
for m in models:
    print(f"  {m:20s}: {mean_absolute_error(y, oof_avg[m]):.3f}")

print("\n" + "="*70)
print("[STEP 3] Robust stacking (Huber) on base preds + baseline")
print("="*70)
Z_tr = np.column_stack([oof_avg[m] for m in models] + [baseline_tr])
Z_te = np.column_stack([test_avg[m] for m in models] + [baseline_te])

oof_stack, test_stack, meta_info = robust_stack_huber(y, Z_tr, Z_te, strat_labels, w_imp)
print("[stack] OOF quantiles:", qdict(oof_stack))

print("\n" + "="*70)
print("[STEP 4] Shrunk chronic shifts (safer)")
print("="*70)
chronic_tr = train_feat["chronic_encoded"].values.astype(int)
chronic_te = test_feat["chronic_encoded"].values.astype(int)

shifts, global_med = compute_shrunk_group_shifts(
    y=y, pred_oof=oof_stack, group=chronic_tr,
    k=CFG.SHIFT_K, cap=CFG.SHIFT_CAP
)
print(f"[shift] global median residual={global_med:+.3f}")
print(f"[shift] shifts={shifts}")

oof_final = np.clip(apply_group_shifts(oof_stack, chronic_tr, shifts), 0.0, None)
test_final = np.clip(apply_group_shifts(test_stack, chronic_te, shifts), 0.0, None)

mae_pre = float(mean_absolute_error(y, oof_stack))
mae_post = float(mean_absolute_error(y, oof_final))
print(f"[OOF] stack MAE pre-shift : {mae_pre:.3f}")
print(f"[OOF] final MAE post-shift: {mae_post:.3f}")

# final clip
y_max = float(np.max(y))
test_final = np.clip(test_final, 0.0, y_max * CFG.PRED_CLIP_MULT)

print("\n" + "="*70)
print("[STEP 5] Write submission")
print("="*70)
sub = pd.DataFrame({
    ID_COL: test_df[ID_COL].values.astype(int),
    TARGET: np.round(test_final.astype(float), 2),
})[[ID_COL, TARGET]]

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print("Saved:", str(out_path))
print("submission stats:", sub[TARGET].describe().to_string())
print("pred quantiles:", qdict(sub[TARGET].values))

print("\nAfter you submit, paste back:")
print("  (1) LB MAE")
print("  (2) [adv] AUC + weight stats")
print("  (3) base avg OOF MAEs")
print("  (4) stack MAE pre/post + shifts")


ITERATION 10 | FIXED (Pool weights) | Iter7-Core + AdvWeights + RobustStack + Shrunk Chronic Shifts

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[admissions] building aggregates...
  admissions features: (4000, 4)

[receipts] loading receipts_parsed.joblib -> Iter7 low-dim...
  receipts features: (4000, 15)

[features] building train/test frames...
  FULL feature count:   35
  PRUNED feature count: 31

[STEP 1] Adversarial importance weights
[adv] train-vs-test AUC=0.4897
[adv] weights stats: min=0.736 p50=1.014 p90=1.423 max=1.423

[STEP 2] Base models with weights (Pool fix)

[base training] (FIXED) using Pool(weight=...) for train/eval
Seeds=5, Folds=7, Models=['A_R

# Iteration 6
- 451.6564

In [13]:
# === ITERATION 11: The "Actuarial" Pivot (Tweedie Loss) ===
# Diagnosis: Iter 8/9/10 failed due to Loss Mismatch (Log vs MAE) and Signal Saturation.
# Solution: Use Tweedie Loss (Compound Poisson-Gamma). This is the standard actuarial 
#           approach for zero-inflated, right-skewed cost data. It models BOTH the 
#           probability of having $0 cost AND the magnitude of cost if > $0.

import os, re, sys, gc, math, warnings, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
TARGET = "ed_cost_next3y_usd"

print("="*90)
print("ITERATION 11 | The Actuarial Pivot (Tweedie Loss)")
print("Goal: Break 450 barrier by correctly modeling the Zero-Inflated/Long-Tail distribution.")
print("Key Change: Replacing RMSE/Log models with Tweedie:variance_power=1.5")
print("="*90)

# -----------------------------
# Deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Config
# -----------------------------
class CFG:
    N_FOLDS = 7   # High folds for stability
    N_SEEDS = 5   # Standard stability
    ITERS = 3500  # Tweedie converges slowly, need more iters
    ES_ROUNDS = 150
    LR = 0.025    # Lower LR for Tweedie stability
    RSM = 0.75
    
    # Ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.02, 0.05] 
    
    # Penalties
    STD_PEN = 0.20
    LAM_PEN = 6.0

# -----------------------------
# Utilities
# -----------------------------
def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan": return None
    if re.fullmatch(r"\d+\.0+", s): s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)): return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s: return None
    return s.zfill(3)

# -----------------------------
# Features (Iter 7/9 Standard - Proven Robust)
# -----------------------------
def load_admissions_features(adm_train_path, adm_test_path):
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns: df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs: return None
    adm = pd.concat(dfs, ignore_index=True)
    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")
    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
        n_adm=("patient_id", "count") # Added simple count
    ).reset_index()
    for c in ["charlson_max","charlson_mean","pct_emergent", "n_adm"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
    return out

def build_pdf_features_from_lineitems(li):
    li = li.copy()
    code_col = next((c for c in ["code","cpt","cpt_code","hcpcs","proc_code"] if c in li.columns), None)
    total_col = next((c for c in ["line_total","line_total_usd","total","amount"] if c in li.columns), None)
    if not code_col or not total_col or "patient_id" not in li.columns: return None

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)
    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()
    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).clip(lower=0.0)

    total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)

    is_em = li["code"].isin(["99281","99282","99283","99284","99285"])
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)
    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])
    
    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    share = (li["amt"] / denom).fillna(0.0)
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")
    has_obs = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    def has_c(c, n): return (li["code"].eq(c).astype(int).groupby(li["patient_id"]).max()).rename(n)
    has_99285 = has_c("99285","has_99285")
    
    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    out = pd.concat([n_unique_codes, cost_hhi, n_em_codes, max_em_level, has_critical_care, has_high_acuity, has_imaging, has_obs, has_99285, total], axis=1).reset_index()
    for s in [em_total, crit_total, proc_total, high_total]: out = out.merge(s.reset_index(), on="patient_id", how="left")
    for c in ["em_total","crit_total","proc_total","high_total","receipt_total"]:
        if c not in out.columns: out[c] = 0.0
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
        
    d2 = out["receipt_total"].replace(0.0, np.nan)
    out["pct_cost_critical"] = (out["crit_total"] / d2).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / d2).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / d2).fillna(0.0)
    out["cost_per_em"] = np.where(out["n_em_codes"]>0, out["receipt_total"]/out["n_em_codes"].clip(lower=1), out["receipt_total"])
    out["n_high_acuity_total"] = (out["has_high_acuity"].fillna(0) + out["has_critical_care"].fillna(0)).astype(int)

    return out

def load_receipts(path):
    if not os.path.exists(path): return None
    try:
        d = joblib_load(path)
        if isinstance(d, dict):
            for k in ["lineitems_df","lineitems","items_df"]:
                if k in d and isinstance(d[k], pd.DataFrame): return build_pdf_features_from_lineitems(d[k])
        if isinstance(d, pd.DataFrame): return build_pdf_features_from_lineitems(d)
    except: pass
    return None

def build_features(ed, pts, adm, pdf):
    f = ed.copy()
    cmap = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    f["primary_chronic"] = f["primary_chronic"].astype(str)
    f["chronic_encoded"] = f["primary_chronic"].str.upper().map(cmap).fillna(-1).astype(float)
    
    f["prior_ed_visits_5y"] = pd.to_numeric(f["prior_ed_visits_5y"], errors="coerce").fillna(0.0)
    f["prior_ed_cost_5y_usd"] = pd.to_numeric(f["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0)
    
    # Actuarial Features
    f["sqrt_prior_cost"] = np.sqrt(f["prior_ed_cost_5y_usd"].clip(lower=0))
    f["log_prior_cost"] = np.log1p(f["prior_ed_cost_5y_usd"].clip(lower=0))
    f["cost_per_visit"] = f["prior_ed_cost_5y_usd"] / f["prior_ed_visits_5y"].clip(lower=1)
    f["prior_cost_cap20k"] = f["prior_ed_cost_5y_usd"].clip(upper=20000)
    f["baseline_next3y"] = f["prior_ed_cost_5y_usd"] * 0.6
    
    # NEW: Zero/Low cost flags (Crucial for Tweedie)
    f["is_zero_cost_prior"] = (f["prior_ed_cost_5y_usd"] <= 1.0).astype(int)
    f["is_low_freq_prior"]  = (f["prior_ed_visits_5y"] <= 1).astype(int)
    
    p = pts.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper()=="M").astype(int)
    ins_map = {"private":2,"public":1,"self_pay":0,"selfpay":0}
    p["insurance_encoded"] = p["insurance"].astype(str).str.lower().map(ins_map).fillna(-1)
    z = p["zip3"].apply(standardize_zip3).astype(str).str[0]
    p["zip_region"] = pd.to_numeric(z, errors="coerce").fillna(-1)
    
    f = f.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    f["ins_x_chronic"] = f["insurance_encoded"] * f["chronic_encoded"]
    
    if adm is not None:
        f = f.merge(adm, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent","n_adm"]:
            if c in f.columns: f[c] = f[c].fillna(f[c].median())
    if pdf is not None:
        f = f.merge(pdf, on="patient_id", how="left")
        for c in pdf.columns:
            if c!="patient_id" and pd.api.types.is_numeric_dtype(pdf[c]): f[c] = f[c].fillna(f[c].median())

    if "pct_cost_critical" in f.columns: f["logprior_x_pctcritical"] = f["log_prior_cost"] * f["pct_cost_critical"]
    if "n_high_acuity_total" in f.columns: f["logprior_x_highacu"] = f["log_prior_cost"] * f["n_high_acuity_total"]
    if "n_unique_codes" in f.columns: f["cost_per_code"] = f["prior_ed_cost_5y_usd"] / f["n_unique_codes"].clip(lower=1)

    for c in f.columns:
        if c not in ["patient_id","primary_chronic",TARGET]: f[c] = pd.to_numeric(f[c], errors="coerce").fillna(0.0)
    return f

# -----------------------------
# Execution
# -----------------------------
print("\n[load] Reading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
pts_df = pd.read_csv(PATIENTS_PATH)
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
pdf_df = load_receipts(RECEIPTS_JOBLIB_PATH)

for d in [train_df, test_df, pts_df]: d["patient_id"] = pd.to_numeric(d["patient_id"], errors="coerce").astype(int)

train_feat = build_features(train_df, pts_df, adm_df, pdf_df)
test_feat = build_features(test_df, pts_df, adm_df, pdf_df)

pruned_cols = [
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","sqrt_prior_cost","log_prior_cost","cost_per_visit",
    "baseline_next3y", "is_zero_cost_prior", "is_low_freq_prior",
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    "charlson_max","charlson_mean","pct_emergent","n_adm",
    "cost_per_em","cost_hhi","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","n_unique_codes",
    "logprior_x_pctcritical","logprior_x_highacu","cost_per_code"
]
all_cols = [c for c in train_feat.columns if c not in ["patient_id","primary_chronic",TARGET,"sex","insurance","zip3"] and pd.api.types.is_numeric_dtype(train_feat[c])]
feat_full = [c for c in all_cols if train_feat[c].nunique() > 1]
feat_pruned = [c for c in pruned_cols if c in train_feat.columns and train_feat[c].nunique() > 1]
print(f"Features: Full={len(feat_full)}, Pruned={len(feat_pruned)}")

# -----------------------------
# Training: The Actuarial Suite
# -----------------------------
y = train_feat[TARGET].values.astype(float)
tmp = train_feat[["primary_chronic", TARGET]].copy()
tmp["bin"] = pd.qcut(tmp[TARGET], 5, labels=False)
tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["bin"].astype(str)
strat = LabelEncoder().fit_transform(tmp["strat"])

# Model Specs:
# T: Tweedie (1.5) - Best for compound Poisson-Gamma (Ins Costs)
# M: MAE - Robust Median anchor
# A: RMSE (Raw) - Mean anchor (kept one, but depth reduced)

model_specs = {
    "T_TWEEDIE_pruned_d5": dict(
        loss_function="Tweedie:variance_power=1.5", 
        eval_metric="MAE", # Evaluate on MAE, optimize on Tweedie
        depth=5, l2_leaf_reg=4, min_data_in_leaf=32,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=1.2,
        early_stopping_rounds=CFG.ES_ROUNDS
    ),
    "M_MAE_pruned_d4": dict(
        loss_function="MAE", eval_metric="MAE", 
        depth=4, l2_leaf_reg=7, min_data_in_leaf=36,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=1.5,
        early_stopping_rounds=CFG.ES_ROUNDS
    ),
    "A_RMSE_full_d5": dict(
        loss_function="RMSE", eval_metric="MAE", 
        depth=5, l2_leaf_reg=5, min_data_in_leaf=28,
        learning_rate=CFG.LR, iterations=CFG.ITERS, rsm=CFG.RSM, random_strength=1.0,
        early_stopping_rounds=CFG.ES_ROUNDS
    )
}
cols_map = {"T_TWEEDIE_pruned_d5": feat_pruned, "M_MAE_pruned_d4": feat_pruned, "A_RMSE_full_d5": feat_full}

oof_by_seed = {m: [] for m in model_specs}
test_by_seed = {m: [] for m in model_specs}

print("\n[Training] 5 Seeds x 7 Folds (Tweedie + MAE + RMSE)...")
for seed_idx in range(CFG.N_SEEDS):
    rs = SEED + seed_idx * 7
    kf = StratifiedKFold(CFG.N_FOLDS, shuffle=True, random_state=rs)
    
    s_oof = {m: np.zeros(len(y)) for m in model_specs}
    s_test = {m: np.zeros(len(test_feat)) for m in model_specs}
    
    for fold, (ti, vi) in enumerate(kf.split(train_feat, strat)):
        for m, p in model_specs.items():
            cols = cols_map[m]
            X_tr, y_tr = train_feat[cols].iloc[ti], y[ti]
            X_va, y_va = train_feat[cols].iloc[vi], y[vi]
            
            cb = CatBoostRegressor(**p, task_type="CPU", thread_count=-1, verbose=0, random_seed=rs+fold*99)
            cb.fit(X_tr, y_tr, eval_set=(X_va, y_va))
            
            s_oof[m][vi] = np.clip(cb.predict(X_va), 0, None)
            s_test[m] += np.clip(cb.predict(test_feat[cols]), 0, None) / CFG.N_FOLDS
            
    for m in model_specs:
        oof_by_seed[m].append(s_oof[m])
        test_by_seed[m].append(s_test[m])
        
    print(f"  Seed {seed_idx+1} MAE: " + " | ".join([f"{m}={mean_absolute_error(y, s_oof[m]):.2f}" for m in model_specs]))

oof_avg = {m: np.mean(np.vstack(oof_by_seed[m]), axis=0) for m in model_specs}
test_avg = {m: np.mean(np.vstack(test_by_seed[m]), axis=0) for m in model_specs}

# -----------------------------
# Ensemble Selection
# -----------------------------
baseline_tr = train_feat["baseline_next3y"].values
baseline_te = test_feat["baseline_next3y"].values
models = ["T_TWEEDIE_pruned_d5", "M_MAE_pruned_d4", "A_RMSE_full_d5"]

best_ens = None
best_score = float("inf")

print("\n[Ensemble] Searching weights (Tweedie/MAE/RMSE)...")
grid = np.arange(0.0, 1.01, CFG.W_STEP)
for wT in grid:
    for wM in grid:
        wA = 1.0 - wT - wM
        if wA < -1e-9: continue
        wA = max(0.0, wA)
        if abs(wT+wM+wA-1.0) > 1e-5: continue
        
        for lam in CFG.LAM_GRID:
            maes = []
            for s in range(CFG.N_SEEDS):
                p = wT*oof_by_seed[models[0]][s] + wM*oof_by_seed[models[1]][s] + wA*oof_by_seed[models[2]][s]
                p = (1-lam)*p + lam*baseline_tr
                maes.append(mean_absolute_error(y, p))
            
            mean_m = np.mean(maes)
            std_m = np.std(maes)
            obj = mean_m + CFG.STD_PEN*std_m + CFG.LAM_PEN*lam
            
            if obj < best_score:
                best_score = obj
                best_ens = (wT, wM, wA, lam)

wT, wM, wA, lam = best_ens
print(f"  Best Weights: Tweedie={wT:.2f}, MAE={wM:.2f}, RMSE={wA:.2f}")
print(f"  Best Lambda: {lam:.2f}")

# Construct Final
oof_final = wT*oof_avg[models[0]] + wM*oof_avg[models[1]] + wA*oof_avg[models[2]]
oof_final = (1-lam)*oof_final + lam*baseline_tr
test_final = wT*test_avg[models[0]] + wM*test_avg[models[1]] + wA*test_avg[models[2]]
test_final = (1-lam)*test_final + lam*baseline_te

print(f"  Final Ensemble MAE: {mean_absolute_error(y, oof_final):.3f}")

# -----------------------------
# Submission
# -----------------------------
sub = pd.DataFrame({
    "patient_id": test_df["patient_id"].values,
    "ed_cost_next3y_usd": np.round(test_final, 2)
})
out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print("\n[Sanity Check]")
print(sub[TARGET].describe())
print(f"\nDone. Saved to {OUT_SUB_PATH}")
print("Please run this and report: (1) Tweedie OOF MAE vs MAE OOF MAE, (2) Final Ensemble MAE.")

ITERATION 11 | The Actuarial Pivot (Tweedie Loss)
Goal: Break 450 barrier by correctly modeling the Zero-Inflated/Long-Tail distribution.
Key Change: Replacing RMSE/Log models with Tweedie:variance_power=1.5

[load] Reading data...
Features: Full=40, Pruned=30

[Training] 5 Seeds x 7 Folds (Tweedie + MAE + RMSE)...
  Seed 1 MAE: T_TWEEDIE_pruned_d5=448.39 | M_MAE_pruned_d4=437.53 | A_RMSE_full_d5=432.04
  Seed 2 MAE: T_TWEEDIE_pruned_d5=448.65 | M_MAE_pruned_d4=441.94 | A_RMSE_full_d5=433.03
  Seed 3 MAE: T_TWEEDIE_pruned_d5=447.22 | M_MAE_pruned_d4=439.35 | A_RMSE_full_d5=430.84
  Seed 4 MAE: T_TWEEDIE_pruned_d5=449.38 | M_MAE_pruned_d4=441.00 | A_RMSE_full_d5=432.09
  Seed 5 MAE: T_TWEEDIE_pruned_d5=445.48 | M_MAE_pruned_d4=442.83 | A_RMSE_full_d5=432.33

[Ensemble] Searching weights (Tweedie/MAE/RMSE)...
  Best Weights: Tweedie=0.00, MAE=0.05, RMSE=0.95
  Best Lambda: 0.00
  Final Ensemble MAE: 429.517

[Sanity Check]
count     2000.000000
mean      3916.268835
std       1744.846249

# Iteration 7
- 451.5770

In [21]:
# === CODE 19 / "The Huber-Anchor" (Aiming for 448.xx) ===
# Base: Code 18 (MAE 449.41)
# Improvement 1: Add "Model D" using Huber Loss + Depth 3 (Ultra-Robust Anchor).
# Improvement 2: Add 'max_unit_price' to receipts (Severity Intensity).
# Improvement 3: Add 'age_x_chronic' interaction.
# Logic: Huber loss handles the heavy tail better than RMSE without the slowness of MAE.
# Output: D:\AgentDs\agent_ds_healthcare\submission.csv

import os, re, sys, gc, math, warnings, random, zlib
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
ID_COL = "patient_id"
TARGET = "ed_cost_next3y_usd"

print("="*95)
print("CODE 19 | The Huber-Anchor Strategy")
print("Base: Code 18 (449.4)")
print("New:  + Huber Loss Model (Depth 3) + Max Unit Price Feature")
print("="*95)

# -----------------------------
# Deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Config
# -----------------------------
class CFG:
    N_FOLDS = 7
    N_SEEDS = 5
    ITERS = 3500 # Huber needs slightly more iters
    ES_ROUNDS = 150
    LR = 0.03
    RSM = 0.75      # Slightly tighter
    SUBSAMPLE = 0.75 # Slightly tighter to force diversity
    
    # Aggregation
    TRIM_K = 1 
    
    # Ensemble Search (4 models now)
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.02, 0.05, 0.10]
    SHIFT_GRID = [0.0, 0.5, 1.0]
    
    # Penalties (Keep robust)
    STD_PEN = 0.20
    LAM_PEN = 5.0
    SHIFT_PEN = 0.002

# -----------------------------
# Utilities
# -----------------------------
def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)): return None
    s = re.sub(r"\D", "", str(z).strip())
    return s.zfill(3) if s else None

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan": return None
    if re.fullmatch(r"\d+\.0+", s): s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def safe_num_series(s: pd.Series, default=0.0) -> pd.Series:
    v = pd.to_numeric(s, errors="coerce")
    v = v.replace([np.inf, -np.inf], np.nan).fillna(default)
    return v

def stable_hash(s: str) -> int:
    return int(zlib.crc32(s.encode("utf-8")) & 0xffffffff)

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0 or np.all(x == 0): return 0.0
    x = np.sort(x)
    n = x.size
    cumx = np.cumsum(x)
    return float((n + 1 - 2*np.sum(cumx)/cumx[-1]) / n)

def trimmed_mean(mat: np.ndarray, trim_k: int = 1) -> np.ndarray:
    mat = np.asarray(mat, dtype=float)
    if mat.ndim != 2: raise ValueError("trimmed_mean expects 2D array")
    n = mat.shape[0]
    if trim_k <= 0 or n < 2*trim_k + 1: return np.mean(mat, axis=0)
    s = np.sort(mat, axis=0)
    return np.mean(s[trim_k:n-trim_k, :], axis=0)

# -----------------------------
# Admissions
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns: df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs: return None
    adm = pd.concat(dfs, ignore_index=True)
    
    for c in ["patient_id","charlson_band","acuity_emergent"]:
        adm[c] = pd.to_numeric(adm[c], errors="coerce")
    
    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()
    
    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = safe_num_series(out[c], default=0.0)
    
    out["patient_id"] = out["patient_id"].fillna(-1).astype(int)
    out = out[out["patient_id"] != -1]
    return out

# -----------------------------
# Receipts (Code 18 + max_unit_price)
# -----------------------------
def build_receipt_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()
    
    # Column mapping
    cols = li.columns
    code_col = next((c for c in ["code","cpt","cpt_code","hcpcs","proc_code"] if c in cols), None)
    total_col = next((c for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"] if c in cols), None)
    unit_col = next((c for c in ["unit_price","unitprice","unit_cost","unitcost"] if c in cols), None)
    qty_col = next((c for c in ["qty","quantity","units"] if c in cols), None)

    if "patient_id" not in cols or not code_col or not total_col:
        raise ValueError("Missing essential columns")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)
    
    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()
    
    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    li["unit_price"] = pd.to_numeric(li[unit_col], errors="coerce") if unit_col else np.nan
    
    # Totals
    receipt_total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(receipt_total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)
    share = (li["amt"] / denom).fillna(0.0)

    # Robust Stats
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    top1_share = share.groupby(li["patient_id"]).max().rename("top1_share")
    gini_amt = li.groupby("patient_id")["amt"].apply(lambda s: gini(s.values)).rename("gini_line_total")
    max_line_total = li.groupby("patient_id")["amt"].max().rename("max_line_total")

    # Buckets
    is_em = li["code"].isin(["99281","99282","99283","99284","99285"])
    em_level = li["code"].map({"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}).fillna(0).astype(int)
    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])
    
    # Basic Counts
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    
    # Flags
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_99285 = li["code"].eq("99285").astype(int).groupby(li["patient_id"]).max().rename("has_99285")

    # === NEW: Max Unit Price (Severity Proxy) ===
    # Using max unit price tells us the most expensive single service provided (e.g. surgery vs just Tylenol)
    # independent of quantity.
    if li["unit_price"].isna().all():
        max_unit_p = pd.Series(0.0, index=li["patient_id"].unique(), name="max_unit_price")
        med_unit_p = pd.Series(0.0, index=li["patient_id"].unique(), name="median_unit_price")
    else:
        max_unit_p = li.groupby("patient_id")["unit_price"].max().rename("max_unit_price")
        med_unit_p = li.groupby("patient_id")["unit_price"].median().rename("median_unit_price")
        
    # Aggregate
    out = pd.concat([
        n_unique_codes, receipt_total, cost_hhi, top1_share, gini_amt, max_line_total,
        n_em_codes, max_em_level,
        has_critical_care, has_high_acuity, has_99285,
        med_unit_p, max_unit_p
    ], axis=1).reset_index()
    
    out["cost_per_em"] = np.where(out["n_em_codes"]>0, out["receipt_total"]/out["n_em_codes"].clip(lower=1), out["receipt_total"])
    
    # Logs
    for c in ["max_line_total", "median_unit_price", "max_unit_price"]:
        out[c] = out[c].fillna(0.0)
        out[f"log1p_{c}"] = np.log1p(out[c])

    return out.fillna(0.0)

def load_receipts(path):
    if not os.path.exists(path): return None
    try:
        d = joblib_load(path)
        # Try to find dataframe
        if isinstance(d, dict):
            for k in ["lineitems_df","lineitems","items_df"]:
                if k in d and isinstance(d[k], pd.DataFrame): return build_receipt_features_from_lineitems(d[k])
        if isinstance(d, pd.DataFrame): return build_receipt_features_from_lineitems(d)
    except: pass
    return None

# -----------------------------
# Features
# -----------------------------
def build_features(ed, pts, adm, rcpt):
    f = ed.copy()
    
    # Chronic
    cmap = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    f["primary_chronic"] = f["primary_chronic"].astype(str)
    f["chronic_encoded"] = f["primary_chronic"].str.upper().map(cmap).fillna(-1).astype(float)
    
    # Prior Cost (Transformations)
    f["prior_ed_cost_5y_usd"] = pd.to_numeric(f["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0).clip(lower=0.0)
    f["prior_ed_visits_5y"] = pd.to_numeric(f["prior_ed_visits_5y"], errors="coerce").fillna(0.0).clip(lower=0.0)
    
    f["log_prior_cost"] = np.log1p(f["prior_ed_cost_5y_usd"])
    f["sqrt_prior_cost"] = np.sqrt(f["prior_ed_cost_5y_usd"])
    f["cost_per_visit"] = f["prior_ed_cost_5y_usd"] / f["prior_ed_visits_5y"].clip(lower=1.0)
    f["baseline_next3y"] = f["prior_ed_cost_5y_usd"] * 0.6
    
    # Patients
    p = pts.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper()=="M").astype(int)
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = p["insurance"].astype(str).str.lower().map(ins_map).fillna(-1)
    p["zip_region"] = pd.to_numeric(p["zip3"].apply(standardize_zip3).str[0], errors="coerce").fillna(-1)
    
    f = f.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    
    # Interactions
    f["ins_x_chronic"] = f["insurance_encoded"] * f["chronic_encoded"]
    # === NEW: Age x Chronic (Risk Multiplier) ===
    f["age_x_chronic"] = f["age"] * (f["chronic_encoded"] + 1)
    
    # Merge External
    if adm is not None:
        f = f.merge(adm, on="patient_id", how="left")
    if rcpt is not None:
        f = f.merge(rcpt, on="patient_id", how="left")
        
    # Final Fill
    num_cols = [c for c in f.columns if c not in ["patient_id","primary_chronic",TARGET]]
    for c in num_cols:
        f[c] = pd.to_numeric(f[c], errors="coerce")
        if f[c].isna().any():
            f[c] = f[c].fillna(f[c].median())
            
    return f

# -----------------------------
# Train
# -----------------------------
def train_and_predict(train_df, test_df, feat_cols):
    y = train_df[TARGET].values
    
    # Stratify
    tmp = train_df[["primary_chronic", TARGET]].copy()
    tmp["bin"] = pd.qcut(tmp[TARGET], 5, labels=False)
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"])
    
    # === MODELS ===
    # A/B/C from Code 18
    # D: Huber Loss + Depth 3 (The Anchor)
    model_specs = {
        "A_RMSE_d5": {
            "loss_function": "RMSE", "depth": 5, "l2_leaf_reg": 8, "min_data_in_leaf": 40,
            "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1
        },
        "B_RMSE_d4": {
            "loss_function": "RMSE", "depth": 4, "l2_leaf_reg": 6, "min_data_in_leaf": 50,
            "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 2
        },
        "C_MAE_d4": {
            "loss_function": "MAE", "depth": 4, "l2_leaf_reg": 12, "min_data_in_leaf": 55,
            "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1.5
        },
        "D_HUBER_d3": { # <--- NEW ANCHOR
            "loss_function": "Huber:delta=1.35", # Robust to outliers, diff from MAE
            "depth": 3, # Ultra High Bias / Low Variance
            "l2_leaf_reg": 5, "min_data_in_leaf": 30,
            "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1
        }
    }
    
    oof_preds = {m: [] for m in model_specs}
    test_preds = {m: [] for m in model_specs}
    
    print(f"\n[Training] 4 Models (Huber Added) x {CFG.N_SEEDS} Seeds...")
    
    for seed in range(CFG.N_SEEDS):
        rs = SEED + seed * 11
        skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)
        
        s_oof = {m: np.zeros(len(train_df)) for m in model_specs}
        s_test = {m: np.zeros(len(test_df)) for m in model_specs}
        
        for fold, (ti, vi) in enumerate(skf.split(train_df, strat)):
            X_tr, y_tr = train_df.iloc[ti][feat_cols], y[ti]
            X_va, y_va = train_df.iloc[vi][feat_cols], y[vi]
            X_te = test_df[feat_cols]
            
            for m, params in model_specs.items():
                p = params.copy()
                p.update({
                    "iterations": CFG.ITERS, "learning_rate": CFG.LR, "early_stopping_rounds": CFG.ES_ROUNDS,
                    "verbose": 0, "allow_writing_files": False, "thread_count": -1,
                    "random_seed": rs + fold + stable_hash(m)
                })
                
                model = CatBoostRegressor(**p)
                model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
                
                s_oof[m][vi] = np.clip(model.predict(X_va), 0, None)
                s_test[m] += np.clip(model.predict(X_te), 0, None) / CFG.N_FOLDS
        
        for m in model_specs:
            oof_preds[m].append(s_oof[m])
            test_preds[m].append(s_test[m])
        
        print(f"  Seed {seed+1} MAE: " + " | ".join([f"{m}={mean_absolute_error(y, s_oof[m]):.2f}" for m in model_specs]))

    return oof_preds, test_preds

# -----------------------------
# Ensemble
# -----------------------------
def optimize_ensemble(oof_preds, baseline, y):
    models = list(oof_preds.keys())
    # Trimmed mean aggregation
    oof_agg = {m: trimmed_mean(np.vstack(oof_preds[m]), trim_k=CFG.TRIM_K) for m in models}
    
    best_score = float('inf')
    best_w = None
    best_params = None
    
    # 4-Model Grid Search (Coarse to fit in time)
    # Reducing grid density for 4 dims
    steps = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    
    print("\n[Ensemble] Searching weights (A/B/C/D)...")
    for wA in steps:
        for wB in steps:
            if wA + wB > 1.0: continue
            for wC in steps:
                if wA + wB + wC > 1.0: continue
                wD = 1.0 - wA - wB - wC
                if wD < -1e-9: continue
                
                # Baseline blend & Shift
                for lam in CFG.LAM_GRID:
                    pred = wA*oof_agg[models[0]] + wB*oof_agg[models[1]] + wC*oof_agg[models[2]] + wD*oof_agg[models[3]]
                    pred = (1-lam)*pred + lam*baseline
                    
                    shift = np.median(y - pred) * 0.5 # Conservative shift guess
                    pred_s = pred + shift
                    
                    mae = mean_absolute_error(y, pred_s)
                    if mae < best_score:
                        best_score = mae
                        best_w = [wA, wB, wC, wD]
                        best_params = {"lam": lam, "shift": shift}
                        
    print(f"  Best MAE: {best_score:.4f}")
    print(f"  Weights: {dict(zip(models, best_w))}")
    print(f"  Params: {best_params}")
    
    return best_w, best_params

# -----------------------------
# Main
# -----------------------------
must_exist(TRAIN_PATH, "TRAIN")
print("\n[Load & Feats]")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
pts = pd.read_csv(PATIENTS_PATH)
adm = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
rcpt = load_receipts(RECEIPTS_JOBLIB_PATH)

train_feat = build_features(train, pts, adm, rcpt)
test_feat = build_features(test, pts, adm, rcpt)

feat_cols = [c for c in train_feat.columns if c not in ["patient_id","primary_chronic",TARGET] and pd.api.types.is_numeric_dtype(train_feat[c])]
feat_cols = [c for c in feat_cols if train_feat[c].nunique() > 1]
print(f"  Features: {len(feat_cols)} (Added max_unit_price, age_x_chronic)")

# Train
oof_preds, test_preds = train_and_predict(train_feat, test_feat, feat_cols)

# Ensemble
y = train_feat[TARGET].values
base_oof = train_feat["baseline_next3y"].values
base_test = test_feat["baseline_next3y"].values

w, params = optimize_ensemble(oof_preds, base_oof, y)

# Construct Final
models = list(oof_preds.keys())
test_agg = {m: trimmed_mean(np.vstack(test_preds[m]), trim_k=CFG.TRIM_K) for m in models}
oof_agg = {m: trimmed_mean(np.vstack(oof_preds[m]), trim_k=CFG.TRIM_K) for m in models}

final_test = w[0]*test_agg[models[0]] + w[1]*test_agg[models[1]] + w[2]*test_agg[models[2]] + w[3]*test_agg[models[3]]
final_test = (1-params['lam'])*final_test + params['lam']*base_test + params['shift']

final_oof = w[0]*oof_agg[models[0]] + w[1]*oof_agg[models[1]] + w[2]*oof_agg[models[2]] + w[3]*oof_agg[models[3]]
final_oof = (1-params['lam'])*final_oof + params['lam']*base_oof + params['shift']

# === Post-Processing: Conservative Chronic Shift ===
# Only apply if it helps OOF
resid = y - final_oof
chronic = train_feat["primary_chronic"].astype(str).values
shifts = {}
best_oof_mae = mean_absolute_error(y, final_oof)
final_test_shifted = final_test.copy()

print(f"\n[Post-Process] Base OOF MAE: {best_oof_mae:.4f}")
# Try small shrinkage shifts
shrink = 0.25
for g in np.unique(chronic):
    med = np.median(resid[chronic==g])
    shifts[g] = med * shrink

# Apply to OOF to check
oof_shifted = final_oof.copy()
for g, s in shifts.items():
    oof_shifted[chronic==g] += s
    
new_mae = mean_absolute_error(y, oof_shifted)
if new_mae < best_oof_mae - 0.05:
    print(f"  Applying chronic shift (shrink={shrink}). New OOF MAE: {new_mae:.4f}")
    test_chronic = test_feat["primary_chronic"].astype(str).values
    for g, s in shifts.items():
        final_test_shifted[test_chronic==g] += s
else:
    print("  Chronic shift didn't improve OOF significantly. Keeping base.")
    final_test_shifted = final_test

# Clip
final_test_shifted = np.clip(final_test_shifted, 0, np.max(y)*1.5)

# Save
sub = pd.DataFrame({
    "patient_id": test["patient_id"].values,
    "ed_cost_next3y_usd": np.round(final_test_shifted, 2)
})
sub.to_csv(OUT_SUB_PATH, index=False)
print(f"\nSaved to {OUT_SUB_PATH}")
print(f"Predictions stats: {qdict(sub['ed_cost_next3y_usd'].values)}")

# # === CODE 19 / "The Huber-Anchor" (Aiming for 448.xx) ===
# # Base: Code 18 (MAE 449.41)
# # Improvement 1: Add "Model D" using Huber Loss + Depth 3 (Ultra-Robust Anchor).
# # Improvement 2: Add 'max_unit_price' to receipts (Severity Intensity).
# # Improvement 3: Add 'age_x_chronic' interaction.
# # Logic: Huber loss handles the heavy tail better than RMSE without the slowness of MAE.
# # Output: D:\AgentDs\agent_ds_healthcare\submission.csv

# import os, re, sys, gc, math, warnings, random, zlib
# from pathlib import Path
# from typing import Dict, List, Tuple, Optional

# import numpy as np
# import pandas as pd

# warnings.filterwarnings("ignore")
# pd.options.mode.chained_assignment = None

# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)

# # -----------------------------
# # Paths
# # -----------------------------
# DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
# TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
# TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
# PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
# ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
# ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
# RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
# OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
# ID_COL = "patient_id"
# TARGET = "ed_cost_next3y_usd"

# print("="*95)
# print("CODE 19 | The Huber-Anchor Strategy")
# print("Base: Code 18 (449.4)")
# print("New:  + Huber Loss Model (Depth 3) + Max Unit Price Feature")
# print("="*95)

# # -----------------------------
# # Deps
# # -----------------------------
# def _pip_install(pkg: str):
#     import subprocess
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# try:
#     from joblib import load as joblib_load
# except Exception:
#     _pip_install("joblib")
#     from joblib import load as joblib_load

# try:
#     from sklearn.model_selection import StratifiedKFold
#     from sklearn.preprocessing import LabelEncoder
#     from sklearn.metrics import mean_absolute_error
# except Exception:
#     _pip_install("scikit-learn")
#     from sklearn.model_selection import StratifiedKFold
#     from sklearn.preprocessing import LabelEncoder
#     from sklearn.metrics import mean_absolute_error

# try:
#     from catboost import CatBoostRegressor
# except Exception:
#     _pip_install("catboost")
#     from catboost import CatBoostRegressor

# # -----------------------------
# # Config
# # -----------------------------
# class CFG:
#     N_FOLDS = 7
#     N_SEEDS = 5
#     ITERS = 3500 # Huber needs slightly more iters
#     ES_ROUNDS = 150
#     LR = 0.03
#     RSM = 0.75      # Slightly tighter
#     SUBSAMPLE = 0.75 # Slightly tighter to force diversity
    
#     # Aggregation
#     TRIM_K = 1 
    
#     # Ensemble Search (4 models now)
#     W_STEP = 0.05
#     LAM_GRID = [0.00, 0.02, 0.05, 0.10]
#     SHIFT_GRID = [0.0, 0.5, 1.0]
    
#     # Penalties (Keep robust)
#     STD_PEN = 0.20
#     LAM_PEN = 5.0
#     SHIFT_PEN = 0.002

# # -----------------------------
# # Utilities
# # -----------------------------
# def must_exist(path: str, name: str):
#     if not os.path.exists(path):
#         raise FileNotFoundError(f"Missing {name}: {path}")

# def log_shape(name: str, df: pd.DataFrame):
#     mem = df.memory_usage(deep=True).sum() / (1024**2)
#     print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

# def qdict(x, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)):
#     x = np.asarray(x, dtype=float)
#     return {q: float(np.quantile(x, q)) for q in qs}

# def standardize_zip3(z):
#     if z is None or (isinstance(z, float) and np.isnan(z)): return None
#     s = re.sub(r"\D", "", str(z).strip())
#     return s.zfill(3) if s else None

# def norm_code(x):
#     if x is None or (isinstance(x, float) and np.isnan(x)): return None
#     s = str(x).strip().upper()
#     if s == "" or s.lower() == "nan": return None
#     if re.fullmatch(r"\d+\.0+", s): s = s.split(".")[0]
#     s = re.sub(r"\s+", "", s)
#     return s

# def safe_num_series(s: pd.Series, default=0.0) -> pd.Series:
#     v = pd.to_numeric(s, errors="coerce")
#     v = v.replace([np.inf, -np.inf], np.nan).fillna(default)
#     return v

# def stable_hash(s: str) -> int:
#     return int(zlib.crc32(s.encode("utf-8")) & 0xffffffff)

# def gini(x: np.ndarray) -> float:
#     x = np.asarray(x, dtype=float)
#     x = x[~np.isnan(x)]
#     if x.size == 0 or np.all(x == 0): return 0.0
#     x = np.sort(x)
#     n = x.size
#     cumx = np.cumsum(x)
#     return float((n + 1 - 2*np.sum(cumx)/cumx[-1]) / n)

# def trimmed_mean(mat: np.ndarray, trim_k: int = 1) -> np.ndarray:
#     mat = np.asarray(mat, dtype=float)
#     if mat.ndim != 2: raise ValueError("trimmed_mean expects 2D array")
#     n = mat.shape[0]
#     if trim_k <= 0 or n < 2*trim_k + 1: return np.mean(mat, axis=0)
#     s = np.sort(mat, axis=0)
#     return np.mean(s[trim_k:n-trim_k, :], axis=0)

# # -----------------------------
# # Admissions
# # -----------------------------
# def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
#     dfs = []
#     for path in [adm_train_path, adm_test_path]:
#         if os.path.exists(path):
#             df = pd.read_csv(path)
#             if "readmit_30d" in df.columns: df = df.drop(columns=["readmit_30d"])
#             dfs.append(df)
#     if not dfs: return None
#     adm = pd.concat(dfs, ignore_index=True)
    
#     for c in ["patient_id","charlson_band","acuity_emergent"]:
#         adm[c] = pd.to_numeric(adm[c], errors="coerce")
    
#     out = adm.groupby("patient_id").agg(
#         charlson_max=("charlson_band","max"),
#         charlson_mean=("charlson_band","mean"),
#         pct_emergent=("acuity_emergent","mean"),
#     ).reset_index()
    
#     for c in ["charlson_max","charlson_mean","pct_emergent"]:
#         out[c] = safe_num_series(out[c], default=0.0)
    
#     out["patient_id"] = out["patient_id"].fillna(-1).astype(int)
#     out = out[out["patient_id"] != -1]
#     return out

# # -----------------------------
# # Receipts (Code 18 + max_unit_price)
# # -----------------------------
# def build_receipt_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
#     li = li.copy()
    
#     # Column mapping
#     cols = li.columns
#     code_col = next((c for c in ["code","cpt","cpt_code","hcpcs","proc_code"] if c in cols), None)
#     total_col = next((c for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"] if c in cols), None)
#     unit_col = next((c for c in ["unit_price","unitprice","unit_cost","unitcost"] if c in cols), None)
#     qty_col = next((c for c in ["qty","quantity","units"] if c in cols), None)

#     if "patient_id" not in cols or not code_col or not total_col:
#         raise ValueError("Missing essential columns")

#     li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
#     li = li.dropna(subset=["patient_id"]).copy()
#     li["patient_id"] = li["patient_id"].astype(int)
    
#     li["code"] = li[code_col].map(norm_code)
#     li = li.dropna(subset=["code"]).copy()
    
#     li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).clip(lower=0.0)
#     li["unit_price"] = pd.to_numeric(li[unit_col], errors="coerce") if unit_col else np.nan
    
#     # Totals
#     receipt_total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
#     li = li.join(receipt_total, on="patient_id")
#     denom = li["receipt_total"].replace(0.0, np.nan)
#     share = (li["amt"] / denom).fillna(0.0)

#     # Robust Stats
#     cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
#     top1_share = share.groupby(li["patient_id"]).max().rename("top1_share")
#     gini_amt = li.groupby("patient_id")["amt"].apply(lambda s: gini(s.values)).rename("gini_line_total")
#     max_line_total = li.groupby("patient_id")["amt"].max().rename("max_line_total")

#     # Buckets
#     is_em = li["code"].isin(["99281","99282","99283","99284","99285"])
#     em_level = li["code"].map({"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}).fillna(0).astype(int)
#     is_crit = li["code"].isin(["99291","99292"])
#     is_obs = li["code"].str.startswith("G037", na=False)
#     is_high = li["code"].isin(["31500","36556","32551","36620","92950"])
    
#     # Basic Counts
#     n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
#     n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
#     max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    
#     # Flags
#     has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
#     has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
#     has_99285 = li["code"].eq("99285").astype(int).groupby(li["patient_id"]).max().rename("has_99285")

#     # === NEW: Max Unit Price (Severity Proxy) ===
#     # Using max unit price tells us the most expensive single service provided (e.g. surgery vs just Tylenol)
#     # independent of quantity.
#     if li["unit_price"].isna().all():
#         max_unit_p = pd.Series(0.0, index=li["patient_id"].unique(), name="max_unit_price")
#         med_unit_p = pd.Series(0.0, index=li["patient_id"].unique(), name="median_unit_price")
#     else:
#         max_unit_p = li.groupby("patient_id")["unit_price"].max().rename("max_unit_price")
#         med_unit_p = li.groupby("patient_id")["unit_price"].median().rename("median_unit_price")
        
#     # Aggregate
#     out = pd.concat([
#         n_unique_codes, receipt_total, cost_hhi, top1_share, gini_amt, max_line_total,
#         n_em_codes, max_em_level,
#         has_critical_care, has_high_acuity, has_99285,
#         med_unit_p, max_unit_p
#     ], axis=1).reset_index()
    
#     out["cost_per_em"] = np.where(out["n_em_codes"]>0, out["receipt_total"]/out["n_em_codes"].clip(lower=1), out["receipt_total"])
    
#     # Logs
#     for c in ["max_line_total", "median_unit_price", "max_unit_price"]:
#         out[c] = out[c].fillna(0.0)
#         out[f"log1p_{c}"] = np.log1p(out[c])

#     return out.fillna(0.0)

# def load_receipts(path):
#     if not os.path.exists(path): return None
#     try:
#         d = joblib_load(path)
#         # Try to find dataframe
#         if isinstance(d, dict):
#             for k in ["lineitems_df","lineitems","items_df"]:
#                 if k in d and isinstance(d[k], pd.DataFrame): return build_receipt_features_from_lineitems(d[k])
#         if isinstance(d, pd.DataFrame): return build_receipt_features_from_lineitems(d)
#     except: pass
#     return None

# # -----------------------------
# # Features
# # -----------------------------
# def build_features(ed, pts, adm, rcpt):
#     f = ed.copy()
    
#     # Chronic
#     cmap = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
#     f["primary_chronic"] = f["primary_chronic"].astype(str)
#     f["chronic_encoded"] = f["primary_chronic"].str.upper().map(cmap).fillna(-1).astype(float)
    
#     # Prior Cost (Transformations)
#     f["prior_ed_cost_5y_usd"] = pd.to_numeric(f["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0).clip(lower=0.0)
#     f["prior_ed_visits_5y"] = pd.to_numeric(f["prior_ed_visits_5y"], errors="coerce").fillna(0.0).clip(lower=0.0)
    
#     f["log_prior_cost"] = np.log1p(f["prior_ed_cost_5y_usd"])
#     f["sqrt_prior_cost"] = np.sqrt(f["prior_ed_cost_5y_usd"])
#     f["cost_per_visit"] = f["prior_ed_cost_5y_usd"] / f["prior_ed_visits_5y"].clip(lower=1.0)
#     f["baseline_next3y"] = f["prior_ed_cost_5y_usd"] * 0.6
    
#     # Patients
#     p = pts.copy()
#     p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
#     p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
#     p["sex_encoded"] = (p["sex"].astype(str).str.upper()=="M").astype(int)
#     ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
#     p["insurance_encoded"] = p["insurance"].astype(str).str.lower().map(ins_map).fillna(-1)
#     p["zip_region"] = pd.to_numeric(p["zip3"].apply(standardize_zip3).str[0], errors="coerce").fillna(-1)
    
#     f = f.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    
#     # Interactions
#     f["ins_x_chronic"] = f["insurance_encoded"] * f["chronic_encoded"]
#     # === NEW: Age x Chronic (Risk Multiplier) ===
#     f["age_x_chronic"] = f["age"] * (f["chronic_encoded"] + 1)
    
#     # Merge External
#     if adm is not None:
#         f = f.merge(adm, on="patient_id", how="left")
#     if rcpt is not None:
#         f = f.merge(rcpt, on="patient_id", how="left")
        
#     # Final Fill
#     num_cols = [c for c in f.columns if c not in ["patient_id","primary_chronic",TARGET]]
#     for c in num_cols:
#         f[c] = pd.to_numeric(f[c], errors="coerce")
#         if f[c].isna().any():
#             f[c] = f[c].fillna(f[c].median())
            
#     return f

# # -----------------------------
# # Train
# # -----------------------------
# def train_and_predict(train_df, test_df, feat_cols):
#     y = train_df[TARGET].values
    
#     # Stratify
#     tmp = train_df[["primary_chronic", TARGET]].copy()
#     tmp["bin"] = pd.qcut(tmp[TARGET], 5, labels=False)
#     tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["bin"].astype(str)
#     strat = LabelEncoder().fit_transform(tmp["strat"])
    
#     # === MODELS ===
#     # A/B/C from Code 18
#     # D: Huber Loss + Depth 3 (The Anchor)
#     model_specs = {
#         "A_RMSE_d5": {
#             "loss_function": "RMSE", "depth": 5, "l2_leaf_reg": 8, "min_data_in_leaf": 40,
#             "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1
#         },
#         "B_RMSE_d4": {
#             "loss_function": "RMSE", "depth": 4, "l2_leaf_reg": 6, "min_data_in_leaf": 50,
#             "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 2
#         },
#         "C_MAE_d4": {
#             "loss_function": "MAE", "depth": 4, "l2_leaf_reg": 12, "min_data_in_leaf": 55,
#             "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1.5
#         },
#         "D_HUBER_d3": { # <--- NEW ANCHOR
#             "loss_function": "Huber:delta=1.35", # Robust to outliers, diff from MAE
#             "depth": 3, # Ultra High Bias / Low Variance
#             "l2_leaf_reg": 5, "min_data_in_leaf": 30,
#             "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1
#         }
#     }
    
#     oof_preds = {m: [] for m in model_specs}
#     test_preds = {m: [] for m in model_specs}
    
#     print(f"\n[Training] 4 Models (Huber Added) x {CFG.N_SEEDS} Seeds...")
    
#     for seed in range(CFG.N_SEEDS):
#         rs = SEED + seed * 11
#         skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)
        
#         s_oof = {m: np.zeros(len(train_df)) for m in model_specs}
#         s_test = {m: np.zeros(len(test_df)) for m in model_specs}
        
#         for fold, (ti, vi) in enumerate(skf.split(train_df, strat)):
#             X_tr, y_tr = train_df.iloc[ti][feat_cols], y[ti]
#             X_va, y_va = train_df.iloc[vi][feat_cols], y[vi]
#             X_te = test_df[feat_cols]
            
#             for m, params in model_specs.items():
#                 p = params.copy()
#                 p.update({
#                     "iterations": CFG.ITERS, "learning_rate": CFG.LR, "early_stopping_rounds": CFG.ES_ROUNDS,
#                     "verbose": 0, "allow_writing_files": False, "thread_count": -1,
#                     "random_seed": rs + fold + stable_hash(m)
#                 })
                
#                 model = CatBoostRegressor(**p)
#                 model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
                
#                 s_oof[m][vi] = np.clip(model.predict(X_va), 0, None)
#                 s_test[m] += np.clip(model.predict(X_te), 0, None) / CFG.N_FOLDS
        
#         for m in model_specs:
#             oof_preds[m].append(s_oof[m])
#             test_preds[m].append(s_test[m])
        
#         print(f"  Seed {seed+1} MAE: " + " | ".join([f"{m}={mean_absolute_error(y, s_oof[m]):.2f}" for m in model_specs]))

#     return oof_preds, test_preds

# # -----------------------------
# # Ensemble
# # -----------------------------
# def optimize_ensemble(oof_preds, baseline, y):
#     models = list(oof_preds.keys())
#     # Trimmed mean aggregation
#     oof_agg = {m: trimmed_mean(np.vstack(oof_preds[m]), trim_k=CFG.TRIM_K) for m in models}
    
#     best_score = float('inf')
#     best_w = None
#     best_params = None
    
#     # 4-Model Grid Search (Coarse to fit in time)
#     # Reducing grid density for 4 dims
#     steps = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    
#     print("\n[Ensemble] Searching weights (A/B/C/D)...")
#     for wA in steps:
#         for wB in steps:
#             if wA + wB > 1.0: continue
#             for wC in steps:
#                 if wA + wB + wC > 1.0: continue
#                 wD = 1.0 - wA - wB - wC
#                 if wD < -1e-9: continue
                
#                 # Baseline blend & Shift
#                 for lam in CFG.LAM_GRID:
#                     pred = wA*oof_agg[models[0]] + wB*oof_agg[models[1]] + wC*oof_agg[models[2]] + wD*oof_agg[models[3]]
#                     pred = (1-lam)*pred + lam*baseline
                    
#                     shift = np.median(y - pred) * 0.5 # Conservative shift guess
#                     pred_s = pred + shift
                    
#                     mae = mean_absolute_error(y, pred_s)
#                     if mae < best_score:
#                         best_score = mae
#                         best_w = [wA, wB, wC, wD]
#                         best_params = {"lam": lam, "shift": shift}
                        
#     print(f"  Best MAE: {best_score:.4f}")
#     print(f"  Weights: {dict(zip(models, best_w))}")
#     print(f"  Params: {best_params}")
    
#     return best_w, best_params

# # -----------------------------
# # Main
# # -----------------------------
# must_exist(TRAIN_PATH, "TRAIN")
# print("\n[Load & Feats]")
# train = pd.read_csv(TRAIN_PATH)
# test = pd.read_csv(TEST_PATH)
# pts = pd.read_csv(PATIENTS_PATH)
# adm = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
# rcpt = load_receipts(RECEIPTS_JOBLIB_PATH)

# train_feat = build_features(train, pts, adm, rcpt)
# test_feat = build_features(test, pts, adm, rcpt)

# feat_cols = [c for c in train_feat.columns if c not in ["patient_id","primary_chronic",TARGET] and pd.api.types.is_numeric_dtype(train_feat[c])]
# feat_cols = [c for c in feat_cols if train_feat[c].nunique() > 1]
# print(f"  Features: {len(feat_cols)} (Added max_unit_price, age_x_chronic)")

# # Train
# oof_preds, test_preds = train_and_predict(train_feat, test_feat, feat_cols)

# # Ensemble
# y = train_feat[TARGET].values
# base_oof = train_feat["baseline_next3y"].values
# base_test = test_feat["baseline_next3y"].values

# w, params = optimize_ensemble(oof_preds, base_oof, y)

# # Construct Final
# models = list(oof_preds.keys())
# test_agg = {m: trimmed_mean(np.vstack(test_preds[m]), trim_k=CFG.TRIM_K) for m in models}
# oof_agg = {m: trimmed_mean(np.vstack(oof_preds[m]), trim_k=CFG.TRIM_K) for m in models}

# final_test = w[0]*test_agg[models[0]] + w[1]*test_agg[models[1]] + w[2]*test_agg[models[2]] + w[3]*test_agg[models[3]]
# final_test = (1-params['lam'])*final_test + params['lam']*base_test + params['shift']

# final_oof = w[0]*oof_agg[models[0]] + w[1]*oof_agg[models[1]] + w[2]*oof_agg[models[2]] + w[3]*oof_agg[models[3]]
# final_oof = (1-params['lam'])*final_oof + params['lam']*base_oof + params['shift']

# # === Post-Processing: Conservative Chronic Shift ===
# # Only apply if it helps OOF
# resid = y - final_oof
# chronic = train_feat["primary_chronic"].astype(str).values
# shifts = {}
# best_oof_mae = mean_absolute_error(y, final_oof)
# final_test_shifted = final_test.copy()

# print(f"\n[Post-Process] Base OOF MAE: {best_oof_mae:.4f}")
# # Try small shrinkage shifts
# shrink = 0.25
# for g in np.unique(chronic):
#     med = np.median(resid[chronic==g])
#     shifts[g] = med * shrink

# # Apply to OOF to check
# oof_shifted = final_oof.copy()
# for g, s in shifts.items():
#     oof_shifted[chronic==g] += s
    
# new_mae = mean_absolute_error(y, oof_shifted)
# if new_mae < best_oof_mae - 0.05:
#     print(f"  Applying chronic shift (shrink={shrink}). New OOF MAE: {new_mae:.4f}")
#     test_chronic = test_feat["primary_chronic"].astype(str).values
#     for g, s in shifts.items():
#         final_test_shifted[test_chronic==g] += s
# else:
#     print("  Chronic shift didn't improve OOF significantly. Keeping base.")
#     final_test_shifted = final_test

# # Clip
# final_test_shifted = np.clip(final_test_shifted, 0, np.max(y)*1.5)

# # Save
# sub = pd.DataFrame({
#     "patient_id": test["patient_id"].values,
# #     "ed_cost_next3y_usd": np.round(final_test_shifted, 2)
# })
# sub.to_csv(OUT_SUB_PATH, index=False)
# print(f"\nSaved to {OUT_SUB_PATH}")
# print(f"Predictions stats: {qdict(sub['ed_cost_next3y_usd'].values)}")

CODE 19 | The Huber-Anchor Strategy
Base: Code 18 (449.4)
New:  + Huber Loss Model (Depth 3) + Max Unit Price Feature

[Load & Feats]
  Features: 33 (Added max_unit_price, age_x_chronic)

[Training] 4 Models (Huber Added) x 5 Seeds...
  Seed 1 MAE: A_RMSE_d5=433.96 | B_RMSE_d4=434.01 | C_MAE_d4=439.67 | D_HUBER_d3=577.03
  Seed 2 MAE: A_RMSE_d5=435.15 | B_RMSE_d4=434.91 | C_MAE_d4=445.52 | D_HUBER_d3=578.84
  Seed 3 MAE: A_RMSE_d5=436.08 | B_RMSE_d4=436.47 | C_MAE_d4=443.60 | D_HUBER_d3=577.10
  Seed 4 MAE: A_RMSE_d5=435.20 | B_RMSE_d4=436.89 | C_MAE_d4=442.08 | D_HUBER_d3=576.20
  Seed 5 MAE: A_RMSE_d5=436.01 | B_RMSE_d4=435.73 | C_MAE_d4=442.73 | D_HUBER_d3=578.59

[Ensemble] Searching weights (A/B/C/D)...
  Best MAE: 431.9494
  Weights: {'A_RMSE_d5': 0.4, 'B_RMSE_d4': 0.6, 'C_MAE_d4': 0.0, 'D_HUBER_d3': 0.0}
  Params: {'lam': 0.0, 'shift': np.float64(-0.2411779593975325)}

[Post-Process] Base OOF MAE: 431.9494
  Chronic shift didn't improve OOF significantly. Keeping base.

Saved to

KeyboardInterrupt: 

# Iteration 8
- 451.5770

In [23]:
# === CODE 20 / "The Log-Hybrid" (Fixing the Huber Scale Issue) ===
# Base: Code 19 features (max_unit_price is GOOD)
# Fix: Replace Failed Huber Model -> Log-Target RMSE Model (Standard for Cost Data)
# Logic: Mixing Raw-Target Models (A/B) with a Log-Target Model (D) reduces variance significantly.
# Output: D:\AgentDs\agent_ds_healthcare\submission.csv

import os, re, sys, gc, math, warnings, random, zlib
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
ID_COL = "patient_id"
TARGET = "ed_cost_next3y_usd"

print("="*95)
print("CODE 20 | The Log-Hybrid Strategy")
print("Fix: Replaced failed Huber model with Log-Target RMSE")
print("Goal: Combine Raw-Scale robustness (Models A/B) with Log-Scale structure (Model D)")
print("="*95)

# -----------------------------
# Deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor

# -----------------------------
# Config
# -----------------------------
class CFG:
    N_FOLDS = 7
    N_SEEDS = 5
    ITERS = 3200
    ES_ROUNDS = 150
    LR = 0.03
    RSM = 0.75
    SUBSAMPLE = 0.75 
    
    # Aggregation
    TRIM_K = 1 
    
    # Ensemble Search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.02, 0.05, 0.10]
    SHIFT_GRID = [0.0, 0.5, 1.0] # Conservative shift search
    
    # Penalties
    STD_PEN = 0.20
    LAM_PEN = 5.0
    SHIFT_PEN = 0.002

# -----------------------------
# Utilities
# -----------------------------
def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)): return None
    s = re.sub(r"\D", "", str(z).strip())
    return s.zfill(3) if s else None

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan": return None
    if re.fullmatch(r"\d+\.0+", s): s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def safe_num_series(s: pd.Series, default=0.0) -> pd.Series:
    v = pd.to_numeric(s, errors="coerce")
    v = v.replace([np.inf, -np.inf], np.nan).fillna(default)
    return v

def stable_hash(s: str) -> int:
    return int(zlib.crc32(s.encode("utf-8")) & 0xffffffff)

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0 or np.all(x == 0): return 0.0
    x = np.sort(x)
    n = x.size
    cumx = np.cumsum(x)
    return float((n + 1 - 2*np.sum(cumx)/cumx[-1]) / n)

def trimmed_mean(mat: np.ndarray, trim_k: int = 1) -> np.ndarray:
    mat = np.asarray(mat, dtype=float)
    if mat.ndim != 2: raise ValueError("trimmed_mean expects 2D array")
    n = mat.shape[0]
    if trim_k <= 0 or n < 2*trim_k + 1: return np.mean(mat, axis=0)
    s = np.sort(mat, axis=0)
    return np.mean(s[trim_k:n-trim_k, :], axis=0)

# -----------------------------
# Admissions
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns: df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs: return None
    adm = pd.concat(dfs, ignore_index=True)
    
    for c in ["patient_id","charlson_band","acuity_emergent"]:
        adm[c] = pd.to_numeric(adm[c], errors="coerce")
    
    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()
    
    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = safe_num_series(out[c], default=0.0)
    
    out["patient_id"] = out["patient_id"].fillna(-1).astype(int)
    out = out[out["patient_id"] != -1]
    return out

# -----------------------------
# Receipts
# -----------------------------
def build_receipt_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()
    
    cols = li.columns
    code_col = next((c for c in ["code","cpt","cpt_code","hcpcs","proc_code"] if c in cols), None)
    total_col = next((c for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price"] if c in cols), None)
    unit_col = next((c for c in ["unit_price","unitprice","unit_cost","unitcost"] if c in cols), None)

    if "patient_id" not in cols or not code_col or not total_col:
        raise ValueError("Missing essential columns")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)
    
    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()
    
    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    li["unit_price"] = pd.to_numeric(li[unit_col], errors="coerce") if unit_col else np.nan
    
    # Totals
    receipt_total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(receipt_total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)
    share = (li["amt"] / denom).fillna(0.0)

    # Robust Stats
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    top1_share = share.groupby(li["patient_id"]).max().rename("top1_share")
    gini_amt = li.groupby("patient_id")["amt"].apply(lambda s: gini(s.values)).rename("gini_line_total")
    max_line_total = li.groupby("patient_id")["amt"].max().rename("max_line_total")

    # Buckets
    is_em = li["code"].isin(["99281","99282","99283","99284","99285"])
    em_level = li["code"].map({"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}).fillna(0).astype(int)
    is_crit = li["code"].isin(["99291","99292"])
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])
    
    # Basic Counts
    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    
    # Flags
    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_99285 = li["code"].eq("99285").astype(int).groupby(li["patient_id"]).max().rename("has_99285")

    # Max Unit Price (Code 19 Feature - KEEPING)
    if li["unit_price"].isna().all():
        max_unit_p = pd.Series(0.0, index=li["patient_id"].unique(), name="max_unit_price")
        med_unit_p = pd.Series(0.0, index=li["patient_id"].unique(), name="median_unit_price")
    else:
        max_unit_p = li.groupby("patient_id")["unit_price"].max().rename("max_unit_price")
        med_unit_p = li.groupby("patient_id")["unit_price"].median().rename("median_unit_price")
        
    out = pd.concat([
        n_unique_codes, receipt_total, cost_hhi, top1_share, gini_amt, max_line_total,
        n_em_codes, max_em_level,
        has_critical_care, has_high_acuity, has_99285,
        med_unit_p, max_unit_p
    ], axis=1).reset_index()
    
    out["cost_per_em"] = np.where(out["n_em_codes"]>0, out["receipt_total"]/out["n_em_codes"].clip(lower=1), out["receipt_total"])
    
    # Logs
    for c in ["max_line_total", "median_unit_price", "max_unit_price"]:
        out[c] = out[c].fillna(0.0)
        out[f"log1p_{c}"] = np.log1p(out[c])

    return out.fillna(0.0)

def load_receipts(path):
    if not os.path.exists(path): return None
    try:
        d = joblib_load(path)
        if isinstance(d, dict):
            for k in ["lineitems_df","lineitems","items_df"]:
                if k in d and isinstance(d[k], pd.DataFrame): return build_receipt_features_from_lineitems(d[k])
        if isinstance(d, pd.DataFrame): return build_receipt_features_from_lineitems(d)
    except: pass
    return None

# -----------------------------
# Features
# -----------------------------
def build_features(ed, pts, adm, rcpt):
    f = ed.copy()
    
    # Chronic
    cmap = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    f["primary_chronic"] = f["primary_chronic"].astype(str)
    f["chronic_encoded"] = f["primary_chronic"].str.upper().map(cmap).fillna(-1).astype(float)
    
    # Prior Cost
    f["prior_ed_cost_5y_usd"] = pd.to_numeric(f["prior_ed_cost_5y_usd"], errors="coerce").fillna(0.0).clip(lower=0.0)
    f["prior_ed_visits_5y"] = pd.to_numeric(f["prior_ed_visits_5y"], errors="coerce").fillna(0.0).clip(lower=0.0)
    
    f["log_prior_cost"] = np.log1p(f["prior_ed_cost_5y_usd"])
    f["sqrt_prior_cost"] = np.sqrt(f["prior_ed_cost_5y_usd"])
    f["cost_per_visit"] = f["prior_ed_cost_5y_usd"] / f["prior_ed_visits_5y"].clip(lower=1.0)
    f["baseline_next3y"] = f["prior_ed_cost_5y_usd"] * 0.6
    
    # Patients
    p = pts.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce").fillna(p["age"].median())
    p["sex_encoded"] = (p["sex"].astype(str).str.upper()=="M").astype(int)
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = p["insurance"].astype(str).str.lower().map(ins_map).fillna(-1)
    p["zip_region"] = pd.to_numeric(p["zip3"].apply(standardize_zip3).str[0], errors="coerce").fillna(-1)
    
    f = f.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    
    # Interactions
    f["ins_x_chronic"] = f["insurance_encoded"] * f["chronic_encoded"]
    f["age_x_chronic"] = f["age"] * (f["chronic_encoded"] + 1)
    
    # Merge External
    if adm is not None:
        f = f.merge(adm, on="patient_id", how="left")
    if rcpt is not None:
        f = f.merge(rcpt, on="patient_id", how="left")
        
    # Final Fill
    num_cols = [c for c in f.columns if c not in ["patient_id","primary_chronic",TARGET]]
    for c in num_cols:
        f[c] = pd.to_numeric(f[c], errors="coerce")
        if f[c].isna().any():
            f[c] = f[c].fillna(f[c].median())
            
    return f

# -----------------------------
# Train
# -----------------------------
def train_and_predict(train_df, test_df, feat_cols):
    y = train_df[TARGET].values
    y_log = np.log1p(y) # For Model D
    
    # Stratify
    tmp = train_df[["primary_chronic", TARGET]].copy()
    tmp["bin"] = pd.qcut(tmp[TARGET], 5, labels=False)
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"])
    
    # === MODELS ===
    # Models A/B (Raw RMSE), C (MAE), D (Log RMSE)
    model_specs = {
        "A_RMSE_d5": {
            "loss_function": "RMSE", "depth": 5, "l2_leaf_reg": 8, "min_data_in_leaf": 40,
            "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1
        },
        "B_RMSE_d4": {
            "loss_function": "RMSE", "depth": 4, "l2_leaf_reg": 6, "min_data_in_leaf": 50,
            "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 2
        },
        "C_MAE_d4": {
            "loss_function": "MAE", "depth": 4, "l2_leaf_reg": 12, "min_data_in_leaf": 55,
            "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1.5
        },
        "D_LOG_d4": { # <--- REPLACED HUBER with LOG RMSE
            "loss_function": "RMSE", "depth": 4, "l2_leaf_reg": 8, "min_data_in_leaf": 45,
            "rsm": CFG.RSM, "bootstrap_type": "Bernoulli", "subsample": CFG.SUBSAMPLE, "random_strength": 1
        }
    }
    
    oof_preds = {m: [] for m in model_specs}
    test_preds = {m: [] for m in model_specs}
    
    print(f"\n[Training] 4 Models (Log-Target Fixed) x {CFG.N_SEEDS} Seeds...")
    
    for seed in range(CFG.N_SEEDS):
        rs = SEED + seed * 11
        skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)
        
        s_oof = {m: np.zeros(len(train_df)) for m in model_specs}
        s_test = {m: np.zeros(len(test_df)) for m in model_specs}
        
        for fold, (ti, vi) in enumerate(skf.split(train_df, strat)):
            X_tr = train_df.iloc[ti][feat_cols]
            X_va = train_df.iloc[vi][feat_cols]
            X_te = test_df[feat_cols]
            
            for m, params in model_specs.items():
                p = params.copy()
                p.update({
                    "iterations": CFG.ITERS, "learning_rate": CFG.LR, "early_stopping_rounds": CFG.ES_ROUNDS,
                    "verbose": 0, "allow_writing_files": False, "thread_count": -1,
                    "random_seed": rs + fold + stable_hash(m)
                })
                
                # Model D uses Log Target
                if m == "D_LOG_d4":
                    y_tr_use = y_log[ti]
                    y_va_use = y_log[vi]
                else:
                    y_tr_use = y[ti]
                    y_va_use = y[vi]
                
                model = CatBoostRegressor(**p)
                model.fit(X_tr, y_tr_use, eval_set=(X_va, y_va_use))
                
                pred_va = model.predict(X_va)
                pred_te = model.predict(X_te)
                
                if m == "D_LOG_d4":
                    pred_va = np.expm1(pred_va)
                    pred_te = np.expm1(pred_te)
                
                s_oof[m][vi] = np.clip(pred_va, 0, None)
                s_test[m] += np.clip(pred_te, 0, None) / CFG.N_FOLDS
        
        for m in model_specs:
            oof_preds[m].append(s_oof[m])
            test_preds[m].append(s_test[m])
        
        print(f"  Seed {seed+1} MAE: " + " | ".join([f"{m}={mean_absolute_error(y, s_oof[m]):.2f}" for m in model_specs]))

    return oof_preds, test_preds

# -----------------------------
# Ensemble
# -----------------------------
def optimize_ensemble(oof_preds, baseline, y):
    models = list(oof_preds.keys())
    oof_agg = {m: trimmed_mean(np.vstack(oof_preds[m]), trim_k=CFG.TRIM_K) for m in models}
    
    best_score = float('inf')
    best_w = None
    best_params = None
    
    # Coarse Grid
    steps = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    
    print("\n[Ensemble] Searching weights...")
    for wA in steps:
        for wB in steps:
            if wA + wB > 1.0: continue
            for wC in steps:
                if wA + wB + wC > 1.0: continue
                wD = 1.0 - wA - wB - wC
                if wD < -1e-9: continue
                
                for lam in CFG.LAM_GRID:
                    pred = wA*oof_agg[models[0]] + wB*oof_agg[models[1]] + wC*oof_agg[models[2]] + wD*oof_agg[models[3]]
                    pred = (1-lam)*pred + lam*baseline
                    
                    shift = np.median(y - pred) * 0.5 
                    pred_s = pred + shift
                    
                    mae = mean_absolute_error(y, pred_s)
                    if mae < best_score:
                        best_score = mae
                        best_w = [wA, wB, wC, wD]
                        best_params = {"lam": lam, "shift": shift}
                        
    print(f"  Best MAE: {best_score:.4f}")
    print(f"  Weights: {dict(zip(models, best_w))}")
    print(f"  Params: {best_params}")
    return best_w, best_params

# -----------------------------
# Main
# -----------------------------
must_exist(TRAIN_PATH, "TRAIN")
print("\n[Load & Feats]")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
pts = pd.read_csv(PATIENTS_PATH)
adm = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
rcpt = load_receipts(RECEIPTS_JOBLIB_PATH)

train_feat = build_features(train, pts, adm, rcpt)
test_feat = build_features(test, pts, adm, rcpt)

feat_cols = [c for c in train_feat.columns if c not in ["patient_id","primary_chronic",TARGET] and pd.api.types.is_numeric_dtype(train_feat[c])]
feat_cols = [c for c in feat_cols if train_feat[c].nunique() > 1]
print(f"  Features: {len(feat_cols)}")

# Train
oof_preds, test_preds = train_and_predict(train_feat, test_feat, feat_cols)

# Ensemble
y = train_feat[TARGET].values
base_oof = train_feat["baseline_next3y"].values
base_test = test_feat["baseline_next3y"].values

w, params = optimize_ensemble(oof_preds, base_oof, y)

# Construct Final
models = list(oof_preds.keys())
test_agg = {m: trimmed_mean(np.vstack(test_preds[m]), trim_k=CFG.TRIM_K) for m in models}
oof_agg = {m: trimmed_mean(np.vstack(oof_preds[m]), trim_k=CFG.TRIM_K) for m in models}

final_test = w[0]*test_agg[models[0]] + w[1]*test_agg[models[1]] + w[2]*test_agg[models[2]] + w[3]*test_agg[models[3]]
final_test = (1-params['lam'])*final_test + params['lam']*base_test + params['shift']

final_oof = w[0]*oof_agg[models[0]] + w[1]*oof_agg[models[1]] + w[2]*oof_agg[models[2]] + w[3]*oof_agg[models[3]]
final_oof = (1-params['lam'])*final_oof + params['lam']*base_oof + params['shift']

# Chronic Shift
resid = y - final_oof
chronic = train_feat["primary_chronic"].astype(str).values
shifts = {}
best_oof_mae = mean_absolute_error(y, final_oof)
final_test_shifted = final_test.copy()

print(f"\n[Post-Process] Base OOF MAE: {best_oof_mae:.4f}")
shrink = 0.25
for g in np.unique(chronic):
    med = np.median(resid[chronic==g])
    shifts[g] = med * shrink

oof_shifted = final_oof.copy()
for g, s in shifts.items():
    oof_shifted[chronic==g] += s
    
new_mae = mean_absolute_error(y, oof_shifted)
if new_mae < best_oof_mae - 0.05:
    print(f"  Applying chronic shift. New OOF MAE: {new_mae:.4f}")
    test_chronic = test_feat["primary_chronic"].astype(str).values
    for g, s in shifts.items():
        final_test_shifted[test_chronic==g] += s
else:
    print("  Keeping base.")
    final_test_shifted = final_test

final_test_shifted = np.clip(final_test_shifted, 0, np.max(y)*1.5)

sub = pd.DataFrame({
    "patient_id": test["patient_id"].values,
    "ed_cost_next3y_usd": np.round(final_test_shifted, 2)
})
sub.to_csv(OUT_SUB_PATH, index=False)
print(f"\nSaved to {OUT_SUB_PATH}")

CODE 20 | The Log-Hybrid Strategy
Fix: Replaced failed Huber model with Log-Target RMSE
Goal: Combine Raw-Scale robustness (Models A/B) with Log-Scale structure (Model D)

[Load & Feats]
  Features: 33

[Training] 4 Models (Log-Target Fixed) x 5 Seeds...
  Seed 1 MAE: A_RMSE_d5=433.96 | B_RMSE_d4=434.01 | C_MAE_d4=439.67 | D_LOG_d4=447.68
  Seed 2 MAE: A_RMSE_d5=435.15 | B_RMSE_d4=434.91 | C_MAE_d4=445.52 | D_LOG_d4=451.18
  Seed 3 MAE: A_RMSE_d5=436.08 | B_RMSE_d4=436.47 | C_MAE_d4=443.60 | D_LOG_d4=450.12
  Seed 4 MAE: A_RMSE_d5=435.20 | B_RMSE_d4=436.89 | C_MAE_d4=442.08 | D_LOG_d4=453.10
  Seed 5 MAE: A_RMSE_d5=436.01 | B_RMSE_d4=435.73 | C_MAE_d4=442.73 | D_LOG_d4=452.67

[Ensemble] Searching weights...
  Best MAE: 431.9494
  Weights: {'A_RMSE_d5': 0.4, 'B_RMSE_d4': 0.6, 'C_MAE_d4': 0.0, 'D_LOG_d4': 0.0}
  Params: {'lam': 0.0, 'shift': np.float64(-0.2411779593975325)}

[Post-Process] Base OOF MAE: 431.9494
  Keeping base.

Saved to D:\AgentDs\agent_ds_healthcare\submission.csv


# Iteration 9
- 453.5903

In [26]:
# === CODE 19 / "Code18+++": per-code mix features + uncertainty shrink + EB group shift ===
# Goal: push LB below 449 by adding low-dim orthogonal receipt signal + variance control.
# Output: D:\AgentDs\agent_ds_healthcare\submission.csv

import os, re, sys, gc, math, warnings, random, zlib
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 220)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Paths
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"

ID_COL = "patient_id"
TARGET = "ed_cost_next3y_usd"

# Known receipt code universe observed in your logs (keep low-dim & stable)
KNOWN_CODES = [
    "31500","36556","36620","70450","71045","74177","84484","85025","87070","92950",
    "99281","99282","99283","99284","99285","99291","99292","G0378"
]
OTHER_CODE = "OTHER"

print("="*95)
print("CODE 19 | Code18+++ : +per-code mix (share/cnt) + uncertainty shrink + EB group shift")
print("="*95)

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor


# -----------------------------
# Config (keep stable / regularized)
# -----------------------------
class CFG:
    N_FOLDS = 7
    N_SEEDS = 5

    ITERS = 3200
    ES_ROUNDS = 130
    LR = 0.03

    RSM = 0.80
    SUBSAMPLE = 0.80

    TRIM_K = 1  # with 5 seeds -> drop min/max

    # ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
    SHIFT_GRID = [0.0, 0.5, 1.0]
    STD_PEN = 0.20
    LAM_PEN = 4.0
    SHIFT_PEN = 0.001

    USE_FULLFIT_BLEND = True
    FULLFIT_BLEND_W = 0.35

    # uncertainty shrink search (dynamic baseline pull for high-uncertainty samples)
    UNC_ALPHA_MAX_GRID = [0.15, 0.25, 0.35, 0.45]
    UNC_QLOW_GRID = [0.50, 0.60, 0.70]
    UNC_QHIGH_GRID = [0.90, 0.95, 0.975]
    UNC_ALPHA_PEN = 0.03  # small simplicity penalty

    # EB group shift
    EB_K_GRID = [20, 40, 60, 80, 120]


# -----------------------------
# Utilities
# -----------------------------
def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def safe_num_series(s: pd.Series, default=0.0) -> pd.Series:
    v = pd.to_numeric(s, errors="coerce")
    v = v.replace([np.inf, -np.inf], np.nan).fillna(default)
    return v

def stable_hash(s: str) -> int:
    return int(zlib.crc32(s.encode("utf-8")) & 0xffffffff)

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return 0.0
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    cumx = np.cumsum(x)
    return float((n + 1 - 2*np.sum(cumx)/cumx[-1]) / n)

def trimmed_mean(mat: np.ndarray, trim_k: int = 1) -> np.ndarray:
    mat = np.asarray(mat, dtype=float)
    if mat.ndim != 2:
        raise ValueError("trimmed_mean expects 2D array")
    n = mat.shape[0]
    if trim_k <= 0 or n < 2*trim_k + 1:
        return np.mean(mat, axis=0)
    s = np.sort(mat, axis=0)
    return np.mean(s[trim_k:n-trim_k, :], axis=0)


# -----------------------------
# Admissions features
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs:
        return None
    adm = pd.concat(dfs, ignore_index=True)

    need = ["patient_id","charlson_band","acuity_emergent"]
    if not all(c in adm.columns for c in need):
        return None

    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")

    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()

    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = safe_num_series(out[c], default=0.0)

    out["patient_id"] = pd.to_numeric(out["patient_id"], errors="coerce").astype("Int64")
    out = out.dropna(subset=["patient_id"]).copy()
    out["patient_id"] = out["patient_id"].astype(int)
    return out


# -----------------------------
# Receipts features: low-dim + per-code share/cnt (A1)
# -----------------------------
def build_receipt_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()

    # locate columns
    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code"]:
        if c in li.columns:
            code_col = c
            break

    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price","sum_line_total"]:
        if c in li.columns:
            total_col = c
            break

    unit_col = None
    for c in ["unit_price","unitprice","unit_cost","unitcost"]:
        if c in li.columns:
            unit_col = c
            break

    qty_col = None
    for c in ["qty","quantity","units"]:
        if c in li.columns:
            qty_col = c
            break

    if "patient_id" not in li.columns or code_col is None or total_col is None:
        raise ValueError("receipts lineitems missing required columns (need patient_id, code, line_total-like).")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()

    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    li.loc[li["amt"] < 0, "amt"] = 0.0

    if unit_col is not None:
        li["unit_price"] = pd.to_numeric(li[unit_col], errors="coerce").replace([np.inf,-np.inf], np.nan)
    else:
        li["unit_price"] = np.nan

    if qty_col is not None:
        li["qty"] = pd.to_numeric(li[qty_col], errors="coerce").replace([np.inf,-np.inf], np.nan)
    else:
        li["qty"] = np.nan

    # totals
    receipt_total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(receipt_total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)

    share = (li["amt"] / denom).fillna(0.0)

    # concentration add-ons
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    top1_share = share.groupby(li["patient_id"]).max().rename("top1_share")

    def _topk_sum(s, k=3):
        a = np.sort(s.values.astype(float))[::-1]
        return float(a[:k].sum()) if a.size else 0.0

    top3_share = share.groupby(li["patient_id"]).apply(lambda s: _topk_sum(s, 3)).rename("top3_share")
    gini_amt = li.groupby("patient_id")["amt"].apply(lambda s: gini(s.values)).rename("gini_line_total")
    max_line_total = li.groupby("patient_id")["amt"].max().rename("max_line_total")

    # code numeric where possible
    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")

    # buckets
    em_codes = ["99281","99282","99283","99284","99285"]
    is_em = li["code"].isin(em_codes)
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)

    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])

    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")

    # EM stats
    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    sum_em_level = (em_level * is_em.astype(int)).groupby(li["patient_id"]).sum().rename("sum_em_level")
    avg_em_level = (sum_em_level / n_em_codes.replace(0, np.nan)).fillna(0.0).rename("avg_em_level")
    n_high_em = ((em_level >= 4) & is_em).astype(int).groupby(li["patient_id"]).sum().rename("n_high_em")

    # totals by bucket
    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    img_total = li.loc[is_imaging].groupby("patient_id")["amt"].sum().rename("img_total")
    lab_total = li.loc[is_lab].groupby("patient_id")["amt"].sum().rename("lab_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    # counts by bucket
    n_procedures = is_proc_any.astype(int).groupby(li["patient_id"]).sum().rename("n_procedures")
    n_imaging = is_imaging.astype(int).groupby(li["patient_id"]).sum().rename("n_imaging")
    n_lab = is_lab.astype(int).groupby(li["patient_id"]).sum().rename("n_lab")

    # flags
    def has_code(code: str, name: str):
        return (li["code"].eq(code).astype(int).groupby(li["patient_id"]).max()).rename(name)

    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_observation = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")

    has_intub_31500 = has_code("31500","has_intub_31500")
    has_cvc_36556 = has_code("36556","has_cvc_36556")
    has_cpr_92950 = has_code("92950","has_cpr_92950")
    has_artline_36620 = has_code("36620","has_artline_36620")
    has_ct_head_70450 = has_code("70450","has_ct_head_70450")
    has_99285 = has_code("99285","has_99285")
    has_ct_abdpel_74177 = has_code("74177","has_ct_abdpel_74177")
    has_troponin_84484 = has_code("84484","has_troponin_84484")
    has_obs_G0378 = has_code("G0378","has_obs_G0378")

    # price level add-ons: median unit_price overall + by buckets
    def _median_unit(mask):
        if unit_col is None:
            return None
        sub = li.loc[mask & li["unit_price"].notna(), ["patient_id","unit_price"]]
        if sub.empty:
            return None
        return sub.groupby("patient_id")["unit_price"].median()

    med_unit_all = li.loc[li["unit_price"].notna()].groupby("patient_id")["unit_price"].median().rename("median_unit_price")
    med_unit_em = _median_unit(is_em)
    if med_unit_em is not None:
        med_unit_em = med_unit_em.rename("median_unit_price_em")
    med_unit_img = _median_unit(is_imaging)
    if med_unit_img is not None:
        med_unit_img = med_unit_img.rename("median_unit_price_imaging")
    med_unit_lab = _median_unit(is_lab)
    if med_unit_lab is not None:
        med_unit_lab = med_unit_lab.rename("median_unit_price_lab")

    # ---- NEW: per-code mix features (share + count) over known codes + OTHER ----
    li["code_known"] = li["code"].where(li["code"].isin(KNOWN_CODES), OTHER_CODE)

    amt_by_code = (
        li.groupby(["patient_id","code_known"])["amt"].sum().unstack(fill_value=0.0)
    )
    cnt_by_code = (
        li.groupby(["patient_id","code_known"])["amt"].size().unstack(fill_value=0)
    )

    # ensure all columns exist
    all_codes = KNOWN_CODES + [OTHER_CODE]
    for c in all_codes:
        if c not in amt_by_code.columns:
            amt_by_code[c] = 0.0
        if c not in cnt_by_code.columns:
            cnt_by_code[c] = 0

    amt_by_code = amt_by_code[all_codes].astype(float)
    cnt_by_code = cnt_by_code[all_codes].astype(float)  # keep numeric; cast later

    rec_total2 = amt_by_code.sum(axis=1).replace(0.0, np.nan)
    share_by_code = amt_by_code.div(rec_total2, axis=0).fillna(0.0).astype(float)

    share_by_code.columns = [f"share_code_{c}" for c in share_by_code.columns]
    cnt_by_code.columns = [f"cnt_code_{c}" for c in cnt_by_code.columns]

    # assemble (all index=patient_id)
    frames = [
        n_unique_codes,
        receipt_total,
        cost_hhi, top1_share, top3_share, gini_amt, max_line_total,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_procedures, n_imaging, n_lab,
        share_by_code, cnt_by_code
    ]
    if med_unit_all is not None:
        frames.append(med_unit_all)
    if med_unit_em is not None:
        frames.append(med_unit_em)
    if med_unit_img is not None:
        frames.append(med_unit_img)
    if med_unit_lab is not None:
        frames.append(med_unit_lab)

    out = pd.concat(frames, axis=1).reset_index()

    # merge bucket totals to compute pct ratios, then drop totals again
    for s in [em_total, crit_total, proc_total, img_total, lab_total, high_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")
    for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    denom2 = out["receipt_total"].replace(0.0, np.nan)
    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_imaging"] = (out["img_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_lab"] = (out["lab_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)

    out["cost_per_em"] = np.where(out["n_em_codes"] > 0, out["receipt_total"] / out["n_em_codes"].clip(lower=1), out["receipt_total"])

    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int)
        + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int)
        + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    # log transforms (few)
    for c in ["median_unit_price","median_unit_price_em","median_unit_price_imaging","median_unit_price_lab","max_line_total"]:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)
        out["log1p_" + c] = np.log1p(out[c].clip(lower=0.0))

    # drop raw totals (keep only ratios + mix + counts)
    out.drop(columns=[c for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"] if c in out.columns], inplace=True)

    # numeric fill
    for c in out.columns:
        if c == "patient_id":
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)

    return out

def load_receipts_joblib(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    data = joblib_load(joblib_path)

    if isinstance(data, dict):
        for k in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if k in data and isinstance(data[k], pd.DataFrame):
                return build_receipt_features_from_lineitems(data[k])
        try:
            df = pd.DataFrame.from_dict(data, orient="index")
            df.index.name = "patient_id"
            df = df.reset_index()
            return df
        except Exception:
            return None

    if isinstance(data, pd.DataFrame):
        df = data
        if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
            return build_receipt_features_from_lineitems(df)
        return df

    if isinstance(data, (list, tuple)):
        dfs = [x for x in data if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
                return build_receipt_features_from_lineitems(df)
        for df in dfs:
            if "patient_id" in df.columns:
                return df

    return None


# -----------------------------
# Feature engineering
# -----------------------------
def build_features(ed_df: pd.DataFrame,
                   patients_df: pd.DataFrame,
                   adm_df: Optional[pd.DataFrame],
                   rcpt_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    feat = ed_df.copy()

    chronic_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    feat["prior_ed_visits_5y"] = safe_num_series(feat["prior_ed_visits_5y"], default=0.0).clip(lower=0.0)
    feat["prior_ed_cost_5y_usd"] = safe_num_series(feat["prior_ed_cost_5y_usd"], default=0.0).clip(lower=0.0)

    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost_cap20k"] = np.log1p(feat["prior_cost_cap20k"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)

    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype("Int64")
    p = p.dropna(subset=["patient_id"]).copy()
    p["patient_id"] = p["patient_id"].astype(int)

    p["age"] = pd.to_numeric(p["age"], errors="coerce")
    if p["age"].isna().any():
        p["age"] = p["age"].fillna(p["age"].median())
    p["age"] = p["age"].clip(lower=0, upper=110)

    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)

    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)

    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    feat = feat.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in feat.columns:
                feat[c] = safe_num_series(feat[c], default=0.0)

        # small bin for EB shift grouping (optional)
        if "charlson_max" in feat.columns:
            feat["charlson_bin3"] = np.clip(np.floor(feat["charlson_max"].fillna(0.0)), 0, 3).astype(int)
    else:
        feat["charlson_bin3"] = 0

    if rcpt_df is not None:
        feat = feat.merge(rcpt_df, on="patient_id", how="left")
        for c in rcpt_df.columns:
            if c == "patient_id":
                continue
            feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan)
            if feat[c].isna().any():
                med = feat[c].median() if not feat[c].isna().all() else 0.0
                feat[c] = feat[c].fillna(med)

    if "pct_cost_critical" in feat.columns:
        feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * feat["pct_cost_critical"]
    if "n_high_acuity_total" in feat.columns:
        feat["logprior_x_highacu"] = feat["log_prior_cost"] * feat["n_high_acuity_total"]

    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)

    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in ["patient_id", "primary_chronic", TARGET]:
            continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median() if not feat[c].isna().all() else 0.0)

    return feat

def get_numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3"}
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def drop_constants(cols: List[str], df: pd.DataFrame) -> List[str]:
    out = []
    for c in cols:
        if c not in df.columns:
            continue
        if df[c].nunique(dropna=False) <= 1:
            continue
        out.append(c)
    return out


# -----------------------------
# Training
# -----------------------------
def train_models(train_feat: pd.DataFrame, test_feat: pd.DataFrame,
                 feat_full: List[str], feat_pruned: List[str]) -> Tuple[Dict[str, List[np.ndarray]], Dict[str, List[np.ndarray]], Dict[str, List[int]]]:
    y = train_feat[TARGET].values.astype(float)

    tmp = train_feat[["primary_chronic", TARGET]].copy()
    tmp["cost_bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop")
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"].values)

    # fallback if too granular for N_FOLDS
    vc = pd.Series(strat).value_counts()
    if int(vc.min()) < CFG.N_FOLDS:
        print(f"[warn] strat too granular (min class count {int(vc.min())}). Falling back to stratify by primary_chronic.")
        strat = LabelEncoder().fit_transform(train_feat["primary_chronic"].astype(str).values)

    model_specs = {
        "A_RMSE_full_d5": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=5, l2_leaf_reg=8, min_data_in_leaf=40,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
            random_strength=1.0,
        ),
        "B_RMSE_pruned_d4": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=4, l2_leaf_reg=6, min_data_in_leaf=50,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
            random_strength=2.0,
        ),
        "C_MAE_pruned_d4": dict(
            loss_function="MAE", eval_metric="MAE",
            depth=4, l2_leaf_reg=12, min_data_in_leaf=55,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
            random_strength=1.5,
        ),
    }
    model_featcols = {
        "A_RMSE_full_d5": feat_full,
        "B_RMSE_pruned_d4": feat_pruned,
        "C_MAE_pruned_d4": feat_pruned,
    }

    oof_by_seed = {m: [] for m in model_specs.keys()}
    test_by_seed = {m: [] for m in model_specs.keys()}
    best_iters = {m: [] for m in model_specs.keys()}

    print("\n[training] CatBoost CPU | shallow trees | rsm=0.8 | subsample=0.8 | multi-seed bagging")
    print("Models:", list(model_specs.keys()))
    print(f"Seeds={CFG.N_SEEDS}, Folds={CFG.N_FOLDS}\n")

    for seed_idx in range(CFG.N_SEEDS):
        rs = SEED + seed_idx * 13
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs.keys()}
        test_seed_foldbag = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs.keys()}
        best_iters_seed = {m: [] for m in model_specs.keys()}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat), 1):
            for mname, params in model_specs.items():
                cols = model_featcols[mname]
                X_tr = train_feat[cols].iloc[ti]
                y_tr = y[ti]
                X_va = train_feat[cols].iloc[vi]
                y_va = y[vi]
                X_te = test_feat[cols]

                cb = CatBoostRegressor(
                    **params,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=int(rs + fold * 31 + stable_hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )
                cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)

                try:
                    bi = int(cb.get_best_iteration())
                except Exception:
                    bi = None
                if bi is not None and bi > 0:
                    best_iters[mname].append(bi)
                    best_iters_seed[mname].append(bi)

                pred_va = cb.predict(X_va)
                pred_te = cb.predict(X_te)

                oof_seed[mname][vi] = pred_va
                test_seed_foldbag[mname] += pred_te / CFG.N_FOLDS

                del cb
                gc.collect()

        # optional: full-fit per seed blend
        test_seed_final = {}
        if CFG.USE_FULLFIT_BLEND:
            for mname, params in model_specs.items():
                cols = model_featcols[mname]
                X_all = train_feat[cols]
                X_te = test_feat[cols]

                if best_iters_seed[mname]:
                    it_med = int(np.median(best_iters_seed[mname]))
                else:
                    it_med = int(CFG.ITERS * 0.45)
                it_use = int(max(300, min(CFG.ITERS, it_med + 150)))

                params_full = dict(params)
                params_full["iterations"] = it_use
                cb_full = CatBoostRegressor(
                    **params_full,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=int(rs + 999 + stable_hash("FULL_"+mname) % 997),
                )
                cb_full.fit(X_all, y, verbose=0)
                pred_te_full = cb_full.predict(X_te)
                del cb_full
                gc.collect()

                test_seed_final[mname] = (1.0 - CFG.FULLFIT_BLEND_W) * test_seed_foldbag[mname] + CFG.FULLFIT_BLEND_W * pred_te_full
        else:
            test_seed_final = test_seed_foldbag

        seed_maes = {m: float(mean_absolute_error(y, oof_seed[m])) for m in model_specs.keys()}
        print(f"  Seed {seed_idx+1}/{CFG.N_SEEDS} OOF MAE: " + " | ".join([f"{m}={seed_maes[m]:.2f}" for m in model_specs.keys()]))

        for m in model_specs.keys():
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed_final[m])

    print("\n[seed-aggregated OOF MAE per model] (trimmed mean across seeds)")
    for m in oof_by_seed.keys():
        mat = np.vstack(oof_by_seed[m])
        avg_oof = trimmed_mean(mat, trim_k=CFG.TRIM_K)
        print(f"  {m:18s}: {mean_absolute_error(y, avg_oof):.2f}")

    print("\n[median best_iteration per model] (reference)")
    for m in best_iters.keys():
        if best_iters[m]:
            print(f"  {m:18s}: {int(np.median(best_iters[m]))}")
        else:
            print(f"  {m:18s}: (n/a)")

    return oof_by_seed, test_by_seed, best_iters


# -----------------------------
# Ensemble selection (stability across seeds)
# -----------------------------
def stable_ensemble_search(train_feat: pd.DataFrame,
                           oof_by_seed: Dict[str, List[np.ndarray]],
                           baseline_vec: np.ndarray) -> Tuple[Dict, List[Tuple]]:
    y = train_feat[TARGET].values.astype(float)
    model_names = list(oof_by_seed.keys())
    assert len(model_names) == 3, "This search expects exactly 3 models."

    oof_agg = {m: trimmed_mean(np.vstack(oof_by_seed[m]), trim_k=CFG.TRIM_K) for m in model_names}

    step = CFG.W_STEP
    grid = np.round(np.arange(0.0, 1.0 + 1e-9, step), 10)

    best = None
    top_list = []

    def eval_combo(wA, wB, wC, lam, shift_mult):
        maes = []
        pred_avg = wA*oof_agg[model_names[0]] + wB*oof_agg[model_names[1]] + wC*oof_agg[model_names[2]]
        pred_avg = (1.0-lam)*pred_avg + lam*baseline_vec
        shift = float(np.median(y - pred_avg)) * shift_mult

        for s in range(CFG.N_SEEDS):
            pred = wA*oof_by_seed[model_names[0]][s] + wB*oof_by_seed[model_names[1]][s] + wC*oof_by_seed[model_names[2]][s]
            pred = (1.0-lam)*pred + lam*baseline_vec
            pred = pred + shift
            maes.append(float(mean_absolute_error(y, pred)))

        mean_m = float(np.mean(maes))
        std_m = float(np.std(maes, ddof=0))
        obj = mean_m + CFG.STD_PEN*std_m + CFG.LAM_PEN*lam + CFG.SHIFT_PEN*abs(shift_mult)
        return obj, mean_m, std_m, shift

    for wA in grid:
        for wB in grid:
            wC = 1.0 - wA - wB
            if wC < -1e-9:
                continue
            wC = float(max(0.0, wC))
            if abs((wA+wB+wC) - 1.0) > 1e-6:
                continue
            for lam in CFG.LAM_GRID:
                for sm in CFG.SHIFT_GRID:
                    obj, mean_m, std_m, shift = eval_combo(wA, wB, wC, lam, sm)
                    rec = (obj, mean_m, std_m, wA, wB, wC, lam, sm, shift)
                    top_list.append(rec)
                    if best is None or obj < best[0]:
                        best = rec

    top_list.sort(key=lambda x: x[0])
    print("\n[ensemble search] Top candidates (obj = mean + std_pen + simplicity_pen):")
    for i, rec in enumerate(top_list[:10], 1):
        obj, mean_m, std_m, wA, wB, wC, lam, sm, shift = rec
        print(f"  #{i:02d} obj={obj:.3f} meanMAE={mean_m:.3f} std={std_m:.3f} | w=({wA:.2f},{wB:.2f},{wC:.2f}) | lam={lam:.2f} | shift_mult={sm:.1f} | shift={shift:.2f}")

    obj, mean_m, std_m, wA, wB, wC, lam, sm, shift = best
    meta = {
        "models_order": model_names,
        "weights": (float(wA), float(wB), float(wC)),
        "lam_baseline": float(lam),
        "shift_mult": float(sm),
        "shift_value": float(shift),
        "oof_mean_mae_across_seeds": float(mean_m),
        "oof_std_mae_across_seeds": float(std_m),
    }
    return meta, top_list


# -----------------------------
# Build seed-level ensemble matrices (needed for uncertainty shrink)
# -----------------------------
def build_seed_ensemble_mats(oof_by_seed, test_by_seed, meta, baseline_oof, baseline_test):
    mA, mB, mC = meta["models_order"]
    wA, wB, wC = meta["weights"]
    lam = meta["lam_baseline"]
    shift = meta["shift_value"]

    oof_mat = []
    test_mat = []
    for s in range(CFG.N_SEEDS):
        o_s = wA*oof_by_seed[mA][s] + wB*oof_by_seed[mB][s] + wC*oof_by_seed[mC][s]
        o_s = (1.0-lam)*o_s + lam*baseline_oof
        o_s = o_s + shift
        oof_mat.append(o_s)

        t_s = wA*test_by_seed[mA][s] + wB*test_by_seed[mB][s] + wC*test_by_seed[mC][s]
        t_s = (1.0-lam)*t_s + lam*baseline_test
        t_s = t_s + shift
        test_mat.append(t_s)

    return np.vstack(oof_mat), np.vstack(test_mat)


# -----------------------------
# Uncertainty shrink (B1)
# -----------------------------
def apply_uncertainty_shrink(pred_center, pred_std, baseline, alpha_max, ql_val, qh_val):
    denom = float(max(qh_val - ql_val, 1e-9))
    alpha = np.clip((pred_std - ql_val) / denom, 0.0, 1.0) * float(alpha_max)
    out = (1.0 - alpha) * pred_center + alpha * baseline
    return out, alpha

def search_uncertainty_shrink(y, pred_center, pred_std, baseline):
    best = None
    cands = []
    for alpha_max in CFG.UNC_ALPHA_MAX_GRID:
        for qlow in CFG.UNC_QLOW_GRID:
            for qhigh in CFG.UNC_QHIGH_GRID:
                if qhigh <= qlow + 1e-9:
                    continue
                ql_val = float(np.quantile(pred_std, qlow))
                qh_val = float(np.quantile(pred_std, qhigh))
                pred2, alpha = apply_uncertainty_shrink(pred_center, pred_std, baseline, alpha_max, ql_val, qh_val)
                mae = float(mean_absolute_error(y, pred2))
                obj = mae + CFG.UNC_ALPHA_PEN * float(alpha_max)
                rec = (obj, mae, float(alpha_max), float(qlow), float(qhigh), ql_val, qh_val, float(np.mean(alpha)))
                cands.append(rec)
                if best is None or obj < best[0]:
                    best = rec

    cands.sort(key=lambda x: x[0])
    print("\n[uncertainty shrink] Top candidates (obj = MAE + alpha_pen):")
    for i, rec in enumerate(cands[:10], 1):
        obj, mae, alpha_max, qlow, qhigh, ql_val, qh_val, amean = rec
        print(f"  #{i:02d} obj={obj:.4f} mae={mae:.4f} | alpha_max={alpha_max:.2f} | qlow={qlow:.2f} qhigh={qhigh:.3f} | ql={ql_val:.4f} qh={qh_val:.4f} | mean_alpha={amean:.4f}")

    obj, mae, alpha_max, qlow, qhigh, ql_val, qh_val, amean = best
    meta = dict(alpha_max=alpha_max, qlow=qlow, qhigh=qhigh, ql_val=ql_val, qh_val=qh_val, mean_alpha=amean, oof_mae=mae)
    return meta


# -----------------------------
# EB group shift (A2)
# -----------------------------
def make_key(df: pd.DataFrame, cols: List[str]) -> pd.Series:
    parts = []
    for c in cols:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            s = pd.to_numeric(s, errors="coerce").fillna(-999).round().astype(int).astype(str)
        else:
            s = s.astype(str)
        parts.append(s)
    key = parts[0]
    for p in parts[1:]:
        key = key.str.cat(p, sep="|")
    return key

def apply_eb_group_shift(train_feat, y, pred, group_cols, k):
    key_tr = make_key(train_feat, group_cols)
    resid = y - pred
    tmp = pd.DataFrame({"key": key_tr.values, "resid": resid})
    med = tmp.groupby("key")["resid"].median()
    n = tmp.groupby("key")["resid"].size()
    shrink = (n / (n + float(k))).astype(float)
    shift = (med * shrink).to_dict()
    pred2 = pred + key_tr.map(shift).fillna(0.0).values.astype(float)
    return pred2, shift

def apply_shift_map(test_feat, pred, shift_map, group_cols):
    key_te = make_key(test_feat, group_cols)
    return pred + key_te.map(shift_map).fillna(0.0).values.astype(float)

def search_eb_group_shift(train_feat, y, pred_oof):
    base_mae = float(mean_absolute_error(y, pred_oof))
    best = (base_mae, None, None, None, None)
    print("\n[EB group shift] searching... (only accept if improves)")
    group_sets = [
        ["primary_chronic"],
        ["primary_chronic","insurance_encoded"],
        ["primary_chronic","insurance_encoded","charlson_bin3"],
    ]
    for cols in group_sets:
        if not all(c in train_feat.columns for c in cols):
            continue
        for k in CFG.EB_K_GRID:
            pred2, shift_map = apply_eb_group_shift(train_feat, y, pred_oof, cols, k)
            mae = float(mean_absolute_error(y, pred2))
            print(f"  cols={cols} k={k:3d} -> MAE={mae:.4f} (delta {mae-base_mae:+.4f})")
            if mae < best[0]:
                best = (mae, cols, k, shift_map, pred2)

    mae, cols, k, shift_map, pred2 = best
    if cols is None or mae > base_mae - 0.05:  # require meaningful improvement to reduce overfit risk
        print("  -> EB shift skipped (no robust improvement).")
        return dict(use=False, base_mae=base_mae)
    print(f"  -> EB shift SELECTED: cols={cols} k={k} | MAE {base_mae:.4f} -> {mae:.4f}")
    return dict(use=True, base_mae=base_mae, best_mae=mae, cols=cols, k=k, shift_map=shift_map)


# -----------------------------
# Main
# -----------------------------
must_exist(TRAIN_PATH, "TRAIN")
must_exist(TEST_PATH, "TEST")
must_exist(PATIENTS_PATH, "patients")
must_exist(ADM_TRAIN_PATH, "admissions_train")
must_exist(ADM_TEST_PATH, "admissions_test")
if not os.path.exists(RECEIPTS_JOBLIB_PATH):
    print("[warn] receipts_parsed.joblib missing -> receipts features will be empty (likely worse).")

print("\n[load] reading CSVs...")
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)

print("\n[target stats]")
print(train[TARGET].describe().to_string())

# ids
for df in [train, test, patients]:
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype(int)

# admissions
print("\n[admissions] building robust aggregates...")
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
if adm_df is None:
    print("  admissions features: None")
else:
    print(f"  admissions features: {adm_df.shape} | cols={list(adm_df.columns)}")

# receipts
print("\n[receipts] loading receipts_parsed.joblib and building low-dim receipt features (+per-code mix)...")
rcpt_df = None
if os.path.exists(RECEIPTS_JOBLIB_PATH):
    try:
        rcpt_df = load_receipts_joblib(RECEIPTS_JOBLIB_PATH)
        if rcpt_df is not None:
            rcpt_df["patient_id"] = pd.to_numeric(rcpt_df["patient_id"], errors="coerce").astype("Int64")
            rcpt_df = rcpt_df.dropna(subset=["patient_id"]).copy()
            rcpt_df["patient_id"] = rcpt_df["patient_id"].astype(int)
            rcpt_df = rcpt_df.drop_duplicates("patient_id", keep="last")
            print(f"  receipt_feat shape: {rcpt_df.shape}")
            print(f"  receipt_feat cols ({len(rcpt_df.columns)-1}): {len([c for c in rcpt_df.columns if c!='patient_id'])}")
        else:
            print("  [warn] could not build receipt features from joblib structure.")
    except Exception as e:
        print(f"  [warn] receipts joblib load/build failed: {e}")
        rcpt_df = None
else:
    print("  [warn] receipts joblib missing; skipping receipts features.")

# build features
print("\n[features] building train/test feature frames...")
train_feat = build_features(train, patients, adm_df, rcpt_df)
test_feat  = build_features(test,  patients, adm_df, rcpt_df)

# choose features
feat_full = get_numeric_feature_cols(train_feat)
feat_full = [c for c in feat_full if c != TARGET]
feat_full = drop_constants(feat_full, train_feat)

# PRUNED set: stable low-dim list + per-code mix
pruned_candidates = [
    # priors + transforms
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","prior_cost_cap20k","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k","cost_per_visit","log_visits",
    "baseline_next3y",
    # demographics
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    # admissions
    "charlson_max","charlson_mean","pct_emergent","charlson_bin3",
    # receipt robust
    "cost_per_em","cost_hhi","pct_cost_em","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity","pct_cost_imaging","pct_cost_lab",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","avg_em_level","n_high_em","n_unique_codes",
    "top1_share","top3_share","gini_line_total","max_line_total",
    "median_unit_price","median_unit_price_em","median_unit_price_imaging","median_unit_price_lab",
    "log1p_median_unit_price","log1p_median_unit_price_em","log1p_median_unit_price_imaging","log1p_median_unit_price_lab","log1p_max_line_total",
    # light interactions
    "logprior_x_pctcritical","logprior_x_highacu",
    # stable ratio
    "cost_per_code",
]

# add per-code mix features
for c in KNOWN_CODES + [OTHER_CODE]:
    pruned_candidates.append(f"share_code_{c}")
    pruned_candidates.append(f"cnt_code_{c}")

feat_pruned = [c for c in pruned_candidates if c in train_feat.columns]
feat_pruned = drop_constants(feat_pruned, train_feat)

print(f"  FULL feature count:   {len(feat_full)}")
print(f"  PRUNED feature count: {len(feat_pruned)}")

# safety fill
for c in set(feat_full + feat_pruned):
    med = train_feat[c].median() if c in train_feat.columns and not train_feat[c].isna().all() else 0.0
    train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(med)
    test_feat[c]  = pd.to_numeric(test_feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(med)

# train
oof_by_seed, test_by_seed, best_iters = train_models(train_feat, test_feat, feat_full, feat_pruned)

# baseline vectors for blending / shrink
baseline_oof  = train_feat["baseline_next3y"].values.astype(float)
baseline_test = test_feat["baseline_next3y"].values.astype(float)

# stable ensemble search on OOF
meta, _ = stable_ensemble_search(train_feat, oof_by_seed, baseline_oof)

# build seed-level ensemble mats (for uncertainty shrink)
oof_seed_mat, test_seed_mat = build_seed_ensemble_mats(oof_by_seed, test_by_seed, meta, baseline_oof, baseline_test)

# central predictions (trimmed mean across seeds AFTER ensembling)
oof_center  = trimmed_mean(oof_seed_mat, trim_k=CFG.TRIM_K)
test_center = trimmed_mean(test_seed_mat, trim_k=CFG.TRIM_K)

# seed-based uncertainty
oof_std  = np.std(oof_seed_mat, axis=0, ddof=0)
test_std = np.std(test_seed_mat, axis=0, ddof=0)

y = train_feat[TARGET].values.astype(float)
base_mae = float(mean_absolute_error(y, oof_center))
print("\n" + "="*75)
print("[BASE ENSEMBLE]")
print(f"  OOF MAE (trimmed mean across seed-ensembles): {base_mae:.4f}")
print("  meta:", meta)
print("="*75)

# uncertainty shrink search + apply
unc_meta = search_uncertainty_shrink(y, oof_center, oof_std, baseline_oof)
oof_unc, alpha_oof = apply_uncertainty_shrink(
    oof_center, oof_std, baseline_oof,
    alpha_max=unc_meta["alpha_max"],
    ql_val=unc_meta["ql_val"],
    qh_val=unc_meta["qh_val"],
)
test_unc, alpha_test = apply_uncertainty_shrink(
    test_center, test_std, baseline_test,
    alpha_max=unc_meta["alpha_max"],
    ql_val=unc_meta["ql_val"],
    qh_val=unc_meta["qh_val"],
)
mae_unc = float(mean_absolute_error(y, oof_unc))
print("\n[UNCERTAINTY SHRINK APPLIED]")
print(f"  OOF MAE: {mae_unc:.4f} (delta {mae_unc-base_mae:+.4f})")
print(f"  unc_meta: {unc_meta}")
print(f"  alpha_oof mean/p95/max: {float(np.mean(alpha_oof)):.4f} / {float(np.quantile(alpha_oof,0.95)):.4f} / {float(np.max(alpha_oof)):.4f}")

# EB group shift search + apply (on top of uncertainty shrink)
eb_meta = search_eb_group_shift(train_feat, y, oof_unc)
oof_final = oof_unc.copy()
test_final = test_unc.copy()
extra_shift = {"type": "none"}

if eb_meta.get("use", False):
    cols = eb_meta["cols"]
    shift_map = eb_meta["shift_map"]
    # apply to OOF
    oof_final = apply_shift_map(train_feat, oof_unc, shift_map, cols)
    # apply to test
    test_final = apply_shift_map(test_feat, test_unc, shift_map, cols)
    extra_shift = {"type": "eb_group", "cols": cols, "k": eb_meta["k"]}

final_oof_mae = float(mean_absolute_error(y, oof_final))
print("\n" + "="*75)
print("[FINAL OOF]")
print(f"  OOF MAE (base -> unc_shrink -> EB_shift): {final_oof_mae:.4f}")
print("  extra shift:", extra_shift)
print("  OOF pred quantiles:", qdict(oof_final, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("="*75)

# clip predictions (LB-safe)
y_max = float(np.max(y))
test_final = np.clip(test_final, 0.0, y_max * 1.5)

# feature importance from a full-fit Model A (quick insight)
print("\n[full-train] fitting Model A on full train for feature importance...")
A_params = dict(
    loss_function="RMSE", eval_metric="MAE",
    depth=5, l2_leaf_reg=8, min_data_in_leaf=40,
    learning_rate=CFG.LR, iterations=CFG.ITERS,
    rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
    random_strength=1.0,
    task_type="CPU", thread_count=-1,
    verbose=0, allow_writing_files=False,
    random_seed=SEED,
)
mA_full = CatBoostRegressor(**A_params)
mA_full.fit(train_feat[feat_full], y, verbose=0)
try:
    imp = mA_full.get_feature_importance()
    imp_df = pd.DataFrame({"feature": feat_full, "importance": imp}).sort_values("importance", ascending=False).head(15)
    print("\n[Top 15 feature importances] (Model A full)")
    print(imp_df.to_string(index=False))
except Exception as e:
    print(f"[warn] feature importance failed: {e}")
del mA_full
gc.collect()

# write submission
sub = pd.DataFrame({
    "patient_id": test["patient_id"].values.astype(int),
    "ed_cost_next3y_usd": np.round(test_final.astype(float), 2)
})[["patient_id","ed_cost_next3y_usd"]]

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

print("\n[SUBMISSION sanity checks]")
print("submission shape:", sub.shape)
print("columns:", list(sub.columns))
print("any NaNs:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(sub["ed_cost_next3y_usd"].min()), float(sub["ed_cost_next3y_usd"].median()), float(sub["ed_cost_next3y_usd"].max()))
print("pred quantiles:", qdict(sub["ed_cost_next3y_usd"].values))
print("\nSaved submission to:", str(out_path))
print("\nPaste back: (1) leaderboard MAE, (2) FINAL OOF MAE, (3) meta + unc_meta + extra_shift, (4) pred quantiles.")


CODE 19 | Code18+++ : +per-code mix (share/cnt) + uncertainty shrink + EB group shift

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[admissions] building robust aggregates...
  admissions features: (4000, 4) | cols=['patient_id', 'charlson_max', 'charlson_mean', 'pct_emergent']

[receipts] loading receipts_parsed.joblib and building low-dim receipt features (+per-code mix)...
  receipt_feat shape: (4000, 82)
  receipt_feat cols (81): 81

[features] building train/test feature frames...
  FULL feature count:   101
  PRUNED feature count: 86

[training] CatBoost CPU | shallow trees | rsm=0.8 | subsample=0.8 | multi-seed bagging
Models: ['A_RMSE_full_d5', 'B_RMSE_pruned_d4

# Iteration 10
- 450.9103

In [29]:
# === CODE 19 / "Code18 + CH1-risk bridge" ===
# Goal: push LB below ~449 by adding ONE low-dim, OOF-consistent "readmission risk" signal learned from Challenge 1 labels
#       (admissions_train readmit_30d) + discharge notes -> patient-level risk aggregates -> merge into Challenge 2 regression.
# Keep the Code18 spirit: LOW-DIM + shallow CatBoost + strong regularization + multi-seed + stable ensemble.

import os, re, sys, gc, math, warnings, random, zlib, json
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Paths (must match prompt)
# -----------------------------
DATA_DIR = r"D:\AgentDs\agent_ds_healthcare"
TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv"
TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv"
PATIENTS_PATH = r"D:\AgentDs\agent_ds_healthcare\patients.csv"
ADM_TRAIN_PATH = r"D:\AgentDs\agent_ds_healthcare\admissions_train.csv"
ADM_TEST_PATH  = r"D:\AgentDs\agent_ds_healthcare\admissions_test.csv"
RECEIPTS_JOBLIB_PATH = r"D:\AgentDs\agent_ds_healthcare\receipts_parsed.joblib"
RECEIPTS_PDF_DIR = r"D:\AgentDs\agent_ds_healthcare\receipts_pdf"  # last resort only (we will NOT parse)
DISCHARGE_NOTES_PATH = r"D:\AgentDs\agent_ds_healthcare\discharge_notes.json"

# caches
CH1_RISK_CACHE_PATH  = r"D:\AgentDs\agent_ds_healthcare\cache_ch1_patient_risk.joblib"

OUT_SUB_PATH = r"D:\AgentDs\agent_ds_healthcare\submission.csv"
ID_COL = "patient_id"
TARGET = "ed_cost_next3y_usd"

print("="*110)
print("CODE 19 | Code18 + CH1-risk bridge: LOW-DIM receipts + shallow CatBoost + strong reg + multi-seed + STABLE ensemble")
print("Add-on: train auxiliary CH1 readmit model (OOF-consistent) -> patient risk aggregates -> feed to Ch2 regression.")
print("="*110)

# -----------------------------
# Minimal deps
# -----------------------------
def _pip_install(pkg: str):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    from joblib import load as joblib_load, dump as joblib_dump
except Exception:
    _pip_install("joblib")
    from joblib import load as joblib_load, dump as joblib_dump

try:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error, f1_score
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
except Exception:
    _pip_install("scikit-learn")
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_absolute_error, f1_score
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD

try:
    from catboost import CatBoostRegressor, CatBoostClassifier
except Exception:
    _pip_install("catboost")
    from catboost import CatBoostRegressor, CatBoostClassifier


# -----------------------------
# Config (keep fast; code16-like regularization)
# -----------------------------
class CFG:
    # Ch2 main
    N_FOLDS = 7
    N_SEEDS = 5
    ITERS = 3200
    ES_ROUNDS = 130
    LR = 0.03
    RSM = 0.80
    SUBSAMPLE = 0.80
    TRIM_K = 1  # with 5 seeds -> drop min/max and average middle 3

    # ensemble search
    W_STEP = 0.05
    LAM_GRID = [0.00, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
    SHIFT_GRID = [0.0, 0.5, 1.0]

    # stability objective (LB-oriented)
    STD_PEN = 0.20
    LAM_PEN = 4.0
    SHIFT_PEN = 0.001

    # optional small full-train-per-seed blend (often helps a bit; still cheap)
    USE_FULLFIT_BLEND = True
    FULLFIT_BLEND_W = 0.35  # final test_pred = (1-w)*foldbag + w*fullfit

    # CH1 aux model -> patient risk features
    CH1_N_SPLITS = 5
    CH1_SVD_DIM = 16


# -----------------------------
# Utilities
# -----------------------------
def must_exist(path: str, name: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")

def log_shape(name: str, df: pd.DataFrame):
    mem = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"[{name}] shape={df.shape} | cols={df.shape[1]} | mem={mem:.2f} MB")

def qdict(x, qs=(0,0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1.0)):
    x = np.asarray(x, dtype=float)
    return {q: float(np.quantile(x, q)) for q in qs}

def standardize_zip3(z):
    if z is None or (isinstance(z, float) and np.isnan(z)):
        return None
    s = re.sub(r"\D", "", str(z).strip())
    if not s:
        return None
    return s.zfill(3)

def norm_code(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().upper()
    if s == "" or s.lower() == "nan":
        return None
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    s = re.sub(r"\s+", "", s)
    return s

def safe_num_series(s: pd.Series, default=0.0) -> pd.Series:
    v = pd.to_numeric(s, errors="coerce")
    v = v.replace([np.inf, -np.inf], np.nan).fillna(default)
    return v

def stable_hash(s: str) -> int:
    return int(zlib.crc32(s.encode("utf-8")) & 0xffffffff)

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return 0.0
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    cumx = np.cumsum(x)
    return float((n + 1 - 2*np.sum(cumx)/cumx[-1]) / n)

def trimmed_mean(mat: np.ndarray, trim_k: int = 1) -> np.ndarray:
    """
    mat: (n_seeds, n_samples)
    if n_seeds >= 2*trim_k + 1, drop extremes along axis=0 then mean.
    """
    mat = np.asarray(mat, dtype=float)
    if mat.ndim != 2:
        raise ValueError("trimmed_mean expects 2D array")
    n = mat.shape[0]
    if trim_k <= 0 or n < 2*trim_k + 1:
        return np.mean(mat, axis=0)
    s = np.sort(mat, axis=0)
    return np.mean(s[trim_k:n-trim_k, :], axis=0)


# -----------------------------
# Admissions features (keep simple, robust) for Ch2 main
# -----------------------------
def load_admissions_features(adm_train_path: str, adm_test_path: str) -> Optional[pd.DataFrame]:
    dfs = []
    for path in [adm_train_path, adm_test_path]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "readmit_30d" in df.columns:
                df = df.drop(columns=["readmit_30d"])
            dfs.append(df)
    if not dfs:
        return None
    adm = pd.concat(dfs, ignore_index=True)

    need = ["patient_id","charlson_band","acuity_emergent"]
    if not all(c in adm.columns for c in need):
        return None

    adm["patient_id"] = pd.to_numeric(adm["patient_id"], errors="coerce")
    adm["charlson_band"] = pd.to_numeric(adm["charlson_band"], errors="coerce")
    adm["acuity_emergent"] = pd.to_numeric(adm["acuity_emergent"], errors="coerce")

    out = adm.groupby("patient_id").agg(
        charlson_max=("charlson_band","max"),
        charlson_mean=("charlson_band","mean"),
        pct_emergent=("acuity_emergent","mean"),
    ).reset_index()

    for c in ["charlson_max","charlson_mean","pct_emergent"]:
        out[c] = safe_num_series(out[c], default=0.0)

    out["patient_id"] = pd.to_numeric(out["patient_id"], errors="coerce").astype("Int64")
    out = out.dropna(subset=["patient_id"]).copy()
    out["patient_id"] = out["patient_id"].astype(int)
    return out


# -----------------------------
# CH1 AUX: discharge notes loader + patient-level risk features (OOF-consistent)
# -----------------------------
def _load_discharge_notes(notes_path: str) -> pd.DataFrame:
    if not os.path.exists(notes_path):
        return pd.DataFrame(columns=["admission_id", "note"])
    try:
        with open(notes_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        if "admission_id" not in df.columns:
            return pd.DataFrame(columns=["admission_id", "note"])
        if "note" not in df.columns:
            df["note"] = ""
        df["admission_id"] = pd.to_numeric(df["admission_id"], errors="coerce")
        df = df.dropna(subset=["admission_id"]).copy()
        df["admission_id"] = df["admission_id"].astype(int)
        df["note"] = df["note"].astype(str).fillna("")
        return df[["admission_id", "note"]]
    except Exception:
        return pd.DataFrame(columns=["admission_id", "note"])

def build_ch1_patient_risk_features(adm_train_path: str,
                                   adm_test_path: str,
                                   patients_df: pd.DataFrame,
                                   notes_path: str,
                                   cache_path: str,
                                   n_splits: int = 5,
                                   svd_dim: int = 16,
                                   seed: int = 42) -> pd.DataFrame:
    """
    Returns patient-level features:
      - ch1_readmit_p_mean / max / std
      - ch1_n_admissions
      - ch1_dx_entropy
    Uses OOF probs for admissions_train and full-fit probs for admissions_test.
    """
    if os.path.exists(cache_path):
        try:
            obj = joblib_load(cache_path)
            if isinstance(obj, pd.DataFrame) and "patient_id" in obj.columns:
                print(f"[ch1-risk] loaded cache -> {cache_path}")
                return obj
        except Exception:
            pass

    if (not os.path.exists(adm_train_path)) or (not os.path.exists(adm_test_path)):
        print("[ch1-risk] admissions_train/test missing -> skip ch1 risk.")
        return pd.DataFrame(columns=["patient_id","ch1_readmit_p_mean","ch1_readmit_p_max","ch1_readmit_p_std","ch1_n_admissions","ch1_dx_entropy"])

    adm_tr = pd.read_csv(adm_train_path)
    adm_te = pd.read_csv(adm_test_path)

    if "readmit_30d" not in adm_tr.columns:
        print("[ch1-risk] admissions_train has no readmit_30d -> skip ch1 risk.")
        return pd.DataFrame(columns=["patient_id","ch1_readmit_p_mean","ch1_readmit_p_max","ch1_readmit_p_std","ch1_n_admissions","ch1_dx_entropy"])

    # required cols
    need_cols = ["admission_id","patient_id","primary_dx","los_days","acuity_emergent","charlson_band","ed_visits_6m","discharge_weekday"]
    for c in need_cols:
        if c not in adm_tr.columns or c not in adm_te.columns:
            print(f"[ch1-risk] missing col: {c} -> skip ch1 risk.")
            return pd.DataFrame(columns=["patient_id","ch1_readmit_p_mean","ch1_readmit_p_max","ch1_readmit_p_std","ch1_n_admissions","ch1_dx_entropy"])

    # ids
    for df in [adm_tr, adm_te]:
        df["admission_id"] = pd.to_numeric(df["admission_id"], errors="coerce")
        df["patient_id"]   = pd.to_numeric(df["patient_id"], errors="coerce")
        df.dropna(subset=["admission_id","patient_id"], inplace=True)
        df["admission_id"] = df["admission_id"].astype(int)
        df["patient_id"]   = df["patient_id"].astype(int)

    adm_tr = adm_tr.copy()
    adm_te = adm_te.copy()
    adm_tr["is_train"] = 1
    adm_te["is_train"] = 0
    adm_all = pd.concat([adm_tr, adm_te], ignore_index=True)

    # merge patients (low-dim demographics)
    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce")
    p = p.dropna(subset=["patient_id"]).copy()
    p["patient_id"] = p["patient_id"].astype(int)
    p["age"] = pd.to_numeric(p["age"], errors="coerce")
    p["age"] = p["age"].fillna(p["age"].median() if not p["age"].isna().all() else 0.0).clip(0, 110)
    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)
    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)
    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    adm_all = adm_all.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")

    # merge notes
    notes_df = _load_discharge_notes(notes_path)
    if not notes_df.empty:
        adm_all = adm_all.merge(notes_df, on="admission_id", how="left")
    else:
        adm_all["note"] = ""
    adm_all["note"] = adm_all["note"].astype(str).fillna("")

    # dx encoding
    dx_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    adm_all["primary_dx"] = adm_all["primary_dx"].astype(str)
    adm_all["dx_encoded"] = adm_all["primary_dx"].str.upper().map(dx_map).fillna(-1).astype(float)

    # numeric features
    num_cols = ["los_days","acuity_emergent","charlson_band","ed_visits_6m","discharge_weekday",
                "age","sex_encoded","insurance_encoded","zip_region","dx_encoded"]
    for c in num_cols:
        adm_all[c] = pd.to_numeric(adm_all[c], errors="coerce").replace([np.inf,-np.inf], np.nan)
        adm_all[c] = adm_all[c].fillna(adm_all[c].median() if not adm_all[c].isna().all() else 0.0)

    adm_all["note_len"] = adm_all["note"].str.len().clip(0, 800)
    X_num = adm_all[num_cols + ["note_len"]].values.astype(float)

    # text -> tfidf -> svd (low-dim)
    use_text = True
    X_svd = np.zeros((len(adm_all), 0), dtype=float)
    try:
        tfidf = TfidfVectorizer(
            max_features=12000,
            ngram_range=(1,2),
            min_df=2
        )
        X_t = tfidf.fit_transform(adm_all["note"].values)
        max_svd = min(svd_dim, X_t.shape[1]-1) if X_t.shape[1] > 1 else 0
        if max_svd <= 0:
            use_text = False
        else:
            svd = TruncatedSVD(n_components=max_svd, random_state=seed)
            X_svd = svd.fit_transform(X_t).astype(float)
    except Exception:
        use_text = False
        X_svd = np.zeros((len(adm_all), 0), dtype=float)

    X_all = np.hstack([X_num, X_svd])

    tr_mask = (adm_all["is_train"].values.astype(int) == 1)
    X_tr = X_all[tr_mask]
    y_tr = pd.to_numeric(adm_all.loc[tr_mask, "readmit_30d"], errors="coerce").fillna(0).astype(int).values
    X_te = X_all[~tr_mask]

    # stratify by dx + label
    strat = (adm_all.loc[tr_mask, "primary_dx"].astype(str).str.upper().fillna("UNK") + "_" + y_tr.astype(str)).values
    strat = LabelEncoder().fit_transform(strat)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y_tr), dtype=float)

    base_params = dict(
        loss_function="Logloss",
        eval_metric="Logloss",
        depth=4,
        learning_rate=0.05,
        iterations=2200,
        l2_leaf_reg=10,
        min_data_in_leaf=60,
        rsm=0.85,
        bootstrap_type="Bernoulli",
        subsample=0.85,
        random_strength=1.2,
        task_type="CPU",
        thread_count=-1,
        verbose=0,
        allow_writing_files=False,
    )

    for fold, (ti, vi) in enumerate(skf.split(X_tr, strat), 1):
        cb = CatBoostClassifier(
            **base_params,
            random_seed=int(seed + 1000 + fold*37),
            early_stopping_rounds=120,
        )
        cb.fit(X_tr[ti], y_tr[ti], eval_set=(X_tr[vi], y_tr[vi]), verbose=0)
        oof[vi] = cb.predict_proba(X_tr[vi])[:, 1]
        del cb
        gc.collect()

    try:
        pred_lbl = (oof >= 0.5).astype(int)
        mf1 = float(f1_score(y_tr, pred_lbl, average="macro"))
        print(f"[ch1-risk] aux OOF macro-F1@0.5: {mf1:.4f} | text={use_text} | svd_dim={X_svd.shape[1]}")
    except Exception:
        pass

    # full-fit for admissions_test
    cb_full = CatBoostClassifier(**base_params, random_seed=int(seed + 9999))
    cb_full.fit(X_tr, y_tr, verbose=0)
    p_te = cb_full.predict_proba(X_te)[:, 1] if len(X_te) else np.array([], dtype=float)
    del cb_full
    gc.collect()

    # attach preds back to admissions
    adm_pred_tr = adm_all.loc[tr_mask, ["admission_id","patient_id","primary_dx"]].copy()
    adm_pred_tr["readmit_p"] = oof
    adm_pred_te = adm_all.loc[~tr_mask, ["admission_id","patient_id","primary_dx"]].copy()
    adm_pred_te["readmit_p"] = p_te
    adm_pred = pd.concat([adm_pred_tr, adm_pred_te], ignore_index=True)

    # patient aggregates
    g = adm_pred.groupby("patient_id")["readmit_p"]
    out = pd.DataFrame({
        "patient_id": g.mean().index.astype(int),
        "ch1_readmit_p_mean": g.mean().values.astype(float),
        "ch1_readmit_p_max":  g.max().values.astype(float),
        "ch1_readmit_p_std":  g.std().fillna(0.0).values.astype(float),
        "ch1_n_admissions":   g.size().values.astype(int),
    })

    # dx entropy
    dx_ct = adm_pred.groupby(["patient_id","primary_dx"]).size().unstack(fill_value=0)
    dx_prop = dx_ct.div(dx_ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    ent = -(dx_prop * np.log(dx_prop + 1e-12)).sum(axis=1)
    out = out.merge(ent.rename("ch1_dx_entropy").reset_index(), on="patient_id", how="left")
    out["ch1_dx_entropy"] = pd.to_numeric(out["ch1_dx_entropy"], errors="coerce").fillna(0.0)

    # cache
    try:
        joblib_dump(out, cache_path)
        print(f"[ch1-risk] cached -> {cache_path}")
    except Exception:
        pass

    return out


# -----------------------------
# Low-dim receipts features from parsed lineitems (v3-style, + tiny robust add-ons)
# -----------------------------
def build_receipt_features_from_lineitems(li: pd.DataFrame) -> pd.DataFrame:
    li = li.copy()

    # locate columns
    code_col = None
    for c in ["code","cpt","cpt_code","hcpcs","proc_code"]:
        if c in li.columns:
            code_col = c
            break

    total_col = None
    for c in ["line_total","line_total_usd","total","amount","line_cost","sum_items","item_total","extended_price","sum_line_total"]:
        if c in li.columns:
            total_col = c
            break

    unit_col = None
    for c in ["unit_price","unitprice","unit_cost","unitcost"]:
        if c in li.columns:
            unit_col = c
            break

    qty_col = None
    for c in ["qty","quantity","units"]:
        if c in li.columns:
            qty_col = c
            break

    if "patient_id" not in li.columns or code_col is None or total_col is None:
        raise ValueError("receipts lineitems missing required columns (need patient_id, code, line_total-like).")

    li["patient_id"] = pd.to_numeric(li["patient_id"], errors="coerce").astype("Int64")
    li = li.dropna(subset=["patient_id"]).copy()
    li["patient_id"] = li["patient_id"].astype(int)

    li["code"] = li[code_col].map(norm_code)
    li = li.dropna(subset=["code"]).copy()

    li["amt"] = pd.to_numeric(li[total_col], errors="coerce").fillna(0.0).astype(float)
    li.loc[li["amt"] < 0, "amt"] = 0.0

    if unit_col is not None:
        li["unit_price"] = pd.to_numeric(li[unit_col], errors="coerce").replace([np.inf,-np.inf], np.nan)
    else:
        li["unit_price"] = np.nan

    if qty_col is not None:
        li["qty"] = pd.to_numeric(li[qty_col], errors="coerce").replace([np.inf,-np.inf], np.nan)
    else:
        li["qty"] = np.nan

    receipt_total = li.groupby("patient_id")["amt"].sum().rename("receipt_total")
    li = li.join(receipt_total, on="patient_id")
    denom = li["receipt_total"].replace(0.0, np.nan)
    share = (li["amt"] / denom).fillna(0.0)

    # concentration
    cost_hhi = (share * share).groupby(li["patient_id"]).sum().rename("cost_hhi")
    top1_share = share.groupby(li["patient_id"]).max().rename("top1_share")
    def _topk_sum(s, k=3):
        a = np.sort(s.values.astype(float))[::-1]
        return float(a[:k].sum()) if a.size else 0.0
    top3_share = share.groupby(li["patient_id"]).apply(lambda s: _topk_sum(s, 3)).rename("top3_share")
    gini_amt = li.groupby("patient_id")["amt"].apply(lambda s: gini(s.values)).rename("gini_line_total")
    max_line_total = li.groupby("patient_id")["amt"].max().rename("max_line_total")

    code_num = pd.to_numeric(li["code"].where(li["code"].str.fullmatch(r"\d+"), None), errors="coerce")

    em_codes = ["99281","99282","99283","99284","99285"]
    is_em = li["code"].isin(em_codes)
    em_map = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
    em_level = li["code"].map(em_map).fillna(0).astype(float)

    is_crit = li["code"].isin(["99291","99292"])
    is_obs = li["code"].str.startswith("G037", na=False)
    is_high = li["code"].isin(["31500","36556","32551","36620","92950"])

    is_lab = code_num.between(80000, 89999)
    is_imaging = code_num.between(70000, 79999)
    is_proc_general = code_num.between(10000, 69999)
    is_proc_any = is_high | (is_proc_general & (~is_high) & (~is_em) & (~is_crit))

    n_unique_codes = li.groupby("patient_id")["code"].nunique().rename("n_unique_codes")

    n_em_codes = is_em.astype(int).groupby(li["patient_id"]).sum().rename("n_em_codes")
    max_em_level = em_level.groupby(li["patient_id"]).max().rename("max_em_level")
    sum_em_level = (em_level * is_em.astype(int)).groupby(li["patient_id"]).sum().rename("sum_em_level")
    avg_em_level = (sum_em_level / n_em_codes.replace(0, np.nan)).fillna(0.0).rename("avg_em_level")
    n_high_em = ((em_level >= 4) & is_em).astype(int).groupby(li["patient_id"]).sum().rename("n_high_em")

    em_total = li.loc[is_em].groupby("patient_id")["amt"].sum().rename("em_total")
    crit_total = li.loc[is_crit].groupby("patient_id")["amt"].sum().rename("crit_total")
    proc_total = li.loc[is_proc_any].groupby("patient_id")["amt"].sum().rename("proc_total")
    img_total = li.loc[is_imaging].groupby("patient_id")["amt"].sum().rename("img_total")
    lab_total = li.loc[is_lab].groupby("patient_id")["amt"].sum().rename("lab_total")
    high_total = li.loc[is_high].groupby("patient_id")["amt"].sum().rename("high_total")

    n_procedures = is_proc_any.astype(int).groupby(li["patient_id"]).sum().rename("n_procedures")
    n_imaging = is_imaging.astype(int).groupby(li["patient_id"]).sum().rename("n_imaging")
    n_lab = is_lab.astype(int).groupby(li["patient_id"]).sum().rename("n_lab")

    def has_code(code: str, name: str):
        return (li["code"].eq(code).astype(int).groupby(li["patient_id"]).max()).rename(name)

    has_critical_care = is_crit.astype(int).groupby(li["patient_id"]).max().rename("has_critical_care")
    has_high_acuity = is_high.astype(int).groupby(li["patient_id"]).max().rename("has_high_acuity")
    has_observation = is_obs.astype(int).groupby(li["patient_id"]).max().rename("has_observation")
    has_imaging = is_imaging.astype(int).groupby(li["patient_id"]).max().rename("has_imaging")

    has_intub_31500 = has_code("31500","has_intub_31500")
    has_cvc_36556 = has_code("36556","has_cvc_36556")
    has_cpr_92950 = has_code("92950","has_cpr_92950")
    has_artline_36620 = has_code("36620","has_artline_36620")
    has_ct_head_70450 = has_code("70450","has_ct_head_70450")
    has_99285 = has_code("99285","has_99285")
    has_ct_abdpel_74177 = has_code("74177","has_ct_abdpel_74177")
    has_troponin_84484 = has_code("84484","has_troponin_84484")
    has_obs_G0378 = has_code("G0378","has_obs_G0378")

    # price level add-ons: median unit_price overall + by bucket
    def _median_unit(mask):
        if unit_col is None:
            return None
        sub = li.loc[mask & li["unit_price"].notna(), ["patient_id","unit_price"]]
        if sub.empty:
            return None
        return sub.groupby("patient_id")["unit_price"].median()

    med_unit_all = li.loc[li["unit_price"].notna()].groupby("patient_id")["unit_price"].median().rename("median_unit_price")
    med_unit_em = _median_unit(is_em)
    if med_unit_em is not None:
        med_unit_em = med_unit_em.rename("median_unit_price_em")
    med_unit_img = _median_unit(is_imaging)
    if med_unit_img is not None:
        med_unit_img = med_unit_img.rename("median_unit_price_imaging")
    med_unit_lab = _median_unit(is_lab)
    if med_unit_lab is not None:
        med_unit_lab = med_unit_lab.rename("median_unit_price_lab")

    out = pd.concat([
        n_unique_codes,
        receipt_total,
        cost_hhi, top1_share, top3_share, gini_amt, max_line_total,
        n_em_codes, max_em_level, avg_em_level, n_high_em,
        has_critical_care, has_high_acuity, has_observation, has_imaging,
        has_intub_31500, has_cvc_36556, has_cpr_92950, has_artline_36620,
        has_ct_head_70450, has_99285, has_ct_abdpel_74177, has_troponin_84484, has_obs_G0378,
        n_procedures, n_imaging, n_lab
    ], axis=1).reset_index()

    for s in [em_total, crit_total, proc_total, img_total, lab_total, high_total]:
        out = out.merge(s.reset_index(), on="patient_id", how="left")
    for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    denom2 = out["receipt_total"].replace(0.0, np.nan)
    out["pct_cost_em"] = (out["em_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_procedure"] = (out["proc_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_critical"] = (out["crit_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_imaging"] = (out["img_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_lab"] = (out["lab_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["pct_cost_high_acuity"] = (out["high_total"] / denom2).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["cost_per_em"] = np.where(out["n_em_codes"] > 0, out["receipt_total"] / out["n_em_codes"].clip(lower=1), out["receipt_total"])

    out["n_high_acuity_total"] = (
        out["has_intub_31500"].fillna(0).astype(int)
        + out["has_cvc_36556"].fillna(0).astype(int)
        + out["has_cpr_92950"].fillna(0).astype(int)
        + out["has_artline_36620"].fillna(0).astype(int)
        + out["has_critical_care"].fillna(0).astype(int)
    ).astype(int)

    # attach unit-price medians
    out = out.merge(med_unit_all.reset_index(), on="patient_id", how="left")
    if med_unit_em is not None:
        out = out.merge(med_unit_em.reset_index(), on="patient_id", how="left")
    else:
        out["median_unit_price_em"] = np.nan
    if med_unit_img is not None:
        out = out.merge(med_unit_img.reset_index(), on="patient_id", how="left")
    else:
        out["median_unit_price_imaging"] = np.nan
    if med_unit_lab is not None:
        out = out.merge(med_unit_lab.reset_index(), on="patient_id", how="left")
    else:
        out["median_unit_price_lab"] = np.nan

    for c in ["median_unit_price","median_unit_price_em","median_unit_price_imaging","median_unit_price_lab","max_line_total"]:
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)
        out["log1p_" + c] = np.log1p(out[c].clip(lower=0.0))

    # cleanup helper totals
    out.drop(columns=[c for c in ["em_total","crit_total","proc_total","img_total","lab_total","high_total","receipt_total"] if c in out.columns], inplace=True)

    for c in out.columns:
        if c == "patient_id":
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0.0)

    return out

def load_receipts_joblib(joblib_path: str) -> Optional[pd.DataFrame]:
    if not os.path.exists(joblib_path):
        return None
    data = joblib_load(joblib_path)

    if isinstance(data, dict):
        for k in ["lineitems_df","lineitems","items_df","items","line_items_df","line_items"]:
            if k in data and isinstance(data[k], pd.DataFrame):
                return build_receipt_features_from_lineitems(data[k])
        try:
            df = pd.DataFrame.from_dict(data, orient="index")
            df.index.name = "patient_id"
            df = df.reset_index()
            return df
        except Exception:
            return None

    if isinstance(data, pd.DataFrame):
        df = data
        if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
            return build_receipt_features_from_lineitems(df)
        return df

    if isinstance(data, (list, tuple)):
        dfs = [x for x in data if isinstance(x, pd.DataFrame)]
        for df in dfs:
            if "patient_id" in df.columns and any(c in df.columns for c in ["code","cpt","cpt_code","hcpcs","proc_code"]):
                return build_receipt_features_from_lineitems(df)
        for df in dfs:
            if "patient_id" in df.columns:
                return df

    return None


# -----------------------------
# Feature engineering (numeric-only, v3 style) + CH1-risk merge
# -----------------------------
def build_features(ed_df: pd.DataFrame,
                   patients_df: pd.DataFrame,
                   adm_df: Optional[pd.DataFrame],
                   rcpt_df: Optional[pd.DataFrame],
                   ch1_risk_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    feat = ed_df.copy()

    chronic_map = {"PNEUMONIA":0, "DIABETESCOMP":1, "HF":2}
    feat["primary_chronic"] = feat["primary_chronic"].astype(str)
    feat["chronic_encoded"] = feat["primary_chronic"].str.upper().map(chronic_map).fillna(-1).astype(float)

    feat["prior_ed_visits_5y"] = safe_num_series(feat["prior_ed_visits_5y"], default=0.0).clip(lower=0.0)
    feat["prior_ed_cost_5y_usd"] = safe_num_series(feat["prior_ed_cost_5y_usd"], default=0.0).clip(lower=0.0)

    feat["prior_cost_cap20k"] = feat["prior_ed_cost_5y_usd"].clip(upper=20000.0)
    feat["sqrt_prior_cost"] = np.sqrt(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost"] = np.log1p(feat["prior_ed_cost_5y_usd"].clip(lower=0.0))
    feat["log_prior_cost_cap20k"] = np.log1p(feat["prior_cost_cap20k"].clip(lower=0.0))
    feat["log_visits"] = np.log1p(feat["prior_ed_visits_5y"].clip(lower=0.0))
    feat["cost_per_visit"] = feat["prior_ed_cost_5y_usd"] / feat["prior_ed_visits_5y"].clip(lower=1.0)

    # baseline for LB-safe blending
    feat["baseline_next3y"] = feat["prior_ed_cost_5y_usd"] * (3.0/5.0)

    # patients encodings
    p = patients_df.copy()
    p["patient_id"] = pd.to_numeric(p["patient_id"], errors="coerce").astype("Int64")
    p = p.dropna(subset=["patient_id"]).copy()
    p["patient_id"] = p["patient_id"].astype(int)

    p["age"] = pd.to_numeric(p["age"], errors="coerce")
    if p["age"].isna().any():
        p["age"] = p["age"].fillna(p["age"].median())
    p["age"] = p["age"].clip(lower=0, upper=110)

    p["sex_encoded"] = (p["sex"].astype(str).str.upper() == "M").astype(int)

    ins = p["insurance"].astype(str).str.lower()
    ins_map = {"private":2, "public":1, "self_pay":0, "selfpay":0}
    p["insurance_encoded"] = ins.map(ins_map).fillna(-1).astype(float)

    z3 = p["zip3"].apply(standardize_zip3).astype("string")
    zr = z3.fillna("000").str.replace(r"\D","", regex=True).str.zfill(3).str[0]
    p["zip_region"] = pd.to_numeric(zr, errors="coerce").fillna(-1).astype(float)

    feat = feat.merge(p[["patient_id","age","sex_encoded","insurance_encoded","zip_region"]], on="patient_id", how="left")
    feat["ins_x_chronic"] = feat["insurance_encoded"].fillna(-1) * feat["chronic_encoded"].fillna(-1)

    # admissions aggregates (Ch2 classic)
    if adm_df is not None:
        feat = feat.merge(adm_df, on="patient_id", how="left")
        for c in ["charlson_max","charlson_mean","pct_emergent"]:
            if c in feat.columns:
                feat[c] = safe_num_series(feat[c], default=0.0)

    # receipts features
    if rcpt_df is not None:
        feat = feat.merge(rcpt_df, on="patient_id", how="left")
        for c in rcpt_df.columns:
            if c == "patient_id":
                continue
            feat[c] = safe_num_series(feat[c], default=np.nan)
            if feat[c].isna().any():
                feat[c] = feat[c].fillna(feat[c].median())

    # CH1 risk features (OOF-consistent)
    if ch1_risk_df is not None and (not ch1_risk_df.empty):
        feat = feat.merge(ch1_risk_df, on="patient_id", how="left")
        for c in ["ch1_readmit_p_mean","ch1_readmit_p_max","ch1_readmit_p_std","ch1_n_admissions","ch1_dx_entropy"]:
            if c in feat.columns:
                feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan)
                feat[c] = feat[c].fillna(feat[c].median() if not feat[c].isna().all() else 0.0)

        if "ch1_readmit_p_mean" in feat.columns:
            feat["logprior_x_readmitp"] = feat["log_prior_cost"] * feat["ch1_readmit_p_mean"]
            feat["visits_x_readmitp"]   = feat["log_visits"]     * feat["ch1_readmit_p_mean"]

    # a couple of LIGHT interactions (still low-risk)
    if "pct_cost_critical" in feat.columns:
        feat["logprior_x_pctcritical"] = feat["log_prior_cost"] * feat["pct_cost_critical"]
    if "n_high_acuity_total" in feat.columns:
        feat["logprior_x_highacu"] = feat["log_prior_cost"] * feat["n_high_acuity_total"]

    # derived stable ratios
    if "n_unique_codes" in feat.columns:
        feat["cost_per_code"] = feat["prior_ed_cost_5y_usd"] / feat["n_unique_codes"].clip(lower=1.0)

    # numeric safety
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    for c in feat.columns:
        if c in ["patient_id", "primary_chronic", TARGET]:
            continue
        feat[c] = pd.to_numeric(feat[c], errors="coerce")
        if feat[c].isna().any():
            feat[c] = feat[c].fillna(feat[c].median() if not feat[c].isna().all() else 0.0)

    return feat

def get_numeric_feature_cols(df: pd.DataFrame) -> List[str]:
    exclude = {"patient_id","primary_chronic",TARGET,"sex","insurance","zip3"}
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def drop_constants(cols: List[str], df: pd.DataFrame) -> List[str]:
    out = []
    for c in cols:
        if c not in df.columns:
            continue
        if df[c].nunique(dropna=False) <= 1:
            continue
        out.append(c)
    return out


# -----------------------------
# Training (multi-seed + fold bagging)
# -----------------------------
def train_models(train_feat: pd.DataFrame, test_feat: pd.DataFrame,
                 feat_full: List[str], feat_pruned: List[str]) -> Tuple[Dict[str, List[np.ndarray]], Dict[str, List[np.ndarray]], Dict[str, List[int]]]:
    y = train_feat[TARGET].values.astype(float)

    # stratify: chronic + target bin (v3)
    tmp = train_feat[["primary_chronic", TARGET]].copy()
    tmp["cost_bin"] = pd.qcut(tmp[TARGET], q=5, labels=False, duplicates="drop")
    tmp["strat"] = tmp["primary_chronic"].astype(str) + "_" + tmp["cost_bin"].astype(str)
    strat = LabelEncoder().fit_transform(tmp["strat"].values)

    # 3 models (keep "less is more")
    model_specs = {
        "A_RMSE_full_d5": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=5, l2_leaf_reg=8, min_data_in_leaf=40,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
            random_strength=1.0,
        ),
        "B_RMSE_pruned_d4": dict(
            loss_function="RMSE", eval_metric="MAE",
            depth=4, l2_leaf_reg=6, min_data_in_leaf=50,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
            random_strength=2.0,
        ),
        "C_MAE_pruned_d4": dict(
            loss_function="MAE", eval_metric="MAE",
            depth=4, l2_leaf_reg=12, min_data_in_leaf=55,
            learning_rate=CFG.LR, iterations=CFG.ITERS,
            rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
            random_strength=1.5,
        ),
    }
    model_featcols = {
        "A_RMSE_full_d5": feat_full,
        "B_RMSE_pruned_d4": feat_pruned,
        "C_MAE_pruned_d4": feat_pruned,
    }

    oof_by_seed = {m: [] for m in model_specs.keys()}
    test_by_seed = {m: [] for m in model_specs.keys()}
    best_iters = {m: [] for m in model_specs.keys()}

    print("\n[training] CatBoost CPU | shallow trees | rsm=0.8 | subsample=0.8 | multi-seed bagging")
    print("Models:", list(model_specs.keys()))
    print(f"Seeds={CFG.N_SEEDS}, Folds={CFG.N_FOLDS}\n")

    for seed_idx in range(CFG.N_SEEDS):
        rs = SEED + seed_idx * 13
        kf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=rs)

        oof_seed = {m: np.zeros(len(train_feat), dtype=float) for m in model_specs.keys()}
        test_seed_foldbag = {m: np.zeros(len(test_feat), dtype=float) for m in model_specs.keys()}
        best_iters_seed = {m: [] for m in model_specs.keys()}

        for fold, (ti, vi) in enumerate(kf.split(train_feat, strat), 1):
            for mname, params in model_specs.items():
                cols = model_featcols[mname]
                X_tr = train_feat[cols].iloc[ti]
                y_tr = y[ti]
                X_va = train_feat[cols].iloc[vi]
                y_va = y[vi]
                X_te = test_feat[cols]

                cb = CatBoostRegressor(
                    **params,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=int(rs + fold * 31 + stable_hash(mname) % 997),
                    early_stopping_rounds=CFG.ES_ROUNDS,
                )
                cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)

                try:
                    bi = int(cb.get_best_iteration())
                except Exception:
                    bi = None
                if bi is not None and bi > 0:
                    best_iters[mname].append(bi)
                    best_iters_seed[mname].append(bi)

                pred_va = cb.predict(X_va)
                pred_te = cb.predict(X_te)

                oof_seed[mname][vi] = pred_va
                test_seed_foldbag[mname] += pred_te / CFG.N_FOLDS

                del cb
                gc.collect()

        # optional: full-fit per seed
        test_seed_final = {}
        if CFG.USE_FULLFIT_BLEND:
            for mname, params in model_specs.items():
                cols = model_featcols[mname]
                X_all = train_feat[cols]
                X_te = test_feat[cols]

                if best_iters_seed[mname]:
                    it_med = int(np.median(best_iters_seed[mname]))
                else:
                    it_med = int(CFG.ITERS * 0.45)
                it_use = int(max(300, min(CFG.ITERS, it_med + 150)))

                params_full = dict(params)
                params_full["iterations"] = it_use
                cb_full = CatBoostRegressor(
                    **params_full,
                    task_type="CPU",
                    thread_count=-1,
                    verbose=0,
                    allow_writing_files=False,
                    random_seed=int(rs + 999 + stable_hash("FULL_"+mname) % 997),
                )
                cb_full.fit(X_all, y, verbose=0)
                pred_te_full = cb_full.predict(X_te)
                del cb_full
                gc.collect()

                test_seed_final[mname] = (1.0 - CFG.FULLFIT_BLEND_W) * test_seed_foldbag[mname] + CFG.FULLFIT_BLEND_W * pred_te_full
        else:
            test_seed_final = test_seed_foldbag

        seed_maes = {m: float(mean_absolute_error(y, oof_seed[m])) for m in model_specs.keys()}
        print(f"  Seed {seed_idx+1}/{CFG.N_SEEDS} OOF MAE: " + " | ".join([f"{m}={seed_maes[m]:.2f}" for m in model_specs.keys()]))

        for m in model_specs.keys():
            oof_by_seed[m].append(oof_seed[m])
            test_by_seed[m].append(test_seed_final[m])

    print("\n[seed-aggregated OOF MAE per model] (trimmed mean across seeds)")
    for m in oof_by_seed.keys():
        mat = np.vstack(oof_by_seed[m])
        avg_oof = trimmed_mean(mat, trim_k=CFG.TRIM_K)
        print(f"  {m:18s}: {mean_absolute_error(y, avg_oof):.2f}")

    print("\n[median best_iteration per model] (reference)")
    for m in best_iters.keys():
        if best_iters[m]:
            print(f"  {m:18s}: {int(np.median(best_iters[m]))}")
        else:
            print(f"  {m:18s}: (n/a)")

    return oof_by_seed, test_by_seed, best_iters


# -----------------------------
# Ensemble selection (stability across seeds) - for 3 models
# -----------------------------
def stable_ensemble_search(train_feat: pd.DataFrame,
                           oof_by_seed: Dict[str, List[np.ndarray]],
                           baseline_vec: np.ndarray) -> Tuple[np.ndarray, Dict]:
    y = train_feat[TARGET].values.astype(float)
    model_names = list(oof_by_seed.keys())
    assert len(model_names) == 3, "This search expects exactly 3 models."

    oof_agg = {m: trimmed_mean(np.vstack(oof_by_seed[m]), trim_k=CFG.TRIM_K) for m in model_names}

    step = CFG.W_STEP
    grid = np.round(np.arange(0.0, 1.0 + 1e-9, step), 10)

    best = None
    top_list = []

    def eval_combo(wA, wB, wC, lam, shift_mult):
        maes = []
        pred_avg = wA*oof_agg[model_names[0]] + wB*oof_agg[model_names[1]] + wC*oof_agg[model_names[2]]
        pred_avg = (1.0-lam)*pred_avg + lam*baseline_vec
        shift = float(np.median(y - pred_avg)) * shift_mult

        for s in range(CFG.N_SEEDS):
            pred = wA*oof_by_seed[model_names[0]][s] + wB*oof_by_seed[model_names[1]][s] + wC*oof_by_seed[model_names[2]][s]
            pred = (1.0-lam)*pred + lam*baseline_vec
            pred = pred + shift
            maes.append(float(mean_absolute_error(y, pred)))

        mean_m = float(np.mean(maes))
        std_m = float(np.std(maes, ddof=0))
        obj = mean_m + CFG.STD_PEN*std_m + CFG.LAM_PEN*lam + CFG.SHIFT_PEN*abs(shift_mult)
        return obj, mean_m, std_m, shift

    for wA in grid:
        for wB in grid:
            wC = 1.0 - wA - wB
            if wC < -1e-9:
                continue
            wC = float(max(0.0, wC))
            if abs((wA+wB+wC) - 1.0) > 1e-6:
                continue
            for lam in CFG.LAM_GRID:
                for sm in CFG.SHIFT_GRID:
                    obj, mean_m, std_m, shift = eval_combo(wA, wB, wC, lam, sm)
                    rec = (obj, mean_m, std_m, wA, wB, wC, lam, sm, shift)
                    top_list.append(rec)
                    if best is None or obj < best[0]:
                        best = rec

    top_list.sort(key=lambda x: x[0])
    print("\n[ensemble search] Top candidates (robust objective = mean + std_pen + simplicity_pen):")
    for i, rec in enumerate(top_list[:10], 1):
        obj, mean_m, std_m, wA, wB, wC, lam, sm, shift = rec
        print(f"  #{i:02d} obj={obj:.3f} meanMAE={mean_m:.3f} std={std_m:.3f} | w=({wA:.2f},{wB:.2f},{wC:.2f}) | lam={lam:.2f} | shift_mult={sm:.1f} | shift={shift:.2f}")

    obj, mean_m, std_m, wA, wB, wC, lam, sm, shift = best
    mA, mB, mC = model_names
    oof_final = wA*oof_agg[mA] + wB*oof_agg[mB] + wC*oof_agg[mC]
    oof_final = (1.0-lam)*oof_final + lam*baseline_vec
    oof_final = oof_final + shift

    meta = {
        "models_order": model_names,
        "weights": (float(wA), float(wB), float(wC)),
        "lam_baseline": float(lam),
        "shift_mult": float(sm),
        "shift_value": float(shift),
        "oof_mean_mae_across_seeds": float(mean_m),
        "oof_std_mae_across_seeds": float(std_m),
    }
    return oof_final, meta


# -----------------------------
# Optional small group shift (very conservative)
# -----------------------------
def apply_chronic_group_shift(train_feat: pd.DataFrame, pred_oof: np.ndarray, shrink: float = 0.25) -> Tuple[np.ndarray, Dict]:
    y = train_feat[TARGET].values.astype(float)
    chronic = train_feat["primary_chronic"].astype(str).values
    resid = y - pred_oof
    shifts = {}
    for g in np.unique(chronic):
        med = float(np.median(resid[chronic == g]))
        shifts[g] = shrink * med
    pred2 = pred_oof.copy()
    for g, s in shifts.items():
        pred2[chronic == g] += s
    return pred2, shifts


# -----------------------------
# Main
# -----------------------------
must_exist(TRAIN_PATH, "TRAIN")
must_exist(TEST_PATH, "TEST")
must_exist(PATIENTS_PATH, "patients")
must_exist(ADM_TRAIN_PATH, "admissions_train")
must_exist(ADM_TEST_PATH, "admissions_test")
if not os.path.exists(RECEIPTS_JOBLIB_PATH):
    print("[warn] receipts_parsed.joblib missing -> receipts features will be empty (likely worse).")

print("\n[load] reading CSVs...")
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
patients = pd.read_csv(PATIENTS_PATH)

log_shape("ed_cost_train", train)
log_shape("ed_cost_test", test)
log_shape("patients", patients)

print("\n[target stats]")
print(train[TARGET].describe().to_string())

# ids
for df in [train, test, patients]:
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype(int)

# Ch1-risk patient features (OOF-consistent)
print("\n[ch1-risk] building auxiliary patient readmission risk features (OOF-consistent)...")
try:
    ch1_risk_df = build_ch1_patient_risk_features(
        ADM_TRAIN_PATH, ADM_TEST_PATH,
        patients_df=patients,
        notes_path=DISCHARGE_NOTES_PATH,
        cache_path=CH1_RISK_CACHE_PATH,
        n_splits=CFG.CH1_N_SPLITS,
        svd_dim=CFG.CH1_SVD_DIM,
        seed=SEED
    )
    print(f"  ch1 risk features: {ch1_risk_df.shape} | cols={list(ch1_risk_df.columns)}")
except Exception as e:
    print(f"  [warn] ch1 risk build failed: {e}")
    ch1_risk_df = None

# admissions aggregates for Ch2 main
print("\n[admissions] building robust aggregates (Ch2 classic)...")
adm_df = load_admissions_features(ADM_TRAIN_PATH, ADM_TEST_PATH)
if adm_df is None:
    print("  admissions features: None")
else:
    print(f"  admissions features: {adm_df.shape} | cols={list(adm_df.columns)}")

# receipts
print("\n[receipts] loading receipts_parsed.joblib and building low-dim receipt features...")
rcpt_df = None
if os.path.exists(RECEIPTS_JOBLIB_PATH):
    try:
        rcpt_df = load_receipts_joblib(RECEIPTS_JOBLIB_PATH)
        if rcpt_df is not None:
            rcpt_df["patient_id"] = pd.to_numeric(rcpt_df["patient_id"], errors="coerce").astype("Int64")
            rcpt_df = rcpt_df.dropna(subset=["patient_id"]).copy()
            rcpt_df["patient_id"] = rcpt_df["patient_id"].astype(int)
            rcpt_df = rcpt_df.drop_duplicates("patient_id", keep="last")
            print(f"  receipt_feat shape: {rcpt_df.shape}")
            print(f"  receipt_feat cols ({len(rcpt_df.columns)-1}): {[c for c in rcpt_df.columns if c!='patient_id']}")
        else:
            print("  [warn] could not build receipt features from joblib structure.")
    except Exception as e:
        print(f"  [warn] receipts joblib load/build failed: {e}")
        rcpt_df = None
else:
    print("  [warn] receipts joblib missing; skipping receipts features.")

# build features
print("\n[features] building train/test feature frames...")
train_feat = build_features(train, patients, adm_df, rcpt_df, ch1_risk_df)
test_feat  = build_features(test,  patients, adm_df, rcpt_df, ch1_risk_df)

# choose features
feat_full = get_numeric_feature_cols(train_feat)
feat_full = [c for c in feat_full if c != TARGET]
feat_full = drop_constants(feat_full, train_feat)

# PRUNED set: stable low-dim list (extend Code18 with ONLY ch1-risk low-dim)
pruned_candidates = [
    # priors + transforms
    "prior_ed_visits_5y","prior_ed_cost_5y_usd","prior_cost_cap20k","sqrt_prior_cost","log_prior_cost","log_prior_cost_cap20k","cost_per_visit","log_visits",
    "baseline_next3y",
    # demographics
    "chronic_encoded","age","sex_encoded","insurance_encoded","zip_region","ins_x_chronic",
    # admissions
    "charlson_max","charlson_mean","pct_emergent",
    # ch1 risk
    "ch1_readmit_p_mean","ch1_readmit_p_max","ch1_readmit_p_std","ch1_n_admissions","ch1_dx_entropy",
    "logprior_x_readmitp","visits_x_readmitp",
    # receipt robust
    "cost_per_em","cost_hhi","pct_cost_em","pct_cost_procedure","pct_cost_critical","pct_cost_high_acuity","pct_cost_imaging","pct_cost_lab",
    "n_high_acuity_total","has_critical_care","has_99285","max_em_level","avg_em_level","n_high_em","n_unique_codes",
    "top1_share","top3_share","gini_line_total","max_line_total",
    "median_unit_price","median_unit_price_em","median_unit_price_imaging","median_unit_price_lab",
    "log1p_median_unit_price","log1p_median_unit_price_em","log1p_median_unit_price_imaging","log1p_median_unit_price_lab","log1p_max_line_total",
    # light interactions
    "logprior_x_pctcritical","logprior_x_highacu",
    # stable ratio
    "cost_per_code",
]
feat_pruned = [c for c in pruned_candidates if c in train_feat.columns]
feat_pruned = drop_constants(feat_pruned, train_feat)

print(f"  FULL feature count:   {len(feat_full)}")
print(f"  PRUNED feature count: {len(feat_pruned)}")
print(f"  PRUNED features: {feat_pruned}")

# safety fill
for c in set(feat_full + feat_pruned):
    med = train_feat[c].median() if c in train_feat.columns and not train_feat[c].isna().all() else 0.0
    train_feat[c] = pd.to_numeric(train_feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(med)
    test_feat[c]  = pd.to_numeric(test_feat[c], errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(med)

# train
oof_by_seed, test_by_seed, best_iters = train_models(train_feat, test_feat, feat_full, feat_pruned)

# baseline vectors for blending
baseline_oof  = train_feat["baseline_next3y"].values.astype(float)
baseline_test = test_feat["baseline_next3y"].values.astype(float)

# stable ensemble search on OOF
oof_ens, meta = stable_ensemble_search(train_feat, oof_by_seed, baseline_oof)

# build final test ensemble using chosen weights + baseline blend + shift
mA, mB, mC = meta["models_order"]
wA, wB, wC = meta["weights"]
lam = meta["lam_baseline"]
shift = meta["shift_value"]

test_agg = {m: trimmed_mean(np.vstack(test_by_seed[m]), trim_k=CFG.TRIM_K) for m in meta["models_order"]}
test_ens = wA*test_agg[mA] + wB*test_agg[mB] + wC*test_agg[mC]
test_ens = (1.0-lam)*test_ens + lam*baseline_test
test_ens = test_ens + shift

# optional chronic shift (very conservative; require noticeable OOF gain)
y = train_feat[TARGET].values.astype(float)
base_mae = float(mean_absolute_error(y, oof_ens))
best_oof = oof_ens
best_shift = {"type": "none"}

for shrink in [0.0, 0.20, 0.30]:
    if shrink <= 0:
        continue
    oof2, shifts = apply_chronic_group_shift(train_feat, oof_ens, shrink=shrink)
    m2 = float(mean_absolute_error(y, oof2))
    if m2 < base_mae - 0.12:
        base_mae = m2
        best_oof = oof2
        best_shift = {"type": "chronic_group", "shrink": shrink, "shifts": shifts}

if best_shift["type"] == "chronic_group":
    test_chronic = test_feat["primary_chronic"].astype(str).values
    for g, s in best_shift["shifts"].items():
        test_ens[test_chronic == g] += s

# clip predictions to a reasonable range (LB-safe)
y_max = float(np.max(y))
test_ens = np.clip(test_ens, 0.0, y_max * 1.5)

# feature importance from a full-fit Model A (quick insight)
print("\n[full-train] fitting Model A on full train for feature importance...")
A_params = dict(
    loss_function="RMSE", eval_metric="MAE",
    depth=5, l2_leaf_reg=8, min_data_in_leaf=40,
    learning_rate=CFG.LR, iterations=CFG.ITERS,
    rsm=CFG.RSM, bootstrap_type="Bernoulli", subsample=CFG.SUBSAMPLE,
    random_strength=1.0,
    task_type="CPU", thread_count=-1,
    verbose=0, allow_writing_files=False,
    random_seed=SEED,
)
mA_full = CatBoostRegressor(**A_params)
mA_full.fit(train_feat[feat_full], y, verbose=0)
try:
    imp = mA_full.get_feature_importance()
    imp_df = pd.DataFrame({"feature": feat_full, "importance": imp}).sort_values("importance", ascending=False).head(12)
    print("\n[Top feature importances] (Model A full)")
    print(imp_df.to_string(index=False))
except Exception as e:
    print(f"[warn] feature importance failed: {e}")
del mA_full
gc.collect()

# final logs
final_oof_mae = float(mean_absolute_error(y, best_oof))
print("\n" + "="*90)
print("[FINAL OOF]")
print(f"  ensemble OOF MAE (stable search + optional chronic shift): {final_oof_mae:.3f}")
print("  ensemble meta:", meta)
print("  extra shift:", best_shift["type"], ("shrink="+str(best_shift.get("shrink")) if best_shift["type"]!="none" else ""))
print("  OOF pred quantiles:", qdict(best_oof, qs=(0,0.01,0.05,0.1,0.5,0.9,0.95,0.99,1.0)))
print("="*90)

# write submission
sub = pd.DataFrame({
    "patient_id": test["patient_id"].values.astype(int),
    "ed_cost_next3y_usd": np.round(test_ens.astype(float), 2)
})
sub = sub[["patient_id", "ed_cost_next3y_usd"]]

out_path = Path(OUT_SUB_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)

# sanity checks
print("\n[SUBMISSION sanity checks]")
print("submission shape:", sub.shape)
print("submission columns exactly:", list(sub.columns))
print("any NaNs in preds:", bool(np.isnan(sub["ed_cost_next3y_usd"]).any()))
print("pred min/median/max:", float(sub["ed_cost_next3y_usd"].min()), float(sub["ed_cost_next3y_usd"].median()), float(sub["ed_cost_next3y_usd"].max()))
print("pred quantiles:", qdict(sub["ed_cost_next3y_usd"].values))
print("\nSaved submission to:", str(out_path))
print("\nPaste back: (1) leaderboard MAE, (2) FINAL OOF MAE, (3) ensemble meta + extra shift, (4) pred quantiles.")


CODE 19 | Code18 + CH1-risk bridge: LOW-DIM receipts + shallow CatBoost + strong reg + multi-seed + STABLE ensemble
Add-on: train auxiliary CH1 readmit model (OOF-consistent) -> patient risk aggregates -> feed to Ch2 regression.

[load] reading CSVs...
[ed_cost_train] shape=(2000, 5) | cols=5 | mem=0.17 MB
[ed_cost_test] shape=(2000, 4) | cols=4 | mem=0.15 MB
[patients] shape=(4000, 5) | cols=5 | mem=0.49 MB

[target stats]
count     2000.00000
mean      3908.25191
std       1822.40192
min        306.88000
25%       2548.76750
50%       3569.09500
75%       4956.42250
max      11184.61000

[ch1-risk] building auxiliary patient readmission risk features (OOF-consistent)...
[ch1-risk] aux OOF macro-F1@0.5: 0.7034 | text=True | svd_dim=16
[ch1-risk] cached -> D:\AgentDs\agent_ds_healthcare\cache_ch1_patient_risk.joblib
  ch1 risk features: (4000, 6) | cols=['patient_id', 'ch1_readmit_p_mean', 'ch1_readmit_p_max', 'ch1_readmit_p_std', 'ch1_n_admissions', 'ch1_dx_entropy']

[admissions] bui

# New Iter

# Submission

In [ ]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 2, "D:/AgentDs/agent_ds_healthcare/submission.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 3!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 450.9103 (MAE)
✅ Validation passed
✅ Submission successful!
   📊 Score: 450.9103
   📏 Metric: MAE
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. Move on to Healthcare Challenge 3!


: 